# Residential Smart Meter Archetype Clustering
## CEB High-Resolution Telemetry Pipeline

**Research objective:** Identify interpretable household electricity-use
archetypes from 15-minute smart-meter telemetry and evaluate their robustness
using internal, algorithmic, temporal, and sensitivity analyses.

**Dataset:** Ceylon Electricity Board smart-meter telemetry

- Three 15-minute CSV files containing approximately 45 million raw records.
- Raw coverage spans October 2023 to December 2024.
- The raw files contain 2,938 metered premises.
- After retaining Load Profile (`LP`) records and excluding premises represented
  only by `DBE` event logs, 2,932 households have consumption telemetry.

The primary clustering analysis is restricted to the corrected May–November
2024 golden window and uses the locked six-feature valid-day pipeline defined
below.


> **Public review draft (not an independently executable release).** Nine case-specific diagnostic code cells have been withheld; one population-wide audit cell and one reporting cell have been edited to remove household-specific references. The main clustering algorithm and feature construction code are otherwise retained, but have not been rerun after curation. Required data are restricted and absent. See `REPRODUCIBILITY.md`.


### 1. Temporal Boundary Definition: Golden Window

The raw telemetry audit shows a fleet-level reduction in active household
coverage during February–March 2024. The available data do not establish the
operational cause of this disruption; therefore, it is treated as an observed
telemetry-instability period rather than attributed to a specific outage or
firmware event.

To prevent this coverage disruption from biasing temporal feature extraction,
the primary analysis uses a continuous seven-month golden window:

- start: **May 1, 2024 00:00:00**, inclusive;
- end: **December 1, 2024 00:00:00**, exclusive.

This includes the full May–November 2024 period and the final possible
15-minute timestamp of **November 30, 2024 23:45:00**.


# FINAL CORRECTION NOTE

This notebook contains the corrected valid-day-only feature pipeline and the final locked research protocol:

- household-days require at least 90 valid 15-minute readings;
- households require at least 30 valid days;
- features are built only from retained valid days;
- the raw activity rule (`log_daily_kwh > 0.05`) is applied before clipping;
- 1st–99th percentile clipping bounds are calculated from the active population;
- the final model uses `K=4`, `random_state=42`, and `n_init=20`.

All stored outputs were cleared during final correction. Restart the kernel and run all cells sequentially so every table, figure, and export is regenerated from the corrected population.

- Alternative clustering checks now include explicitly converged GMM, multi-start Fuzzy C-Means, and dependency-free K-Medoids implementations.
- Monthly retention and transition analysis reuse one authoritative monthly feature matrix.
- Tariff-block coverage, ±15-minute boundary sensitivity, and higher-cap sensitivity are included as explicit diagnostics.


### 2. Data Ingestion and Quality Gates

Before feature engineering, the telemetry passes the following locked quality
gates:

1. **OBIS filtering:** Only Load Profile (`LP`) records are retained. `DBE`
   event-log records are excluded during ingestion.
2. **Solar exclusion:** Households whose cumulative export-register delta
   exceeds 1.0 kWh across the dataset are excluded as active exporters.
3. **Conservative interval cap:** Import increments above 6.0 kWh per
   15-minute interval are masked in the primary specification. The threshold is
   based on a conservative three-phase 30 A benchmark with a small margin.
   Because higher-capacity supplies may exist, 8.0 and 10.5 kWh alternatives
   are evaluated separately as sensitivity analyses.
4. **Valid-day requirement:** A valid household-day contains at least 90 valid
   15-minute readings.
5. **Household eligibility:** A household contributes at least 30 valid days
   within the golden window.
6. **Ghost purge:** The raw household feature `log_daily_kwh` must exceed
   0.05. This rule is applied before clipping.
7. **Outlier treatment:** Each feature is winsorised to the 1st and 99th
   percentiles calculated from the surviving active-household population.

These rules define the primary analysis. Diagnostic alternatives are reported
without silently replacing the locked specification.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 [CODE] — ALL IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import datetime
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import entropy, kruskal
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score,
    silhouette_score,
    silhouette_samples,
    davies_bouldin_score,
    calinski_harabasz_score
)
from sklearn.utils import resample
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully.")



In [ ]:
# CELL 5 [CODE] — MASTER PIPELINE (Phases 1–6)
# Outputs: golden_df, feature_matrix
# ─────────────────────────────────────────────────────────────────────────────
print("--- PHASE 1: MEMORY-SAFE INGESTION & DOMAIN PURGE ---")
folder_path = "."
files = sorted([
    os.path.join(folder_path, f)
    for f in os.listdir(folder_path)
    if f in ["smart_15min_1.csv", "smart_15min_2.csv", "smart_15min_3.csv"]
])
if len(files) != 3:
    raise FileNotFoundError(
        "Expected smart_15min_1.csv, smart_15min_2.csv, and smart_15min_3.csv "
        f"in {os.path.abspath(folder_path)}, but found: "
        f"{[os.path.basename(f) for f in files]}"
    )

# Read only fields used by this notebook. Each file is processed in chunks,
# LP rows are retained immediately, local timestamps are parsed, and the
# discarded raw fields never enter the combined dataframe. This prevents the
# multi-gigabyte allocation produced by reading all source columns at once.
RAW_USECOLS = [
    "household_ID",
    "obis",
    "date",
    "time",
    "importkwh(kwh)",
    "exportkwh(kwh)"
]
RAW_DTYPES = {
    "household_ID": "string",
    "obis": "category",
    "date": "string",
    "time": "string",
    "importkwh(kwh)": "float64",
    "exportkwh(kwh)": "float64"
}
READ_CHUNK_SIZE = 500_000

_household_to_code = {}
_code_to_household = []
_processed_chunks = []
invalid_meter_dt = 0
raw_rows_read = 0
lp_rows_retained = 0

for file_path in files:
    file_rows = 0
    file_lp_rows = 0

    reader = pd.read_csv(
        file_path,
        usecols=RAW_USECOLS,
        dtype=RAW_DTYPES,
        chunksize=READ_CHUNK_SIZE,
        low_memory=False
    )

    for chunk in reader:
        raw_rows_read += len(chunk)
        file_rows += len(chunk)

        lp_chunk = chunk.loc[chunk["obis"].eq("LP")].copy()
        if lp_chunk.empty:
            continue

        # Parse only the retained LP rows. Chunk-level string construction
        # limits peak memory while preserving the local date/time ground truth.
        lp_chunk["meter_dt"] = pd.to_datetime(
            lp_chunk["date"].str.strip() + " " + lp_chunk["time"].str.strip(),
            errors="coerce"
        )

        bad_dt = int(lp_chunk["meter_dt"].isna().sum())
        invalid_meter_dt += bad_dt
        if bad_dt:
            lp_chunk = lp_chunk.dropna(subset=["meter_dt"])

        if lp_chunk.empty:
            continue

        # Encode repeated household identifiers as compact integer codes while
        # reading. The readable categorical household_ID is restored once.
        ids = lp_chunk["household_ID"].astype(str)
        new_ids = pd.Index(ids.unique()).difference(_household_to_code.keys())
        for household_id in new_ids:
            _household_to_code[household_id] = len(_code_to_household)
            _code_to_household.append(household_id)

        lp_chunk["_household_code"] = (
            ids.map(_household_to_code).astype("int32")
        )

        processed = lp_chunk[
            [
                "_household_code",
                "meter_dt",
                "importkwh(kwh)",
                "exportkwh(kwh)"
            ]
        ].copy()

        _processed_chunks.append(processed)
        lp_rows_retained += len(processed)
        file_lp_rows += len(processed)

        del chunk, lp_chunk, ids, processed

    print(
        f"{os.path.basename(file_path)}: "
        f"{file_rows:,} raw rows, {file_lp_rows:,} valid LP rows retained."
    )

if not _processed_chunks:
    raise RuntimeError("No valid LP rows were read from the three smart-meter files.")

merged_df = pd.concat(_processed_chunks, ignore_index=True, copy=False)
del _processed_chunks

household_codes = merged_df.pop("_household_code").to_numpy(dtype=np.int32)
merged_df.insert(
    0,
    "household_ID",
    pd.Categorical.from_codes(
        household_codes,
        categories=_code_to_household,
        ordered=False
    )
)
del household_codes, _household_to_code, _code_to_household

required_raw_columns = {
    "household_ID",
    "meter_dt",
    "importkwh(kwh)",
    "exportkwh(kwh)"
}
missing_raw_columns = required_raw_columns.difference(merged_df.columns)
if missing_raw_columns:
    raise ValueError(f"Missing required processed columns: {sorted(missing_raw_columns)}")

if invalid_meter_dt:
    print(f"Dropped {invalid_meter_dt:,} LP rows with unparseable local timestamps.")

print(
    f"Read {raw_rows_read:,} raw rows and retained {len(merged_df):,} valid LP rows "
    f"across {merged_df['household_ID'].nunique()} households."
)

print("\n--- PHASE 2: SOLAR FILTER (DELTA LOGIC) ---")
# Delta method: flags only households with active export activity.
# Prevents false-positives from inherited odometer values.
solar_export_delta = (
    merged_df.groupby('household_ID', observed=True)['exportkwh(kwh)'].max()
    - merged_df.groupby('household_ID', observed=True)['exportkwh(kwh)'].min()
)
solar_households = solar_export_delta[solar_export_delta > 1.0].index.tolist()
clean_df = merged_df[~merged_df['household_ID'].isin(solar_households)].copy()
if isinstance(clean_df["household_ID"].dtype, pd.CategoricalDtype):
    clean_df["household_ID"] = (
        clean_df["household_ID"].cat.remove_unused_categories()
    )
print(f"Dropped {len(solar_households)} active solar PV households.")
print(f"Remaining: {clean_df['household_ID'].nunique()} households.")

print("\n--- PHASE 3: PHYSICS ENFORCEMENT (CONSERVATIVE 6.0 kWh LIMIT) ---")
clean_df.sort_values(['household_ID', 'meter_dt'], inplace=True)
clean_df['time_delta_mins'] = (
    clean_df.groupby('household_ID', observed=True)['meter_dt']
    .diff().dt.total_seconds() / 60.0
)
clean_df['raw_kwh_diff'] = (
    clean_df.groupby('household_ID', observed=True)['importkwh(kwh)'].diff()
)

valid_time_mask = (
    (clean_df['time_delta_mins'] >= 14.5) &
    (clean_df['time_delta_mins'] <= 15.5)
)
valid_energy_mask = clean_df['raw_kwh_diff'] >= 0
# Conservative physical cap for 15-minute import increments.
# A 3-phase 30A supply at 230V over 15 minutes is approximately:
# 3 × 230V × 30A × 0.25h / 1000 ≈ 5.175 kWh.
# Therefore, a 6.0 kWh/15-min cap provides a small margin above this level
# and suppresses implausible register jumps or extreme spikes.
#
# The 6-hour meter metadata confirm that a minority of final households
# have three-phase meter subtypes. Some higher-capacity three-phase
# households may therefore have valid intervals above 6.0 kWh/15-min.
# Such intervals are conservatively masked and this is reported as a
# limitation rather than treated as a modelling flaw.
PHYSICAL_LIMIT = 6.0

clean_df['true_15min_kwh'] = np.where(
    valid_time_mask & valid_energy_mask & (clean_df['raw_kwh_diff'] <= PHYSICAL_LIMIT),
    clean_df['raw_kwh_diff'],
    np.nan
)
valid_pct = clean_df['true_15min_kwh'].notna().mean() * 100
print(f"Valid 15-min readings: {valid_pct:.1f}% of post-solar dataset.")

# Register columns remain available in compact merged_df for later
# import/export audits. They are no longer needed in clean_df after differencing.
clean_df.drop(
    columns=['importkwh(kwh)', 'exportkwh(kwh)'],
    inplace=True,
    errors='ignore'
)
clean_df['time_delta_mins'] = clean_df['time_delta_mins'].astype('float32')

print("\n--- PHASE 4: GOLDEN WINDOW & COMPLETENESS FILTER ---")
# Correct timestamp boundary:
# include all readings from 2024-05-01 00:00:00 through 2024-11-30 23:45:00.
# Therefore use an exclusive upper bound at 2024-12-01 00:00:00.
START_DATE = pd.Timestamp('2024-05-01 00:00:00')
END_DATE = pd.Timestamp('2024-12-01 00:00:00')  # exclusive end; full May-November window
END_DATE_EXCLUSIVE = END_DATE

golden_df = clean_df[
    (clean_df['meter_dt'] >= START_DATE) &
    (clean_df['meter_dt'] < END_DATE_EXCLUSIVE)
].copy()

print(f"Golden window applied: {START_DATE} <= meter_dt < {END_DATE_EXCLUSIVE}")
print(f"Expected final 15-minute timestamp: {END_DATE_EXCLUSIVE - pd.Timedelta(minutes=15)}")

golden_df['date_only'] = golden_df['meter_dt'].dt.normalize()
# Use .count() not .size() — NaN-aware, only counts valid readings
day_counts = (
    golden_df.groupby(['household_ID', 'date_only'], observed=True)['true_15min_kwh']
    .count().reset_index()
)
valid_days = day_counts[day_counts['true_15min_kwh'] >= 90]
hh_validity = valid_days.groupby('household_ID', observed=True).size()
# 30-day minimum = one complete CEB administrative billing cycle.
# Ensures each household contributes one billing-period equivalent of
# valid telemetry aligned with CEB monthly measurement unit.
# Stricter thresholds would exclude households with legitimate short
# meter outages on a grid with documented firmware gaps.
final_households = hh_validity[hh_validity >= 30].index

# Keep golden_df as the complete golden-window dataframe for coverage audits,
# but construct the clustering features only from household-days that passed
# the >=90 valid-interval rule.
golden_df = golden_df[
    golden_df['household_ID'].isin(final_households)
].copy()
if isinstance(golden_df["household_ID"].dtype, pd.CategoricalDtype):
    golden_df["household_ID"] = (
        golden_df["household_ID"].cat.remove_unused_categories()
    )

valid_day_keys = (
    valid_days.loc[
        valid_days['household_ID'].isin(final_households),
        ['household_ID', 'date_only']
    ]
    .drop_duplicates()
)

# Use only the columns required for feature engineering to control memory use.
feature_df = (
    golden_df[
        ['household_ID', 'meter_dt', 'true_15min_kwh', 'date_only']
    ]
    .merge(
        valid_day_keys,
        on=['household_ID', 'date_only'],
        how='inner',
        validate='many_to_one'
    )
)

feature_day_counts = (
    feature_df.groupby(['household_ID', 'date_only'], observed=True)['true_15min_kwh']
    .count()
)

assert not feature_day_counts.empty, "No valid household-days remained."
assert feature_day_counts.min() >= 90, (
    "Feature data contain a household-day with fewer than 90 valid intervals."
)
assert feature_df['household_ID'].nunique() == len(final_households), (
    "At least one eligible household disappeared during valid-day filtering."
)

print(f"Households surviving 30-day Golden Window cut: {len(final_households)}")
print(f"Valid household-days used for feature extraction: {len(valid_day_keys):,}")
print(f"Rows used for feature extraction: {len(feature_df):,}")
print(f"Minimum valid intervals in any retained feature day: "
      f"{int(feature_day_counts.min())}")

print("\n--- PHASE 5: CEB FEATURE EXTRACTION ---")
sl_holidays_2024 = pd.to_datetime([
    '2024-05-01', '2024-05-23', '2024-05-24',
    '2024-06-21', '2024-07-20',
    '2024-08-19', '2024-09-16', '2024-09-17', '2024-10-17', '2024-10-31',
    '2024-11-15'
]).normalize()
feature_df['is_non_working_day'] = (
    (feature_df['meter_dt'].dt.dayofweek >= 5) |
    feature_df['date_only'].isin(sl_holidays_2024)
)

# Bulletproof CEB Time Blocks using datetime.time objects
# Eliminates operator precedence ambiguity in integer-hour comparisons
meter_time = feature_df['meter_dt'].dt.time
t_0530 = datetime.time(5, 30)
t_1830 = datetime.time(18, 30)
t_2230 = datetime.time(22, 30)

conditions = [
    (meter_time >= t_0530) & (meter_time < t_1830),  # Day:     05:30–18:29
    (meter_time >= t_1830) & (meter_time < t_2230)   # Peak:    18:30–22:29
]                                                     # Off-Peak: 22:30–05:29
feature_df['ceb_block'] = pd.Categorical(
    np.select(conditions, ['Day', 'Peak'], default='Off-Peak'),
    categories=['Day', 'Peak', 'Off-Peak'],
    ordered=False
)

daily_block_usage = (
    feature_df.groupby(['household_ID', 'date_only', 'ceb_block'], observed=True)
    ['true_15min_kwh'].sum().unstack(fill_value=0)
)
avg_daily_block = daily_block_usage.groupby('household_ID', observed=True).mean()
total_avg_daily = avg_daily_block.sum(axis=1)

feature_matrix = pd.DataFrame(index=avg_daily_block.index)

# Feature 1: Evening Peak Ratio
feature_matrix['evening_peak_ratio'] = (
    avg_daily_block['Peak'] / total_avg_daily.clip(lower=0.01)
)
# Feature 2: Night Baseload Ratio
feature_matrix['night_baseload_ratio'] = (
    avg_daily_block['Off-Peak'] / total_avg_daily.clip(lower=0.01)
)

# A separate daytime share is intentionally omitted because Day, Peak, and
# Off-Peak shares sum to one. Including all three would introduce exact
# redundancy and over-weight tariff-block composition in Euclidean distance.

# Feature 3: Load Uniformity (Shannon entropy on the 96-interval profile)
feature_df['interval_id'] = (
    feature_df['meter_dt'].dt.hour * 4 +
    feature_df['meter_dt'].dt.minute // 15
).astype('uint8')
interval_profiles = (
    feature_df.groupby(['household_ID', 'interval_id'], observed=True)['true_15min_kwh']
    .mean()
    .unstack()
    .reindex(columns=range(96))
)

# Do not interpret an entirely unobserved interval position as zero usage.
# Households lacking any of the 96 profile positions receive NaN for this
# feature and are removed by the Phase 6 NaN/Inf guard.
complete_profile_mask = interval_profiles.notna().all(axis=1)
print(
    "Households with all 96 interval positions represented:",
    int(complete_profile_mask.sum()),
    "/",
    len(interval_profiles)
)
complete_interval_profiles = interval_profiles.loc[complete_profile_mask]

feature_matrix['load_uniformity'] = complete_interval_profiles.apply(
    lambda x: entropy(x.to_numpy(dtype=float) + 1e-9),
    axis=1
)

# Feature 4: Load Factor (mean/peak — measures consumption smoothness)
hh_stats = feature_df.groupby('household_ID', observed=True)['true_15min_kwh'].agg(['mean', 'max'])
feature_matrix['load_factor'] = hh_stats['mean'] / hh_stats['max'].clip(lower=0.01)

# Feature 5: Leisure Usage Ratio (log-transformed weekend/weekday ratio)
daily_totals = (
    feature_df.groupby(['household_ID', 'date_only', 'is_non_working_day'], observed=True)
    ['true_15min_kwh'].sum().reset_index()
)
day_type_avg = (
    daily_totals.groupby(['household_ID', 'is_non_working_day'], observed=True)
    ['true_15min_kwh'].mean()
    .unstack()
    .reindex(columns=[False, True])
)
feature_matrix['leisure_usage_ratio'] = np.log1p(
    day_type_avg[True] / day_type_avg[False].clip(lower=0.01)
)

# Feature 6: Log Daily kWh (absolute consumption scale)
feature_matrix['log_daily_kwh'] = np.log1p(total_avg_daily)

print("\n--- PHASE 6: RAW FEATURE-MATRIX QUALITY GUARD ---")
pre_drop = len(feature_matrix)
feature_matrix.replace([np.inf, -np.inf], np.nan, inplace=True)
feature_matrix.dropna(inplace=True)
post_drop = len(feature_matrix)
print(f"NaN/Inf guard removed {pre_drop - post_drop} households.")

# Do not clip here. The substantive ghost/activity rule in Cell 6 must be
# applied to raw log_daily_kwh before percentile bounds are calculated.
print(
    f"RAW FEATURE MATRIX COMPLETE: {len(feature_matrix)} households "
    "(not yet ghost-purged or percentile-clipped)."
)


In [ ]:
# ============================================================
# TARIFF-BLOCK-SPECIFIC COVERAGE AUDIT
# Valid days contain at least 90/96 intervals. This audit checks whether the
# remaining missing intervals are concentrated in Day, Peak or Off-Peak.
# It is diagnostic and does not alter the locked feature matrix.
# ============================================================

from pathlib import Path
expected_block_intervals = {
    "Day": 52,
    "Peak": 16,
    "Off-Peak": 28
}

block_coverage = (
    feature_df
    .groupby(["household_ID", "date_only", "ceb_block"], observed=True)["true_15min_kwh"]
    .count()
    .unstack(fill_value=0)
    .reindex(columns=["Day", "Peak", "Off-Peak"], fill_value=0)
)

for block, expected in expected_block_intervals.items():
    block_coverage[f"{block}_coverage_ratio"] = block_coverage[block] / expected
    block_coverage[f"{block}_missing"] = (expected - block_coverage[block]).clip(lower=0)

block_coverage_summary = pd.DataFrame({
    "expected_intervals": pd.Series(expected_block_intervals),
    "minimum_observed_intervals": block_coverage[["Day", "Peak", "Off-Peak"]].min(),
    "household_days_incomplete": pd.Series({
        block: int((block_coverage[block] < expected).sum())
        for block, expected in expected_block_intervals.items()
    }),
    "incomplete_household_day_share": pd.Series({
        block: float((block_coverage[block] < expected).mean())
        for block, expected in expected_block_intervals.items()
    }),
    "mean_coverage_ratio": pd.Series({
        block: float(block_coverage[f"{block}_coverage_ratio"].mean())
        for block in expected_block_intervals
    })
})

peak_household_coverage = (
    block_coverage["Peak"].groupby("household_ID", observed=True).mean()
    / expected_block_intervals["Peak"]
)

print("Tariff-block coverage summary:")
display(block_coverage_summary.round(6))

print("\nPeak-block missing-interval distribution across valid household-days:")
display(block_coverage["Peak_missing"].value_counts().sort_index().rename("household_days"))

print("\nHouseholds with average peak-block coverage below 95%:")
low_peak_coverage_households = peak_household_coverage[
    peak_household_coverage < 0.95
].sort_values()
print(len(low_peak_coverage_households))
display(low_peak_coverage_households.head(30).rename("mean_peak_coverage_ratio"))

Path("tables_outputs").mkdir(exist_ok=True)
block_coverage_summary.to_csv("tables_outputs/tariff_block_coverage_summary.csv")
low_peak_coverage_households.rename("mean_peak_coverage_ratio").to_csv(
    "tables_outputs/households_below_95pct_peak_block_coverage.csv"
)

print(
    "Interpretation rule: feature logic is not changed solely because some "
    "valid days contain 90–95 intervals. Any material concentration of "
    "missingness within the peak block must be reported as a limitation or "
    "evaluated in a separate specification."
)

In [ ]:
# ============================================================
# AUDIT: WHY POPULATION CHANGED FROM 2,226 TO 2,227
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# PART A: Reconstruct the old all-days log_daily_kwh
# ------------------------------------------------------------
# The original pipeline used every available day belonging to
# households that passed the 30-valid-day eligibility criterion.

old_daily_totals = (
    golden_df
    .groupby(['household_ID', 'date_only'], observed=True)['true_15min_kwh']
    .sum(min_count=1)
)

old_average_daily = (
    old_daily_totals
    .groupby('household_ID', observed=True)
    .mean()
)

old_log_daily_kwh = np.log1p(old_average_daily)

# The corrected feature_matrix contains valid-day-only raw features.
new_log_daily_kwh = feature_matrix['log_daily_kwh'].copy()

population_audit = pd.concat(
    [
        old_log_daily_kwh.rename('old_all_days_log_daily_kwh'),
        new_log_daily_kwh.rename('new_valid_days_log_daily_kwh')
    ],
    axis=1
)

population_audit['old_active'] = (
    population_audit['old_all_days_log_daily_kwh'] > 0.05
)

population_audit['new_active'] = (
    population_audit['new_valid_days_log_daily_kwh'] > 0.05
)

print("Old all-days ghost count:",
      int((~population_audit['old_active']).sum()))

print("New valid-days ghost count:",
      int((~population_audit['new_active']).sum()))

print("Old active population:",
      int(population_audit['old_active'].sum()))

print("New active population:",
      int(population_audit['new_active'].sum()))

became_active = population_audit[
    (~population_audit['old_active']) &
    (population_audit['new_active'])
].copy()

became_ghost = population_audit[
    (population_audit['old_active']) &
    (~population_audit['new_active'])
].copy()

print("\nGhost → Active households:", len(became_active))
print("Active → Ghost households:", len(became_ghost))
print("Net population change:",
      len(became_active) - len(became_ghost))

print("\nHouseholds that became active:")
display(became_active)

print("\nHouseholds that became ghosts:")
display(became_ghost)


# ------------------------------------------------------------
# PART B: Check whether clipping-before-purge changed membership
# ------------------------------------------------------------
core_features = [
    'evening_peak_ratio',
    'night_baseload_ratio',
    'load_uniformity',
    'load_factor',
    'leisure_usage_ratio',
    'log_daily_kwh'
]

raw_features = feature_matrix[core_features].copy()

clip_first_features = raw_features.copy()

for col in core_features:
    lower = clip_first_features[col].quantile(0.01)
    upper = clip_first_features[col].quantile(0.99)

    clip_first_features[col] = (
        clip_first_features[col]
        .clip(lower=lower, upper=upper)
    )

raw_purge_active_ids = set(
    raw_features.index[
        raw_features['log_daily_kwh'] > 0.05
    ]
)

clip_first_active_ids = set(
    clip_first_features.index[
        clip_first_features['log_daily_kwh'] > 0.05
    ]
)

rescued_by_clipping = sorted(
    clip_first_active_ids - raw_purge_active_ids
)

lost_due_to_clipping = sorted(
    raw_purge_active_ids - clip_first_active_ids
)

print("\n--- ORDER EFFECT AUDIT ---")
print("Raw-purge population:", len(raw_purge_active_ids))
print("Clip-first population:", len(clip_first_active_ids))
print("Households rescued by clipping:", len(rescued_by_clipping))
print("Households lost due to clipping:", len(lost_due_to_clipping))

print("\nRescued household IDs:")
print(rescued_by_clipping)

assert len(lost_due_to_clipping) == 0, (
    "Clipping unexpectedly removed an already-active household."
)

In [ ]:
# CELL 6 [CODE] — GHOST PURGE, OUTLIER CLIPPING & LOCKED MODEL DEFINITION
# This cell defines the three objects used by ALL downstream cells:
#   scaler, X_clean, optimal_kmeans, CLUSTER_NAME_MAP
# ─────────────────────────────────────────────────────────────────────────────
CORE_FEATURES = [
    'evening_peak_ratio', 'night_baseload_ratio', 'load_uniformity',
    'load_factor', 'leisure_usage_ratio', 'log_daily_kwh'
]

# -------------------------------------------------------------------------
# Correct ordering: apply the substantive activity rule to RAW consumption
# before any percentile clipping.
# -------------------------------------------------------------------------
active_mask = feature_matrix['log_daily_kwh'] > 0.05
clean_fm = feature_matrix.loc[active_mask, CORE_FEATURES].copy()

print(
    f"[AUDIT] Raw ghost purge removed {(~active_mask).sum()} "
    "null/vacant households."
)
print(f"[AUDIT] Active population before clipping: {len(clean_fm)}")

# Calculate percentile limits only from the surviving active population.
# These locked bounds are reused by later sensitivity/temporal cells.
CLIP_BOUNDS = {}

for col in CORE_FEATURES:
    lower = float(clean_fm[col].quantile(0.01))
    upper = float(clean_fm[col].quantile(0.99))

    if not np.isfinite(lower) or not np.isfinite(upper) or lower > upper:
        raise RuntimeError(
            f"Invalid clipping bounds for {col}: lower={lower}, upper={upper}"
        )

    CLIP_BOUNDS[col] = (lower, upper)
    clean_fm[col] = clean_fm[col].clip(lower=lower, upper=upper)

# The activity rule must remain true after clipping because it was evaluated
# on the raw feature matrix before the lower-tail winsorisation.
assert (feature_matrix.loc[clean_fm.index, 'log_daily_kwh'] > 0.05).all()

print("[AUDIT] 1st–99th percentile clipping applied after the ghost purge.")
print(f"[AUDIT] Final active clustering population: {len(clean_fm)}")

# THE SINGLE LOCKED SCALER — fitted once on ghost-purged, clipped data
# All downstream validation and temporal cells must use scaler.transform(),
# never fit_transform(), when assigning households to the locked final KMeans model.
scaler = StandardScaler()
X_clean = scaler.fit_transform(clean_fm[CORE_FEATURES])

# THE LOCKED MODEL — fitted once, used for all validation and temporal analysis
OPTIMAL_K = 4
RANDOM_STATE = 42
OPTIMAL_N_INIT = 20
optimal_kmeans = KMeans(
    n_clusters=OPTIMAL_K,
    random_state=RANDOM_STATE,
    n_init=OPTIMAL_N_INIT
)
clean_fm['final_cluster'] = optimal_kmeans.fit_predict(X_clean)

assert clean_fm.index.is_unique, "Final feature matrix index must be unique."
assert clean_fm[CORE_FEATURES].notna().all().all(), "Final features contain NaN values."
assert np.isfinite(clean_fm[CORE_FEATURES].to_numpy()).all(), "Final features contain non-finite values."
assert set(clean_fm.index).issubset(set(feature_matrix.index)), (
    "Final active population is not a subset of the raw feature matrix."
)

# -------------------------------------------------------------------------
# Dynamic archetype naming
# -------------------------------------------------------------------------
# KMeans numeric labels are arbitrary and may swap after a rerun.
# Therefore, archetype names are assigned from the median feature profile,
# not from hard-coded IDs.

def infer_archetype_name_map(fm, cluster_col='final_cluster'):
    med = fm.groupby(cluster_col)[CORE_FEATURES].median()
    labels = {}
    remaining = set(med.index.astype(int))

    # Heavy Continuous: strongest continuous/high-load signature.
    heavy_id = int(med['load_factor'].idxmax())
    labels[heavy_id] = 'Heavy Continuous'
    remaining.remove(heavy_id)

    # Peak-Exposed: strongest evening concentration among remaining clusters.
    rem = med.loc[list(remaining)]
    peak_id = int(rem['evening_peak_ratio'].idxmax())
    labels[peak_id] = 'Peak-Exposed'
    remaining.remove(peak_id)

    # Daytime-Active: weakest night baseload among remaining clusters.
    rem = med.loc[list(remaining)]
    daytime_id = int(rem['night_baseload_ratio'].idxmin())
    labels[daytime_id] = 'Daytime-Active'
    remaining.remove(daytime_id)

    # The remaining cluster is the modal/balanced household group.
    typical_id = int(next(iter(remaining)))
    labels[typical_id] = 'Typical Household'

    return labels, med

CLUSTER_NAME_MAP, CLUSTER_MEDIANS_AUDIT = infer_archetype_name_map(clean_fm)
ARCHETYPE_ORDER = ['Peak-Exposed', 'Typical Household', 'Heavy Continuous', 'Daytime-Active']
CLUSTER_ORDER = [cid for name in ARCHETYPE_ORDER for cid, nm in CLUSTER_NAME_MAP.items() if nm == name]

ARCHETYPE_COLOR_MAP = {
    'Peak-Exposed': '#e74c3c',
    'Typical Household': '#3498db',
    'Heavy Continuous': '#2ecc71',
    'Daytime-Active': '#95a5a6'
}
CLUSTER_COLOR_MAP = {cid: ARCHETYPE_COLOR_MAP[name] for cid, name in CLUSTER_NAME_MAP.items()}

cluster_counts = clean_fm['final_cluster'].value_counts().to_dict()
CLUSTER_LABEL_MAP = {
    cid: f"{CLUSTER_NAME_MAP[cid]} ({cluster_counts.get(cid, 0) / len(clean_fm) * 100:.1f}%)"
    for cid in sorted(CLUSTER_NAME_MAP)
}

# Readable ID + archetype name used by older plotting/audit cells.
c_names = {cid: f"Cluster {cid}: {CLUSTER_NAME_MAP[cid]}" for cid in sorted(CLUSTER_NAME_MAP)}
clean_fm['cluster_name'] = clean_fm['final_cluster'].map(CLUSTER_NAME_MAP)

print("\n--- DYNAMIC CLUSTER ID → ARCHETYPE MAP ---")
for c in sorted(CLUSTER_NAME_MAP):
    print(f"  Cluster {c}: {CLUSTER_NAME_MAP[c]}")

print("\n--- K=4 CLUSTER POPULATIONS ---")
for c in CLUSTER_ORDER:
    mask = clean_fm['final_cluster'] == c
    print(f"  {c_names[c]:<35} | {mask.sum():>4} HHs "
          f"| {(mask.sum()/len(clean_fm))*100:>5.1f}%")

print("\n--- CLUSTER MEDIAN AUDIT USED FOR NAMING ---")
display(
    CLUSTER_MEDIANS_AUDIT
    .assign(archetype=[CLUSTER_NAME_MAP[int(i)] for i in CLUSTER_MEDIANS_AUDIT.index])
    .sort_values('archetype')
    .round(3)
)

# Export cluster labels for secondary analysis notebooks
clean_fm.to_pickle('clean_fm_clusters.pkl')
print(f"Exported {len(clean_fm)} household cluster labels with corrected archetype names.")


In [ ]:
# ============================================================
# LOCKED MODEL-INPUT AND ROW-ALIGNMENT AUDIT
# Confirms that every supplementary model receives exactly the same
# six clipped features, scaling transformation, household order and KMeans labels.
# ============================================================

MODEL_HOUSEHOLD_ORDER = clean_fm.index.astype(str).to_numpy(copy=True)
MODEL_FEATURE_ORDER = tuple(CORE_FEATURES)

if X_clean.ndim != 2:
    raise RuntimeError("X_clean must be a two-dimensional matrix.")
if X_clean.shape != (len(clean_fm), len(CORE_FEATURES)):
    raise RuntimeError(
        "X_clean shape does not match clean_fm rows and CORE_FEATURES."
    )
if not np.isfinite(X_clean).all():
    raise RuntimeError("X_clean contains NaN or infinite values.")
if list(clean_fm[CORE_FEATURES].columns) != CORE_FEATURES:
    raise RuntimeError("Locked feature-column order changed unexpectedly.")

X_clean_recomputed = scaler.transform(
    clean_fm.loc[:, CORE_FEATURES]
)
if not np.allclose(
    X_clean,
    X_clean_recomputed,
    rtol=1e-10,
    atol=1e-12
):
    raise RuntimeError(
        "X_clean is not the scaler transformation of clean_fm[CORE_FEATURES]."
    )

locked_labels_from_model = np.asarray(
    optimal_kmeans.labels_,
    dtype=int
)
locked_labels_from_frame = (
    clean_fm["final_cluster"]
    .astype(int)
    .to_numpy()
)
if not np.array_equal(
    locked_labels_from_model,
    locked_labels_from_frame
):
    raise RuntimeError(
        "clean_fm final_cluster labels are not aligned with optimal_kmeans.labels_."
    )

model_input_audit = pd.DataFrame([{
    "households": len(clean_fm),
    "features": len(CORE_FEATURES),
    "feature_order": " | ".join(CORE_FEATURES),
    "finite_scaled_values": bool(np.isfinite(X_clean).all()),
    "scaler_transform_exactly_reproduced": True,
    "kmeans_label_order_aligned": True,
    "unique_household_index": bool(clean_fm.index.is_unique)
}])

print("Locked model-input audit:")
display(model_input_audit)
del X_clean_recomputed


In [ ]:
# ============================================================
# OPTIONAL K-MEANS PARAMETER SENSITIVITY CHECK
# Diagnostic only — DOES NOT change the locked final model
#
# Locked model:
#   K = 4
#   n_init = 20
#   random_state = 42
#
# Checks:
#   init
#   max_iter
#   tol
#   algorithm
# ============================================================

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score,
    silhouette_score
)

# ------------------------------------------------------------
# 1. Locked final solution
# ------------------------------------------------------------

reference_labels = np.asarray(optimal_kmeans.labels_)

print("=" * 75)
print("LOCKED FINAL K-MEANS PARAMETERS")
print("=" * 75)

locked_params = optimal_kmeans.get_params()

for p in [
    "n_clusters",
    "init",
    "n_init",
    "max_iter",
    "tol",
    "algorithm",
    "random_state"
]:
    print(f"{p:15s}: {locked_params[p]}")


# ------------------------------------------------------------
# 2. Helper function
# ------------------------------------------------------------

results = []


def evaluate_kmeans_variant(test_name, **changed_params):

    # Start from the SAME locked configuration
    params = {
        "n_clusters": 4,
        "n_init": 20,
        "random_state": 42,
        "init": locked_params["init"],
        "max_iter": locked_params["max_iter"],
        "tol": locked_params["tol"],
        "algorithm": locked_params["algorithm"],
    }

    # Change ONLY the parameter being tested
    params.update(changed_params)

    model = KMeans(**params)

    labels = model.fit_predict(X_clean)

    ari = adjusted_rand_score(
        reference_labels,
        labels
    )

    silhouette = silhouette_score(
        X_clean,
        labels
    )

    changed_households = np.sum(
        labels != reference_labels
    )

    # NOTE:
    # numeric cluster IDs can swap even if the partition is identical.
    # Therefore ARI is the important comparison, not raw label equality.

    results.append({
        "test": test_name,
        "init": str(params["init"]),
        "max_iter": params["max_iter"],
        "tol": params["tol"],
        "algorithm": params["algorithm"],
        "ARI_vs_locked": ari,
        "silhouette": silhouette,
        "inertia": model.inertia_,
        "iterations_used": model.n_iter_,
    })


# ------------------------------------------------------------
# 3. Baseline — should reproduce locked model
# ------------------------------------------------------------

evaluate_kmeans_variant(
    "LOCKED BASELINE"
)


# ------------------------------------------------------------
# 4. INIT sensitivity
# ------------------------------------------------------------

evaluate_kmeans_variant(
    "init = k-means++",
    init="k-means++"
)

evaluate_kmeans_variant(
    "init = random",
    init="random"
)


# ------------------------------------------------------------
# 5. MAX_ITER sensitivity
# ------------------------------------------------------------

for value in [10, 50, 100, 300, 1000]:

    evaluate_kmeans_variant(
        f"max_iter = {value}",
        max_iter=value
    )


# ------------------------------------------------------------
# 6. TOLERANCE sensitivity
# ------------------------------------------------------------

for value in [1e-2, 1e-3, 1e-4, 1e-5, 1e-6]:

    evaluate_kmeans_variant(
        f"tol = {value}",
        tol=value
    )


# ------------------------------------------------------------
# 7. ALGORITHM sensitivity
# ------------------------------------------------------------

for value in ["lloyd", "elkan"]:

    evaluate_kmeans_variant(
        f"algorithm = {value}",
        algorithm=value
    )


# ------------------------------------------------------------
# 8. Results
# ------------------------------------------------------------

sensitivity_results = pd.DataFrame(results)

print("\n" + "=" * 75)
print("PARAMETER SENSITIVITY RESULTS")
print("=" * 75)

display(
    sensitivity_results.round({
        "ARI_vs_locked": 4,
        "silhouette": 4,
        "inertia": 2
    })
)


# ------------------------------------------------------------
# 9. Easy interpretation
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("INTERPRETATION")
print("=" * 75)

for _, row in sensitivity_results.iterrows():

    ari = row["ARI_vs_locked"]

    if ari >= 0.99:
        interpretation = "Virtually identical clustering"
    elif ari >= 0.90:
        interpretation = "Very similar clustering"
    elif ari >= 0.75:
        interpretation = "Some sensitivity"
    else:
        interpretation = "Substantial sensitivity"

    print(
        f"{row['test']:25s} "
        f"ARI={ari:.4f}  ->  {interpretation}"
    )


print("\nIMPORTANT:")
print("This cell is diagnostic only.")
print("It does NOT modify optimal_kmeans or clean_fm['final_cluster'].")

In [ ]:
# ============================================================
# PERCENTILE CLIPPING NECESSITY AUDIT
# Compare:
#   1. Raw active features without clipping
#   2. Active features with 1st–99th percentile clipping
# ============================================================

import numpy as np
import pandas as pd

from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    adjusted_rand_score,
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

# ------------------------------------------------------------
# 1. Use exactly the same active household population
# ------------------------------------------------------------
raw_active = feature_matrix.loc[
    feature_matrix['log_daily_kwh'] > 0.05,
    CORE_FEATURES
].copy()

assert raw_active.index.is_unique
assert raw_active.notna().all().all()

print("Active population used in both variants:", len(raw_active))


# ------------------------------------------------------------
# 2. Construct the clipped variant
# ------------------------------------------------------------
clipped_active = raw_active.copy()

clipping_audit = []
affected_any_feature = pd.Series(
    False,
    index=raw_active.index
)

audit_clip_bounds = {}

for col in CORE_FEATURES:
    lower = float(raw_active[col].quantile(0.01))
    upper = float(raw_active[col].quantile(0.99))

    audit_clip_bounds[col] = (lower, upper)

    affected = (
        (raw_active[col] < lower) |
        (raw_active[col] > upper)
    )

    affected_any_feature = affected_any_feature | affected

    clipping_audit.append({
        'feature': col,
        'lower_1pct': lower,
        'upper_99pct': upper,
        'households_affected': int(affected.sum()),
        'percentage_affected': affected.mean() * 100,
        'raw_min': raw_active[col].min(),
        'raw_max': raw_active[col].max()
    })

    clipped_active[col] = clipped_active[col].clip(
        lower=lower,
        upper=upper
    )

clipping_audit = pd.DataFrame(clipping_audit)

print("\nClipping limits and affected observations:")
display(clipping_audit.round(6))

print(
    "\nUnique households affected in at least one feature:",
    int(affected_any_feature.sum())
)

print(
    "Percentage affected in at least one feature:",
    round(affected_any_feature.mean() * 100, 4),
    "%"
)


# ------------------------------------------------------------
# 3. Fit and evaluate one preprocessing variant
# ------------------------------------------------------------
def evaluate_variant(feature_data, variant_name):
    variant_scaler = StandardScaler()
    X_variant = variant_scaler.fit_transform(
        feature_data[CORE_FEATURES]
    )

    model_20 = KMeans(
        n_clusters=4,
        random_state=42,
        n_init=20
    )

    labels_20 = model_20.fit_predict(X_variant)

    model_10 = KMeans(
        n_clusters=4,
        random_state=42,
        n_init=10
    )

    labels_10 = model_10.fit_predict(X_variant)

    # Seed stability using the same n_init value
    seed_label_sets = []

    for seed in range(20):
        seed_model = KMeans(
            n_clusters=4,
            random_state=seed,
            n_init=20
        )

        seed_label_sets.append(
            seed_model.fit_predict(X_variant)
        )

    pairwise_seed_ari = []

    for i in range(len(seed_label_sets)):
        for j in range(i + 1, len(seed_label_sets)):
            pairwise_seed_ari.append(
                adjusted_rand_score(
                    seed_label_sets[i],
                    seed_label_sets[j]
                )
            )

    return {
        'variant': variant_name,
        'X': X_variant,
        'labels': labels_20,
        'model': model_20,
        'silhouette': silhouette_score(
            X_variant,
            labels_20
        ),
        'davies_bouldin': davies_bouldin_score(
            X_variant,
            labels_20
        ),
        'calinski_harabasz': calinski_harabasz_score(
            X_variant,
            labels_20
        ),
        'inertia': model_20.inertia_,
        'ninit_10_vs_20_ari': adjusted_rand_score(
            labels_10,
            labels_20
        ),
        'seed_mean_ari': np.mean(pairwise_seed_ari),
        'seed_min_ari': np.min(pairwise_seed_ari),
        'smallest_cluster': np.bincount(labels_20).min(),
        'largest_cluster': np.bincount(labels_20).max()
    }


no_clip_result = evaluate_variant(
    raw_active,
    'No clipping'
)

clip_result = evaluate_variant(
    clipped_active,
    '1st–99th clipping'
)


# ------------------------------------------------------------
# 4. Compare model-quality and stability metrics
# ------------------------------------------------------------
comparison_metrics = pd.DataFrame([
    {
        'variant': no_clip_result['variant'],
        'silhouette': no_clip_result['silhouette'],
        'davies_bouldin': no_clip_result['davies_bouldin'],
        'calinski_harabasz':
            no_clip_result['calinski_harabasz'],
        'inertia': no_clip_result['inertia'],
        'ninit_10_vs_20_ari':
            no_clip_result['ninit_10_vs_20_ari'],
        'seed_mean_ari':
            no_clip_result['seed_mean_ari'],
        'seed_min_ari':
            no_clip_result['seed_min_ari'],
        'smallest_cluster':
            no_clip_result['smallest_cluster'],
        'largest_cluster':
            no_clip_result['largest_cluster']
    },
    {
        'variant': clip_result['variant'],
        'silhouette': clip_result['silhouette'],
        'davies_bouldin': clip_result['davies_bouldin'],
        'calinski_harabasz':
            clip_result['calinski_harabasz'],
        'inertia': clip_result['inertia'],
        'ninit_10_vs_20_ari':
            clip_result['ninit_10_vs_20_ari'],
        'seed_mean_ari':
            clip_result['seed_mean_ari'],
        'seed_min_ari':
            clip_result['seed_min_ari'],
        'smallest_cluster':
            clip_result['smallest_cluster'],
        'largest_cluster':
            clip_result['largest_cluster']
    }
]).set_index('variant')

print("\nClipping versus no-clipping comparison:")
display(comparison_metrics.round(6))


# ------------------------------------------------------------
# 5. Compare the two final partitions
# ------------------------------------------------------------
labels_raw = no_clip_result['labels']
labels_clipped = clip_result['labels']

partition_ari = adjusted_rand_score(
    labels_raw,
    labels_clipped
)

cross_tab = pd.crosstab(
    pd.Series(labels_raw, name='No clipping'),
    pd.Series(labels_clipped, name='1st–99th clipping')
)

row_ind, col_ind = linear_sum_assignment(
    -cross_tab.to_numpy()
)

raw_to_clipped_mapping = {
    cross_tab.index[row]: cross_tab.columns[col]
    for row, col in zip(row_ind, col_ind)
}

labels_raw_aligned = np.array([
    raw_to_clipped_mapping[label]
    for label in labels_raw
])

changed_mask = labels_raw_aligned != labels_clipped

print("\nOptimally aligned cross-tabulation:")
display(cross_tab)

print(
    "ARI between clipped and non-clipped partitions:",
    round(partition_ari, 6)
)

print(
    "Households whose assignments changed:",
    int(changed_mask.sum())
)

print(
    "Percentage whose assignments changed:",
    round(changed_mask.mean() * 100, 4),
    "%"
)


# ------------------------------------------------------------
# 6. Compare cluster populations after label alignment
# ------------------------------------------------------------
cluster_size_comparison = pd.DataFrame({
    'No clipping': pd.Series(
        labels_raw_aligned
    ).value_counts().sort_index(),

    '1st–99th clipping': pd.Series(
        labels_clipped
    ).value_counts().sort_index()
}).fillna(0).astype(int)

print("\nCluster-size comparison:")
display(cluster_size_comparison)


# ------------------------------------------------------------
# 7. Compare substantive cluster profiles using raw feature units
# ------------------------------------------------------------
raw_profile_data = raw_active.copy()
raw_profile_data['aligned_cluster'] = labels_raw_aligned

clipped_profile_data = raw_active.copy()
clipped_profile_data['aligned_cluster'] = labels_clipped

raw_cluster_medians = (
    raw_profile_data
    .groupby('aligned_cluster')[CORE_FEATURES]
    .median()
)

clipped_cluster_medians = (
    clipped_profile_data
    .groupby('aligned_cluster')[CORE_FEATURES]
    .median()
)

profile_comparison = pd.concat(
    {
        'No clipping': raw_cluster_medians,
        '1st–99th clipping': clipped_cluster_medians
    },
    names=['variant', 'cluster']
)

print("\nCluster median profiles in original feature units:")
display(profile_comparison.round(4))


# ------------------------------------------------------------
# 8. List changed households
# ------------------------------------------------------------
changed_household_ids = raw_active.index[changed_mask]

print("\nChanged household IDs:")
print(changed_household_ids.tolist())

if len(changed_household_ids) > 0:
    changed_household_features = raw_active.loc[
        changed_household_ids,
        CORE_FEATURES
    ].copy()

    changed_household_features[
        'no_clipping_cluster'
    ] = labels_raw_aligned[changed_mask]

    changed_household_features[
        'clipped_cluster'
    ] = labels_clipped[changed_mask]

    display(changed_household_features)

### Private diagnostic omitted — Borderline household case audit

This source cell concerned identified households and was intentionally withheld from the public review draft. It remains unchanged in the privately retained notebook. The original case-specific analysis cannot be reproduced from this public copy alone.

In [ ]:
# ============================================================
# POPULATION-WIDE ACTIVITY PERSISTENCE AUDIT
# Diagnostic only — does not change the clustering population
# ============================================================

import numpy as np
import pandas as pd

# Daily consumption using only valid household-days
valid_daily_consumption = (
    feature_df
    .groupby(["household_ID", "date_only"], observed=True)["true_15min_kwh"]
    .sum(min_count=1)
)

activity_summary = (
    valid_daily_consumption
    .groupby("household_ID", observed=True)
    .agg(
        mean_daily_kwh="mean",
        median_daily_kwh="median",
        valid_days="size",
        positive_days=lambda x: (x > 0).sum(),
        days_above_0_01=lambda x: (x > 0.01).sum(),
        days_above_0_05=lambda x: (x > 0.05).sum()
    )
)

activity_summary["positive_day_ratio"] = (
    activity_summary["positive_days"]
    / activity_summary["valid_days"]
)

activity_summary["ratio_days_above_0_01"] = (
    activity_summary["days_above_0_01"]
    / activity_summary["valid_days"]
)

activity_summary["ratio_days_above_0_05"] = (
    activity_summary["days_above_0_05"]
    / activity_summary["valid_days"]
)

activity_summary["log_daily_kwh_raw"] = np.log1p(
    activity_summary["mean_daily_kwh"]
)

# Existing ghost rule
mean_rule = activity_summary["log_daily_kwh_raw"] > 0.05

print("Total eligible households:", len(activity_summary))
print("Active under current mean-only rule:", int(mean_rule.sum()))
print("Ghosts under current mean-only rule:", int((~mean_rule).sum()))

# Test persistence thresholds without applying them
threshold_results = []

for persistence_threshold in [0.25, 0.40, 0.50, 0.60, 0.75]:
    combined_rule = (
        mean_rule
        & (
            activity_summary["positive_day_ratio"]
            >= persistence_threshold
        )
    )

    threshold_results.append({
        "minimum_positive_day_ratio": persistence_threshold,
        "active_households": int(combined_rule.sum()),
        "households_removed_from_current_active_set": int(
            (mean_rule & ~combined_rule).sum()
        ),
        "percentage_of_current_active_removed": (
            (mean_rule & ~combined_rule).sum()
            / mean_rule.sum()
            * 100
        )
    })

threshold_results = pd.DataFrame(threshold_results)

print("\nPersistence-threshold sensitivity:")
display(threshold_results.round(4))

# Current active households with positive use on fewer than 50% of days
low_persistence_active = activity_summary[
    mean_rule
    & (activity_summary["positive_day_ratio"] < 0.50)
].sort_values(
    ["positive_day_ratio", "mean_daily_kwh"]
)

print(
    "\nCurrent active households with positive consumption "
    "on fewer than 50% of valid days:",
    len(low_persistence_active)
)

# Household-indexed table suppressed in this public draft.
# This population-wide analysis reports aggregate counts above.

# Private, case-specific percentile lookup omitted in this public draft.


### Private diagnostic omitted — One-household exclusion sensitivity refit

This source cell concerned identified households and was intentionally withheld from the public review draft. It remains unchanged in the privately retained notebook. The original case-specific analysis cannot be reproduced from this public copy alone.

### Private diagnostic omitted — Case-specific before-and-after clustering comparison

This source cell concerned identified households and was intentionally withheld from the public review draft. It remains unchanged in the privately retained notebook. The original case-specific analysis cannot be reproduced from this public copy alone.

In [ ]:
# ============================================================
# COMPREHENSIVE n_init SENSITIVITY ANALYSIS
# Diagnostic only — does not modify the locked final model
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

N_INIT_VALUES = [1, 5, 10, 20, 50, 100]
RANDOM_SEEDS = list(range(20))

# A high-restart anchor is used for partition comparison only. It is not
# described as a proven global optimum because another seed/restart setting
# may find a marginally lower inertia.
high_restart_anchor = KMeans(
    n_clusters=OPTIMAL_K,
    n_init=200,
    random_state=RANDOM_STATE
)
anchor_labels = high_restart_anchor.fit_predict(X_clean)
anchor_inertia = float(high_restart_anchor.inertia_)

n_init_records = []
n_init_label_sets = {}

for n_init_value in N_INIT_VALUES:
    labels_for_setting = []
    inertias = []
    silhouettes = []
    aris_to_anchor = []
    runtimes = []

    for seed in RANDOM_SEEDS:
        start_time = time.perf_counter()

        model = KMeans(
            n_clusters=OPTIMAL_K,
            n_init=n_init_value,
            random_state=seed
        )
        labels_setting = model.fit_predict(X_clean)
        elapsed_time = time.perf_counter() - start_time

        labels_for_setting.append(labels_setting)
        inertias.append(float(model.inertia_))
        silhouettes.append(float(silhouette_score(X_clean, labels_setting)))
        aris_to_anchor.append(float(adjusted_rand_score(anchor_labels, labels_setting)))
        runtimes.append(elapsed_time)

    n_init_label_sets[n_init_value] = labels_for_setting

    pairwise_aris = [
        adjusted_rand_score(labels_for_setting[i], labels_for_setting[j])
        for i in range(len(labels_for_setting))
        for j in range(i + 1, len(labels_for_setting))
    ]

    n_init_records.append({
        'n_init': n_init_value,
        'mean_inertia': np.mean(inertias),
        'minimum_inertia': np.min(inertias),
        'maximum_inertia': np.max(inertias),
        'inertia_std': np.std(inertias),
        'mean_silhouette': np.mean(silhouettes),
        'mean_ari_to_high_restart_anchor': np.mean(aris_to_anchor),
        'minimum_ari_to_high_restart_anchor': np.min(aris_to_anchor),
        'mean_pairwise_seed_ari': np.mean(pairwise_aris),
        'minimum_pairwise_seed_ari': np.min(pairwise_aris),
        'mean_runtime_seconds': np.mean(runtimes)
    })

n_init_results = pd.DataFrame(n_init_records).set_index('n_init')
best_observed_inertia = min(
    anchor_inertia,
    float(n_init_results['minimum_inertia'].min())
)
n_init_results['best_inertia_gap_from_observed_best'] = (
    n_init_results['minimum_inertia'] - best_observed_inertia
)
n_init_results['relative_gap_percent'] = (
    n_init_results['best_inertia_gap_from_observed_best']
    / best_observed_inertia * 100
)

print('High-restart anchor n_init: 200')
print('High-restart anchor inertia:', anchor_inertia)
print('Best inertia observed across all tested runs:', best_observed_inertia)
print('\nComprehensive n_init sensitivity results:')
display(n_init_results.round(6))

print('\nLocked decision: n_init=20 is retained as the smallest value at the practical')
print('stability/inertia plateau; larger values are sensitivity checks, not a change')
print('in model complexity and not an overfitting/underfitting control.')


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
import numpy as np

kmeans_10 = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_10 = kmeans_10.fit_predict(X_clean)

kmeans_20 = KMeans(n_clusters=4, random_state=42, n_init=20)
labels_20 = kmeans_20.fit_predict(X_clean)

print("Same label sequence:", np.array_equal(labels_10, labels_20))
print("ARI between n_init=10 and n_init=20:", adjusted_rand_score(labels_10, labels_20))
print("Inertia n_init=10:", kmeans_10.inertia_)
print("Inertia n_init=20:", kmeans_20.inertia_)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("--- GENERATING DYNAMIC CLUSTER MEDIAN FEATURE HEATMAP ---")

feature_short_names = {
    'evening_peak_ratio': 'EPR',
    'night_baseload_ratio': 'NBR',
    'load_uniformity': 'LU',
    'load_factor': 'LF',
    'leisure_usage_ratio': 'LUR',
    'log_daily_kwh': 'LDK'
}

cluster_medians_plot = (
    clean_fm.groupby('final_cluster')[CORE_FEATURES]
    .median()
    .rename(index=CLUSTER_NAME_MAP, columns=feature_short_names)
    .reindex(ARCHETYPE_ORDER)
)

if cluster_medians_plot.isna().any().any():
    raise RuntimeError("Dynamic cluster median table contains missing values.")

feature_std = cluster_medians_plot.std(axis=0, ddof=0).replace(0, 1.0)
heatmap_values = (
    cluster_medians_plot - cluster_medians_plot.mean(axis=0)
) / feature_std

plt.figure(figsize=(8.5, 3.6))
sns.heatmap(
    heatmap_values,
    annot=cluster_medians_plot,
    fmt='.3f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5,
    cbar_kws={'label': 'Relative feature level'}
)
plt.title('Cluster Median Feature Profiles')
plt.xlabel('Engineered Feature')
plt.ylabel('Archetype')
plt.tight_layout()
plt.savefig('cluster_feature_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================
# CHECK TELEMETRY STABILITY PERIOD: FEB–APR 2024
# Memory-safe two-column coverage audit.
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt

if "clean_df" in globals():
    source_name = "clean_df"
    source_df = clean_df
elif "merged_df" in globals():
    source_name = "merged_df"
    source_df = merged_df
elif "golden_df" in globals():
    source_name = "golden_df"
    source_df = golden_df
    print(
        "WARNING: golden_df is May–November only and cannot verify "
        "February–April coverage."
    )
else:
    raise NameError("Run the master pipeline before the telemetry coverage audit.")

required_cols = {"household_ID", "meter_dt"}
missing_cols = required_cols.difference(source_df.columns)
if missing_cols:
    raise ValueError(
        f"{source_name} is missing required columns: {sorted(missing_cols)}"
    )

# The master pipeline already parsed meter_dt. Restrict first, then copy only
# the two required columns; do not duplicate the full telemetry dataframe.
mask_2024 = (
    source_df["meter_dt"].ge(pd.Timestamp("2024-01-01"))
    & source_df["meter_dt"].lt(pd.Timestamp("2025-01-01"))
)
df_2024 = source_df.loc[
    mask_2024,
    ["household_ID", "meter_dt"]
].copy()

if df_2024.empty:
    raise RuntimeError(f"No 2024 rows were found in {source_name}.")

df_2024["date"] = df_2024["meter_dt"].dt.normalize()
df_2024["month"] = df_2024["meter_dt"].dt.to_period("M")

print(f"Using dataframe: {source_name}")
print(f"2024 rows checked: {len(df_2024):,}")
print(
    "2024 unique households checked:",
    f"{df_2024['household_ID'].nunique():,}"
)
print(
    "2024 date range checked:",
    f"{df_2024['meter_dt'].min()} to {df_2024['meter_dt'].max()}"
)

monthly_active = (
    df_2024
    .groupby("month", observed=True)["household_ID"]
    .nunique()
    .reset_index(name="monthly_unique_households")
)

daily_active = (
    df_2024
    .groupby("date", observed=True)["household_ID"]
    .nunique()
    .reset_index(name="daily_active_households")
)
daily_active["month"] = daily_active["date"].dt.to_period("M")

daily_month_summary = (
    daily_active
    .groupby("month", observed=True)["daily_active_households"]
    .agg(
        daily_min="min",
        daily_median="median",
        daily_mean="mean",
        daily_max="max"
    )
    .reset_index()
)

coverage_2024_summary = monthly_active.merge(
    daily_month_summary,
    on="month",
    how="left",
    validate="one_to_one"
)

print("\n--- MONTHLY AND DAILY ACTIVE HOUSEHOLD COVERAGE, 2024 ---")
display(coverage_2024_summary)

focus_months = pd.PeriodIndex(
    ["2024-02", "2024-03", "2024-04", "2024-05"],
    freq="M"
)
focus = coverage_2024_summary[
    coverage_2024_summary["month"].isin(focus_months)
].copy()

print("\n--- FOCUSED CHECK: FEBRUARY TO MAY 2024 ---")
display(focus)

low_threshold = 1000
low_days = daily_active[
    daily_active["daily_active_households"] <= low_threshold
].copy()

print(f"\n--- DAYS WITH <= {low_threshold} ACTIVE HOUSEHOLDS ---")
if low_days.empty:
    print(f"No days with <= {low_threshold} active households.")
else:
    display(low_days)

print("\nLowest 20 daily active household counts:")
display(
    daily_active
    .sort_values("daily_active_households")
    .head(20)
)

plt.figure(figsize=(10, 4))
plt.plot(
    coverage_2024_summary["month"].astype(str),
    coverage_2024_summary["monthly_unique_households"],
    marker="o"
)
plt.xticks(rotation=45)
plt.ylabel("Monthly unique households")
plt.xlabel("Month")
plt.title("Monthly Active Household Coverage in 2024")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(
    daily_active["date"],
    daily_active["daily_active_households"]
)
plt.axvspan(
    pd.Timestamp("2024-02-01"),
    pd.Timestamp("2024-04-01"),
    alpha=0.2,
    label="Feb–Mar telemetry instability check"
)
plt.axvspan(
    pd.Timestamp("2024-04-01"),
    pd.Timestamp("2024-05-01"),
    alpha=0.10,
    label="April recovery/buffer check"
)
plt.axvline(
    pd.Timestamp("2024-05-01"),
    linestyle="--",
    label="Golden window start: 2024-05-01"
)
plt.ylabel("Daily active households")
plt.xlabel("Date")
plt.title("Daily Active Household Coverage in 2024")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n--- INTERPRETATION GUIDE ---")
month_medians = {}
for period in focus_months:
    values = focus.loc[
        focus["month"].eq(period),
        "daily_median"
    ]
    key = str(period)
    if values.empty:
        month_medians[key] = None
        print(f"{key}: not available")
    else:
        month_medians[key] = float(values.iloc[0])
        print(
            f"{key} daily median active HH: "
            f"{month_medians[key]:.0f}"
        )

feb = month_medians.get("2024-02")
mar = month_medians.get("2024-03")
apr = month_medians.get("2024-04")
may = month_medians.get("2024-05")

if all(value is not None for value in [feb, mar, apr, may]):
    if feb <= low_threshold or mar <= low_threshold:
        if apr > low_threshold:
            print(
                "February–March 2024 shows telemetry instability; "
                "May–November is retained as a conservative post-instability "
                "golden window."
            )
        else:
            print(
                "February–April 2024 shows low coverage; May–November is "
                "retained as the stable golden window."
            )
    else:
        print(
            "This audit does not show a strong February–March reduction "
            "under the selected threshold; interpret the table directly."
        )

print("\nPaper-safe sentence:")
print(
    "The final clustering period was restricted to May–November 2024 "
    "to avoid the earlier telemetry-instability period and immediate "
    "post-disruption recovery effects."
)

# Release the large row-level audit subset; summary tables remain available.
del df_2024, mask_2024


In [ ]:
# CELL 7 [CODE] — 24-HOUR LOAD PROFILE VISUALISATION
# Household-balanced profiles from the valid-day-only 96-slot matrix.
# ============================================================

print("--- GENERATING FINAL K=4 CEB LOAD PROFILES ---")

required_objects = ['interval_profiles', 'clean_fm', 'CLUSTER_NAME_MAP']
missing = [name for name in required_objects if name not in globals()]
if missing:
    raise RuntimeError(f"Missing required objects: {missing}. Run the master pipeline first.")


def build_household_balanced_cluster_profiles():
    profile_matrix = interval_profiles.reindex(clean_fm.index).copy()
    if profile_matrix.isna().any().any():
        missing_hhs = profile_matrix.index[profile_matrix.isna().any(axis=1)].tolist()
        raise RuntimeError(
            "Final clustered households have incomplete 96-slot profiles: "
            f"{missing_hhs[:10]}"
        )

    profile_matrix['final_cluster'] = clean_fm['final_cluster'].astype(int)
    profiles = profile_matrix.groupby('final_cluster').mean().T
    profiles = profiles.reindex(index=range(96), columns=CLUSTER_ORDER)

    if profiles.isna().any().any():
        raise RuntimeError("Cluster load-profile matrix is incomplete.")
    return profiles


cluster_profiles = build_household_balanced_cluster_profiles()
time_axis = np.arange(96) / 4.0
cluster_names_pct = {
    cid: f"Cluster {cid}: {CLUSTER_LABEL_MAP[cid]}"
    for cid in CLUSTER_ORDER
}

fig_load_profiles_overview, ax = plt.subplots(figsize=(14, 7))
for c_id in CLUSTER_ORDER:
    ax.plot(
        time_axis,
        cluster_profiles[c_id].to_numpy(),
        label=cluster_names_pct[c_id],
        linewidth=3 if CLUSTER_NAME_MAP[c_id] == 'Peak-Exposed' else 2,
        color=CLUSTER_COLOR_MAP[c_id]
    )

ax.axvspan(
    18.5, 22.5,
    color='red', alpha=0.1,
    label='CEB evening system peak window (18:30–22:30)'
)
ax.set_title('Sri Lankan Residential Electricity Archetypes (K=4 K-Means)',
             fontsize=16, fontweight='bold')
ax.set_xlabel('Hour of day (0 = midnight, 12 = noon)', fontsize=12)
ax.set_ylabel('Household-balanced mean 15-min kWh', fontsize=12)
ax.set_xlim(0, 24)
ax.set_xticks(np.arange(0, 25, 2))
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend(loc='upper left', fontsize=10)
fig_load_profiles_overview.tight_layout()
fig_load_profiles_overview.savefig(
    'fig_load_profiles_overview.png', dpi=300,
    bbox_inches='tight', facecolor='white'
)
plt.show()


In [ ]:
# Check whether February exists in the dataframe at all
feb_rows = clean_df[
    (clean_df["meter_dt"] >= "2024-02-01") &
    (clean_df["meter_dt"] < "2024-03-01")
]

print("February rows:", len(feb_rows))
print("February households:", feb_rows["household_ID"].nunique())

if len(feb_rows) > 0:
    print("February date range:", feb_rows["meter_dt"].min(), "to", feb_rows["meter_dt"].max())
else:
    print("No valid February 2024 rows exist in clean_df.")

In [ ]:
for name in ["merged_df", "clean_df", "golden_df"]:
    if name in globals():
        temp = globals()[name]
        if "meter_dt" in temp.columns:
            feb = temp[
                (temp["meter_dt"] >= "2024-02-01") &
                (temp["meter_dt"] < "2024-03-01")
            ]
            print(f"{name}: February rows = {len(feb):,}, households = {feb['household_ID'].nunique() if 'household_ID' in feb.columns else 'N/A'}")

In [ ]:
from scipy.optimize import linear_sum_assignment
import pandas as pd
import numpy as np

comparison = pd.crosstab(
    pd.Series(labels_10, name='n_init_10'),
    pd.Series(labels_20, name='n_init_20')
)
row_ind, col_ind = linear_sum_assignment(-comparison.to_numpy())
label_mapping = {
    comparison.index[row]: comparison.columns[col]
    for row, col in zip(row_ind, col_ind)
}
labels_10_aligned = np.array([label_mapping[label] for label in labels_10])
changed_mask = labels_10_aligned != labels_20

print('Optimally aligned cross-tabulation:')
display(comparison)
print('\nHouseholds with changed assignments:', int(changed_mask.sum()))
print('Percentage changed:', round(changed_mask.mean() * 100, 4), '%')

changed_households = clean_fm.index[changed_mask]
changed_details = clean_fm.loc[
    changed_households,
    CORE_FEATURES + ['cluster_name']
].copy()
changed_details['n_init_10_aligned_cluster'] = labels_10_aligned[changed_mask]
changed_details['n_init_20_cluster'] = labels_20[changed_mask]
changed_details['n_init_10_aligned_name'] = changed_details[
    'n_init_10_aligned_cluster'
].map(CLUSTER_NAME_MAP)
changed_details['n_init_20_name'] = changed_details[
    'n_init_20_cluster'
].map(CLUSTER_NAME_MAP)
display(changed_details)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("--- GENERATING FINAL NSRSIT TWO-COLUMN COLOR FIGURE ---")

if 'build_household_balanced_cluster_profiles' not in globals():
    raise RuntimeError('Run the 24-hour load-profile cell first.')
cluster_profiles = build_household_balanced_cluster_profiles()
time_axis = np.arange(96) / 4.0

fig_load_profiles_color, ax = plt.subplots(figsize=(6.6, 4.2))
for c_id in CLUSTER_ORDER:
    ax.plot(
        time_axis,
        cluster_profiles[c_id].to_numpy(),
        label=CLUSTER_LABEL_MAP[c_id],
        color=CLUSTER_COLOR_MAP[c_id],
        linewidth=1.8 if CLUSTER_NAME_MAP[c_id] == 'Peak-Exposed' else 1.5
    )
ax.axvspan(18.5, 22.5, color='#e74c3c', alpha=0.12, zorder=0,
           label='CEB evening peak window 18:30–22:30')
ax.set_xlim(0, 24)
ax.set_xticks(np.arange(0, 25, 2))
ax.set_xlabel('Hour of day', fontsize=10)
ax.set_ylabel('Household-balanced mean 15-min kWh', fontsize=10)
ax.tick_params(axis='both', labelsize=9)
ax.grid(True, linestyle='--', linewidth=0.35, alpha=0.6)
for spine in ax.spines.values():
    spine.set_linewidth(0.8)
ax.legend(loc='upper left', fontsize=8.2, frameon=True, framealpha=1.0,
          edgecolor='0.65', handlelength=2.2, borderpad=0.5, labelspacing=0.4)
fig_load_profiles_color.tight_layout()
fig_load_profiles_color.savefig(
    'fig_load_profiles_color.png', dpi=300,
    bbox_inches='tight', facecolor='white'
)
plt.show()


In [ ]:
print('Saved publication color profile: fig_load_profiles_color.png')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("--- GENERATING FINAL NSRSIT TWO-COLUMN GRAYSCALE FIGURE ---")

if 'build_household_balanced_cluster_profiles' not in globals():
    raise RuntimeError('Run the 24-hour load-profile cell first.')
cluster_profiles = build_household_balanced_cluster_profiles()
time_axis = np.arange(96) / 4.0

style_by_archetype = {
    'Peak-Exposed': {'color': 'black', 'linestyle': '-', 'linewidth': 1.7},
    'Typical Household': {'color': '0.35', 'linestyle': '--', 'linewidth': 1.4},
    'Heavy Continuous': {'color': '0.55', 'linestyle': '-.', 'linewidth': 1.4},
    'Daytime-Active': {'color': '0.72', 'linestyle': ':', 'linewidth': 2.0}
}

fig_load_profiles_bw, ax = plt.subplots(figsize=(6.6, 4.2))
for c_id in CLUSTER_ORDER:
    ax.plot(
        time_axis,
        cluster_profiles[c_id].to_numpy(),
        label=CLUSTER_LABEL_MAP[c_id],
        **style_by_archetype[CLUSTER_NAME_MAP[c_id]]
    )
ax.axvspan(18.5, 22.5, facecolor='0.90', edgecolor='0.75', linewidth=0.4,
           zorder=0, label='CEB evening peak window 18:30–22:30')
ax.set_xlim(0, 24)
ax.set_xticks(np.arange(0, 25, 2))
ax.set_xlabel('Hour of day', fontsize=10)
ax.set_ylabel('Household-balanced mean 15-min kWh', fontsize=10)
ax.tick_params(axis='both', labelsize=9)
ax.grid(True, linestyle='--', linewidth=0.35, color='0.82')
for spine in ax.spines.values():
    spine.set_linewidth(0.8)
ax.legend(loc='upper left', fontsize=8.2, frameon=True, framealpha=1.0,
          edgecolor='0.65', handlelength=2.2, borderpad=0.5, labelspacing=0.4)
fig_load_profiles_bw.tight_layout()
fig_load_profiles_bw.savefig(
    'fig_load_profiles_bw.png', dpi=300,
    bbox_inches='tight', facecolor='white'
)
plt.show()


In [ ]:
# CELL 8 [CODE] — FEATURE CORRELATION HEATMAP
# ─────────────────────────────────────────────────────────────────────────────
plt.figure(figsize=(8, 6))
corr_matrix = clean_fm[CORE_FEATURES].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm',
            vmin=-1, vmax=1, fmt=".2f")
plt.title("Feature Correlation Heatmap (Check for Collinearity)",
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# CELL 9 [CODE] — K-MEANS vs GMM EVALUATION SUITE (K=2–10)
# Uses the exact locked X_clean representation.
# Non-converged GMM fits are recorded but not scored or interpreted.
# ─────────────────────────────────────────────────────────────────────────────
from pathlib import Path

print("--- EXECUTING ADVANCED MODEL EVALUATION SUITE ---")

k_range = list(range(2, 11))
RANDOM_SEED = 42

kmeans_rows = []
gmm_rows = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=20)
    k_labels = km.fit_predict(X_clean)

    kmeans_rows.append({
        "k": k,
        "silhouette": silhouette_score(X_clean, k_labels),
        "davies_bouldin": davies_bouldin_score(X_clean, k_labels),
        "calinski_harabasz": calinski_harabasz_score(X_clean, k_labels),
        "inertia": float(km.inertia_),
        "smallest_cluster": int(np.bincount(k_labels, minlength=k).min()),
        "largest_cluster": int(np.bincount(k_labels, minlength=k).max())
    })

    gmm = GaussianMixture(
        n_components=k,
        covariance_type="full",
        random_state=RANDOM_SEED,
        n_init=5,
        reg_covar=1e-6,
        max_iter=1000
    )
    gmm.fit(X_clean)
    converged = bool(gmm.converged_)

    if converged:
        g_labels = gmm.predict(X_clean)
        gmm_bic = float(gmm.bic(X_clean))
        gmm_aic = float(gmm.aic(X_clean))
        gmm_silhouette = (
            silhouette_score(X_clean, g_labels)
            if len(np.unique(g_labels)) > 1
            else np.nan
        )
        smallest_component = int(np.bincount(g_labels, minlength=k).min())
        largest_component = int(np.bincount(g_labels, minlength=k).max())
    else:
        # Terminal, non-converged parameter estimates are not interpreted.
        gmm_bic = np.nan
        gmm_aic = np.nan
        gmm_silhouette = np.nan
        smallest_component = np.nan
        largest_component = np.nan

    gmm_rows.append({
        "k": k,
        "converged": converged,
        "n_iter": int(gmm.n_iter_),
        "bic": gmm_bic,
        "aic": gmm_aic,
        "hard_label_silhouette": gmm_silhouette,
        "smallest_component": smallest_component,
        "largest_component": largest_component
    })

    print(
        f"  K={k}: KMeans complete | "
        f"GMM converged={converged}, iterations={gmm.n_iter_}"
    )

kmeans_k_sweep = pd.DataFrame(kmeans_rows).set_index("k")
gmm_k_sweep = pd.DataFrame(gmm_rows).set_index("k")

print("\nKMeans K-sweep metrics:")
display(kmeans_k_sweep.round(6))

print("\nGMM K-sweep metrics and convergence:")
display(gmm_k_sweep.round(6))

nonconverged_gmm_k = gmm_k_sweep.index[~gmm_k_sweep["converged"]].tolist()
if nonconverged_gmm_k:
    print(
        "GMM did not converge for K values "
        f"{nonconverged_gmm_k}; their AIC, BIC, silhouette and component-size "
        "metrics are intentionally left missing."
    )
else:
    print("All GMM K=2–10 fits converged within the specified limit.")

fig, axs = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(
    "Cluster Optimization Diagnostics: KMeans and GMM",
    fontsize=16,
    fontweight="bold"
)

axs[0, 0].plot(
    kmeans_k_sweep.index,
    kmeans_k_sweep["silhouette"],
    marker="o",
    linewidth=2,
    label="KMeans"
)
converged_gmm_sweep = gmm_k_sweep[gmm_k_sweep["converged"]]
axs[0, 0].plot(
    converged_gmm_sweep.index,
    converged_gmm_sweep["hard_label_silhouette"],
    marker="s",
    linewidth=2,
    linestyle="--",
    label="GMM hard labels (converged fits)"
)
axs[0, 0].set_title("Silhouette Score")
axs[0, 0].set_xlabel("Number of Clusters (K)")
axs[0, 0].set_ylabel("Silhouette")
axs[0, 0].grid(True, linestyle="--", alpha=0.6)
axs[0, 0].legend()

axs[0, 1].plot(
    kmeans_k_sweep.index,
    kmeans_k_sweep["davies_bouldin"],
    marker="o",
    linewidth=2
)
axs[0, 1].set_title("KMeans Davies–Bouldin Index")
axs[0, 1].set_xlabel("Number of Clusters (K)")
axs[0, 1].set_ylabel("Lower is better")
axs[0, 1].grid(True, linestyle="--", alpha=0.6)

axs[1, 0].plot(
    kmeans_k_sweep.index,
    kmeans_k_sweep["calinski_harabasz"],
    marker="o",
    linewidth=2
)
axs[1, 0].set_title("KMeans Calinski–Harabasz Index")
axs[1, 0].set_xlabel("Number of Clusters (K)")
axs[1, 0].set_ylabel("Higher is better")
axs[1, 0].grid(True, linestyle="--", alpha=0.6)

axs[1, 1].plot(
    converged_gmm_sweep.index,
    converged_gmm_sweep["bic"],
    marker="o",
    linewidth=2,
    label="BIC"
)
axs[1, 1].plot(
    converged_gmm_sweep.index,
    converged_gmm_sweep["aic"],
    marker="x",
    linewidth=2,
    linestyle="--",
    label="AIC"
)
axs[1, 1].set_title("GMM Information Criteria (Converged Fits)")
axs[1, 1].set_xlabel("Number of Components (K)")
axs[1, 1].set_ylabel("Lower is better")
axs[1, 1].grid(True, linestyle="--", alpha=0.6)
axs[1, 1].legend()

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

Path("tables_outputs").mkdir(exist_ok=True)
kmeans_k_sweep.to_csv("tables_outputs/kmeans_k_sweep_metrics.csv")
gmm_k_sweep.to_csv("tables_outputs/gmm_k_sweep_metrics_with_convergence.csv")


In [ ]:
# CELL 10 [CODE] — K=3–6 DNA COMPARISON (KMeans vs GMM)
# Only converged GMM fits are profiled.
# ─────────────────────────────────────────────────────────────────────────────
from pathlib import Path

print("--- K-MEANS vs. GMM DNA COMPARISON (K=3 TO 6) ---")
gmm_dna_convergence_rows = []

for k in range(3, 7):
    print(f"\n{'=' * 60}")
    print(f"  K = {k}")
    print("=" * 60)

    # Isolated reporting dataframe prevents state leakage into clean_fm.
    loop_df = clean_fm[CORE_FEATURES].copy()

    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    loop_df["kmeans_cluster"] = km.fit_predict(X_clean)

    print("\n[ K-MEANS ]")
    k_sizes = (
        loop_df["kmeans_cluster"]
        .value_counts()
        .sort_index()
        .rename("HH_Count")
    )
    k_medians = (
        loop_df
        .groupby("kmeans_cluster")[CORE_FEATURES]
        .median()
        .round(3)
    )
    print(k_medians.join(k_sizes).to_string())

    gmm = GaussianMixture(
        n_components=k,
        covariance_type="full",
        random_state=42,
        n_init=10,
        max_iter=1000,
        reg_covar=1e-6
    )
    gmm.fit(X_clean)

    gmm_dna_convergence_rows.append({
        "k": k,
        "converged": bool(gmm.converged_),
        "n_iter": int(gmm.n_iter_)
    })

    print("\n[ GMM ]")
    if not gmm.converged_:
        print(
            "GMM did not converge; component sizes and medians are omitted "
            "rather than interpreted from terminal parameters."
        )
        continue

    loop_df["gmm_cluster"] = gmm.predict(X_clean)
    g_sizes = (
        loop_df["gmm_cluster"]
        .value_counts()
        .sort_index()
        .rename("HH_Count")
    )
    g_medians = (
        loop_df
        .groupby("gmm_cluster")[CORE_FEATURES]
        .median()
        .round(3)
    )
    print(g_medians.join(g_sizes).to_string())

gmm_dna_convergence = pd.DataFrame(gmm_dna_convergence_rows).set_index("k")
print("\nGMM DNA-comparison convergence record:")
display(gmm_dna_convergence)

Path("tables_outputs").mkdir(exist_ok=True)
gmm_dna_convergence.to_csv(
    "tables_outputs/gmm_k3_to_k6_dna_convergence.csv"
)


In [ ]:
# CELL 11 [CODE] — PRIMARY VALIDATION SUITE
# (Seed Stability, Bootstrap Robustness, Silhouette, CV)
# ─────────────────────────────────────────────────────────────────────────────
print("--- EXECUTING K=4 CLUSTER DIAGNOSTICS ---")
base_labels = optimal_kmeans.labels_

print("\n1. SEED STABILITY (20 Random Seeds)")
ari_seeds = []
for seed in range(20):
    km = KMeans(n_clusters=4, random_state=seed, n_init=20)
    labels = km.fit_predict(X_clean)
    ari_seeds.append(adjusted_rand_score(base_labels, labels))
print(f"   Mean ARI: {np.mean(ari_seeds):.3f} | Min ARI: {np.min(ari_seeds):.3f}")

print("\n2. BOOTSTRAP ROBUSTNESS (100 Iterations)")
ari_boot = []
for i in range(100):
    X_boot = resample(X_clean, random_state=i)
    # random_state=i varies both data AND initialisation independently
    km_boot = KMeans(n_clusters=4, random_state=i, n_init=20)
    km_boot.fit(X_boot)
    labels_boot_full = km_boot.predict(X_clean)
    ari_boot.append(adjusted_rand_score(base_labels, labels_boot_full))
print(f"   Mean ARI: {np.mean(ari_boot):.3f} | Min ARI: {np.min(ari_boot):.3f}")

print("\n2b. BOOTSTRAP — MAJORITY CLUSTER VALIDATION")

majority_cluster_ids = [
    cluster_id
    for cluster_id, cluster_name in CLUSTER_NAME_MAP.items()
    if cluster_name in {
        "Typical Household",
        "Heavy Continuous"
    }
]

assert len(majority_cluster_ids) == 2, (
    "Could not identify the two majority archetypes dynamically."
)

majority_mask = np.isin(
    base_labels,
    majority_cluster_ids
)

ari_boot_majority = []

for i in range(100):
    X_boot = resample(
        X_clean,
        random_state=i
    )

    km_boot = KMeans(
        n_clusters=4,
        random_state=i,
        n_init=20
    )

    km_boot.fit(X_boot)

    labels_full = km_boot.predict(
        X_clean
    )

    ari_boot_majority.append(
        adjusted_rand_score(
            base_labels[majority_mask],
            labels_full[majority_mask]
        )
    )

assert len(ari_boot_majority) == 100, (
    "Majority-cluster bootstrap did not produce 100 ARI values."
)

print(
    f"   Majority-only Mean ARI: "
    f"{np.mean(ari_boot_majority):.3f} | "
    f"Min ARI: {np.min(ari_boot_majority):.3f}"
)

print(
    "   This analysis evaluates the stability of the "
    "Typical Household and Heavy Continuous populations."
)
print("\n3. SILHOUETTE ANALYSIS")
sil_score = silhouette_score(X_clean, base_labels)
sil_samples_arr = silhouette_samples(X_clean, base_labels)
print(f"   Overall K=4 Silhouette: {sil_score:.3f}")
for c in range(4):
    mask = base_labels == c
    print(f"   {c_names[c]:<30}: {sil_samples_arr[mask].mean():.3f} (N={mask.sum()})")

print("\n4. COEFFICIENT OF VARIATION (Internal Spread)")
cluster_std = clean_fm.groupby('final_cluster')[CORE_FEATURES].std()
cluster_mean = clean_fm.groupby('final_cluster')[CORE_FEATURES].mean()
cv = (cluster_std / cluster_mean).abs().round(3)
print(cv.to_string())


In [ ]:
# CELL 12 [CODE] — SENSITIVITY ANALYSIS (RobustScaler)
# Purpose: Confirm StandardScaler is superior. Results are reported as
# a sensitivity analysis, not as an alternative model.
# ─────────────────────────────────────────────────────────────────────────────
print("--- SENSITIVITY ANALYSIS: ROBUST SCALING ---")

robust_scaler = RobustScaler()
X_robust = robust_scaler.fit_transform(clean_fm[CORE_FEATURES])

km_robust = KMeans(n_clusters=4, random_state=42, n_init=20)
base_labels_robust = km_robust.fit_predict(X_robust)
clean_fm['robust_cluster'] = base_labels_robust

print("\n[ ALIGNMENT CHECK ]")
print("Cross-tabulation (Rows: StandardScaler, Cols: RobustScaler):")
print(pd.crosstab(clean_fm['final_cluster'], clean_fm['robust_cluster']))
print("\nMedian Profiles (RobustScaler model):")
print(clean_fm.groupby('robust_cluster')[CORE_FEATURES].median().round(3).to_string())

print("\n0. SEED STABILITY (RobustScaler)")
ari_seeds_r = []
for seed in range(20):
    km_s = KMeans(n_clusters=4, random_state=seed, n_init=20)
    ari_seeds_r.append(adjusted_rand_score(base_labels_robust,
                                            km_s.fit_predict(X_robust)))
print(f"   Mean ARI: {np.mean(ari_seeds_r):.3f} | Min ARI: {np.min(ari_seeds_r):.3f}")

print("\n1. BOOTSTRAP ROBUSTNESS (RobustScaler)")
ari_boot_r = []
for i in range(100):
    X_boot_r = resample(X_robust, random_state=i)
    km_br = KMeans(n_clusters=4, random_state=i, n_init=20)
    km_br.fit(X_boot_r)
    ari_boot_r.append(adjusted_rand_score(base_labels_robust,
                                           km_br.predict(X_robust)))
print(f"   Mean ARI: {np.mean(ari_boot_r):.3f} | Min ARI: {np.min(ari_boot_r):.3f}")

print("\n2. PER-CLUSTER SILHOUETTE (RobustScaler)")
sil_r = silhouette_score(X_robust, base_labels_robust)
sil_s_r = silhouette_samples(X_robust, base_labels_robust)
print(f"   Overall Silhouette: {sil_r:.3f}")
for c in range(4):
    mask = base_labels_robust == c
    print(f"   Cluster {c}: {sil_s_r[mask].mean():.3f} (N={mask.sum()})")

print("\n3. COEFFICIENT OF VARIATION (RobustScaler)")
cv_r = (
    clean_fm.groupby('robust_cluster')[CORE_FEATURES].std() /
    clean_fm.groupby('robust_cluster')[CORE_FEATURES].mean()
).abs().round(3)
print(cv_r.to_string())

# Housekeeping: remove temporary column, clean_fm stays pure
clean_fm.drop(columns=['robust_cluster'], inplace=True, errors='ignore')

In [ ]:
# CELL 14 [CODE] — STATISTICAL TESTS, ABLATION, PCA, SILHOUETTE, ELBOW
# ─────────────────────────────────────────────────────────────────────────────
print("--- 1. STATISTICAL SIGNIFICANCE (Kruskal-Wallis) ---")
for feature in CORE_FEATURES:
    groups = [clean_fm[clean_fm['final_cluster'] == k][feature].values
              for k in range(4)]
    stat, p = kruskal(*groups)
    print(f"  {feature:<25} H-stat: {stat:>8.2f} | p-value: {p:.4e}")

print("\n--- 2. FEATURE ABLATION STUDY ---")
base_labels_ablation = clean_fm['final_cluster'].values
# NOTE: Each ablation run uses a fresh StandardScaler — mathematically
# required because removing a feature changes the shape of the remaining
# feature space. This is NOT a violation of the locked model principle.
# It is an independent experiment measuring feature information contribution. 
ablation_results = {}
for drop_col in CORE_FEATURES:
    remaining = [f for f in CORE_FEATURES if f != drop_col]
    X_drop = StandardScaler().fit_transform(clean_fm[remaining])
    labels_drop = KMeans(n_clusters=4, random_state=42, n_init=20).fit_predict(X_drop)
    ari = adjusted_rand_score(base_labels_ablation, labels_drop)
    ablation_results[drop_col] = ari
    print(f"  Without {drop_col:<25} ARI: {ari:.3f}")

colors = dict(CLUSTER_COLOR_MAP)

# --- PCA Scatter Plot ---
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_clean)

plt.figure(figsize=(10, 7))
for c in CLUSTER_ORDER:
    mask = clean_fm['final_cluster'] == c
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=colors[c], label=c_names[c], alpha=0.5, s=20)
plt.title("PCA Projection of K=4 Clusters", fontsize=14, fontweight='bold')
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% Variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% Variance)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

# --- Silhouette Plot ---
sil_plot_score = silhouette_score(X_clean, base_labels_ablation)
sil_plot_samples = silhouette_samples(X_clean, base_labels_ablation)

fig, ax = plt.subplots(figsize=(10, 6))
y_lower = 10
for c in range(4):
    cluster_sil = sil_plot_samples[base_labels_ablation == c].copy()
    cluster_sil.sort()
    y_upper = y_lower + len(cluster_sil)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil,
                     facecolor=colors[c], alpha=0.7, label=c_names[c])
    y_lower = y_upper + 10
ax.axvline(x=sil_plot_score, color='black', linestyle='--',
           label=f'Mean Silhouette ({sil_plot_score:.3f})')
ax.set_title("Silhouette Plot per Cluster — K=4 K-Means",
             fontsize=14, fontweight='bold')
ax.set_xlabel("Silhouette Coefficient")
ax.set_ylabel("Cluster Instances")
ax.legend()
plt.tight_layout()
plt.show()

# --- Elbow Plot ---
inertias = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    km.fit(X_clean)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(2, 11), inertias, marker='o', color='#2c3e50', linewidth=2)
plt.title("Elbow Method — K-Means Inertia", fontsize=14, fontweight='bold')
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia (Within-Cluster Sum of Squares)")
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(range(2, 11))
plt.tight_layout()
plt.show()



## Supplementary algorithm sensitivity checks

These cells compare alternative clustering formulations against the locked final KMeans K=4 solution. They are supplementary diagnostics and do not replace the final model.


In [ ]:
# ============================================================
# SUPPLEMENTARY GMM POSTERIOR-CONFIDENCE DIAGNOSTIC
# The posterior probabilities describe confidence within the GMM model.
# They are NOT probabilities of membership in the locked KMeans clusters.
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from scipy.optimize import linear_sum_assignment

try:
    from IPython.display import display
except Exception:
    display = print

FEATURES = list(CORE_FEATURES)
EXPECTED_K = 4
cluster_names = dict(CLUSTER_NAME_MAP)

if len(clean_fm) != len(X_clean):
    raise RuntimeError("clean_fm and X_clean row counts differ.")

kmeans_labels = clean_fm["final_cluster"].astype(int).to_numpy()
household_ids = clean_fm.index.astype(str).to_numpy()

if len(np.unique(kmeans_labels)) != EXPECTED_K:
    raise RuntimeError("The locked KMeans solution does not contain four clusters.")

# Use the exact locked standardized representation rather than refitting a scaler.
gmm_k4 = GaussianMixture(
    n_components=EXPECTED_K,
    covariance_type="full",
    n_init=20,
    max_iter=1000,
    random_state=42,
    reg_covar=1e-6
)

gmm_raw_labels = gmm_k4.fit_predict(X_clean)
gmm_prob_raw = gmm_k4.predict_proba(X_clean)

if not gmm_k4.converged_:
    raise RuntimeError(
        "The K=4 GMM did not converge. Increase max_iter/reg_covar before interpretation."
    )

# Align arbitrary GMM component identifiers to the locked KMeans identifiers
# only to make the cross-tabulation readable. Alignment does not turn GMM
# probabilities into KMeans probabilities.
ct = pd.crosstab(
    pd.Series(kmeans_labels, name="kmeans_cluster"),
    pd.Series(gmm_raw_labels, name="gmm_component")
).reindex(index=range(EXPECTED_K), columns=range(EXPECTED_K), fill_value=0)

row_ind, col_ind = linear_sum_assignment(-ct.to_numpy())
gmm_to_kmeans = {
    int(ct.columns[col]): int(ct.index[row])
    for row, col in zip(row_ind, col_ind)
}

gmm_aligned_labels = np.array([gmm_to_kmeans[int(x)] for x in gmm_raw_labels])

prob_aligned = np.zeros_like(gmm_prob_raw)
for gmm_component, aligned_cluster in gmm_to_kmeans.items():
    prob_aligned[:, aligned_cluster] = gmm_prob_raw[:, gmm_component]

if not np.allclose(prob_aligned.sum(axis=1), 1.0, atol=1e-6):
    raise RuntimeError("Aligned GMM posterior probabilities do not sum to one.")

gmm_max_posterior = prob_aligned.max(axis=1)
eps = 1e-12
gmm_normalized_entropy = -np.sum(
    prob_aligned * np.log(prob_aligned + eps), axis=1
) / np.log(EXPECTED_K)

# Descriptive thresholds only. They are not used to alter the KMeans model.
gmm_low_confidence_lt_070 = gmm_max_posterior < 0.70
gmm_very_low_confidence_lt_060 = gmm_max_posterior < 0.60

gmm_posterior_households = pd.DataFrame({
    "household_ID": household_ids,
    "final_kmeans_cluster": kmeans_labels,
    "final_kmeans_cluster_name": pd.Series(kmeans_labels).map(cluster_names).to_numpy(),
    "aligned_gmm_cluster": gmm_aligned_labels,
    "aligned_gmm_cluster_name": pd.Series(gmm_aligned_labels).map(cluster_names).to_numpy(),
    "gmm_max_posterior_probability": gmm_max_posterior,
    "gmm_normalized_posterior_entropy": gmm_normalized_entropy,
    "gmm_low_confidence_lt_0_70": gmm_low_confidence_lt_070,
    "gmm_very_low_confidence_lt_0_60": gmm_very_low_confidence_lt_060
})

for cluster_id in range(EXPECTED_K):
    safe_name = cluster_names[cluster_id].replace(" ", "_").replace("-", "_")
    gmm_posterior_households[f"gmm_posterior_{safe_name}"] = prob_aligned[:, cluster_id]

agreement_rate = float(np.mean(kmeans_labels == gmm_aligned_labels))
gmm_kmeans_ari = float(adjusted_rand_score(kmeans_labels, gmm_aligned_labels))
gmm_kmeans_nmi = float(normalized_mutual_info_score(kmeans_labels, gmm_aligned_labels))

gmm_k4_summary = pd.DataFrame([{
    "n_households": len(clean_fm),
    "gmm_converged": bool(gmm_k4.converged_),
    "gmm_iterations": int(gmm_k4.n_iter_),
    "gmm_bic": float(gmm_k4.bic(X_clean)),
    "gmm_aic": float(gmm_k4.aic(X_clean)),
    "aligned_hard_label_agreement_with_kmeans": agreement_rate,
    "adjusted_rand_index_vs_kmeans": gmm_kmeans_ari,
    "normalized_mutual_information_vs_kmeans": gmm_kmeans_nmi,
    "mean_gmm_max_posterior": float(gmm_max_posterior.mean()),
    "median_gmm_max_posterior": float(np.median(gmm_max_posterior)),
    "gmm_low_confidence_share_lt_0_70": float(gmm_low_confidence_lt_070.mean()),
    "gmm_very_low_confidence_share_lt_0_60": float(gmm_very_low_confidence_lt_060.mean()),
    "mean_gmm_normalized_entropy": float(gmm_normalized_entropy.mean())
}])

gmm_confidence_by_kmeans_cluster = (
    gmm_posterior_households
    .groupby(["final_kmeans_cluster", "final_kmeans_cluster_name"], dropna=False)
    .agg(
        households=("household_ID", "size"),
        mean_gmm_max_posterior=("gmm_max_posterior_probability", "mean"),
        median_gmm_max_posterior=("gmm_max_posterior_probability", "median"),
        low_confidence_households=("gmm_low_confidence_lt_0_70", "sum"),
        low_confidence_share=("gmm_low_confidence_lt_0_70", "mean"),
        very_low_confidence_households=("gmm_very_low_confidence_lt_0_60", "sum"),
        very_low_confidence_share=("gmm_very_low_confidence_lt_0_60", "mean"),
        mean_gmm_normalized_entropy=("gmm_normalized_posterior_entropy", "mean")
    )
    .reset_index()
)

gmm_alignment_table = pd.crosstab(
    pd.Series(kmeans_labels, name="Final KMeans cluster"),
    pd.Series(gmm_aligned_labels, name="Aligned GMM component")
)

gmm_alignment_percent = gmm_alignment_table.div(
    gmm_alignment_table.sum(axis=1), axis=0
) * 100

out_dir = Path("tables_outputs")
out_dir.mkdir(exist_ok=True)

gmm_posterior_households.to_csv(
    out_dir / "supplementary_gmm_posterior_confidence_households.csv",
    index=False
)
gmm_k4_summary.to_csv(
    out_dir / "supplementary_gmm_k4_summary.csv",
    index=False
)
gmm_confidence_by_kmeans_cluster.to_csv(
    out_dir / "supplementary_gmm_confidence_by_kmeans_cluster.csv",
    index=False
)
gmm_alignment_table.to_csv(out_dir / "supplementary_gmm_kmeans_alignment_counts.csv")
gmm_alignment_percent.to_csv(out_dir / "supplementary_gmm_kmeans_alignment_percent.csv")

print("=" * 80)
print("SUPPLEMENTARY GMM K=4 DIAGNOSTIC COMPLETED")
print("=" * 80)
print(
    "Interpretation rule: GMM posterior probabilities quantify confidence "
    "within the alternative GMM partition, not KMeans membership probability."
)
print("\nGMM K=4 summary:")
display(gmm_k4_summary.round(6))
print("\nGMM posterior confidence summarized by locked KMeans cluster:")
display(gmm_confidence_by_kmeans_cluster.round(6))
print("\nAligned hard-label cross-tabulation:")
display(gmm_alignment_table)


In [ ]:
# ============================================================
# REPORT-READY FIGURES FOR GMM POSTERIOR-CONFIDENCE DIAGNOSTIC
# ============================================================

import matplotlib.pyplot as plt
from pathlib import Path

fig_dir = Path("final_figures")
fig_dir.mkdir(exist_ok=True)

if "gmm_posterior_households" not in globals():
    raise RuntimeError("Run the GMM posterior-confidence diagnostic first.")

plt.figure(figsize=(6.6, 4.2))
plt.hist(
    gmm_posterior_households["gmm_max_posterior_probability"],
    bins=30,
    edgecolor="black",
    alpha=0.75
)
plt.axvline(0.70, linestyle="--", linewidth=1.2, label="Descriptive threshold: 0.70")
plt.axvline(0.60, linestyle=":", linewidth=1.2, label="Descriptive threshold: 0.60")
plt.xlabel("Maximum posterior probability within the GMM model")
plt.ylabel("Households")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(fig_dir / "Fig_GMM_Max_Posterior_Probability.png", dpi=600, bbox_inches="tight")
plt.savefig(fig_dir / "Fig_GMM_Max_Posterior_Probability.pdf", bbox_inches="tight")
plt.show()

plot_df = gmm_confidence_by_kmeans_cluster.copy()
plot_df["cluster_label"] = (
    plot_df["final_kmeans_cluster"].astype(str)
    + " - "
    + plot_df["final_kmeans_cluster_name"].fillna("")
)

plt.figure(figsize=(6.6, 4.2))
plt.barh(
    plot_df["cluster_label"],
    plot_df["low_confidence_share"] * 100
)
plt.xlabel("Households with GMM maximum posterior < 0.70 (%)")
plt.ylabel("Locked KMeans cluster used only for grouping")
plt.tight_layout()
plt.savefig(fig_dir / "Fig_GMM_Low_Confidence_Share_By_KMeans_Cluster.png", dpi=600, bbox_inches="tight")
plt.savefig(fig_dir / "Fig_GMM_Low_Confidence_Share_By_KMeans_Cluster.pdf", bbox_inches="tight")
plt.show()

row = gmm_k4_summary.iloc[0]
print("Report-ready interpretation:")
print(
    f"The converged K=4 GMM produced ARI={row['adjusted_rand_index_vs_kmeans']:.3f} "
    f"against the locked KMeans partition and a median within-GMM maximum "
    f"posterior probability of {row['median_gmm_max_posterior']:.3f}. "
    "The posterior values describe confidence under the GMM model and are not "
    "interpreted as KMeans membership probabilities."
)


In [ ]:
# ============================================================
# SUPPLEMENTARY FUZZY C-MEANS K=4 SENSITIVITY CHECK
# Multi-start implementation with explicit convergence control.
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import adjusted_rand_score, silhouette_score


def _fcm_single_run(X, n_clusters, m=2.0, max_iter=2000, tol=1e-6, seed=0):
    if m <= 1:
        raise ValueError("Fuzzifier m must be greater than 1.")

    rng = np.random.default_rng(seed)
    membership = rng.random((len(X), n_clusters))
    membership /= membership.sum(axis=1, keepdims=True)

    objective_history = []
    converged = False

    for iteration in range(1, max_iter + 1):
        weights = membership ** m
        centers = (weights.T @ X) / np.maximum(weights.sum(axis=0)[:, None], 1e-12)

        distances = np.maximum(cdist(X, centers, metric="euclidean"), 1e-12)
        inverse_power = distances ** (-2.0 / (m - 1.0))
        updated_membership = inverse_power / inverse_power.sum(axis=1, keepdims=True)

        objective = float(np.sum((updated_membership ** m) * (distances ** 2)))
        objective_history.append(objective)

        max_change = float(np.max(np.abs(updated_membership - membership)))
        membership = updated_membership

        if max_change < tol:
            converged = True
            break

    # Recompute centres, distances and objective from the final membership
    # matrix so all returned quantities describe the same terminal solution.
    final_weights = membership ** m
    centers = (
        final_weights.T @ X
    ) / np.maximum(final_weights.sum(axis=0)[:, None], 1e-12)
    final_distances = np.maximum(
        cdist(X, centers, metric="euclidean"),
        1e-12
    )
    final_objective = float(
        np.sum((membership ** m) * (final_distances ** 2))
    )

    hard_labels = membership.argmax(axis=1)
    return {
        "centers": centers,
        "membership": membership,
        "labels": hard_labels,
        "objective": final_objective,
        "objective_history": objective_history,
        "converged": converged,
        "n_iter": iteration,
        "seed": seed
    }


def fuzzy_c_means_multi_start(
    X,
    n_clusters,
    m=2.0,
    n_init=20,
    max_iter=2000,
    tol=1e-6,
    random_state=42
):
    seed_rng = np.random.default_rng(random_state)
    seeds = seed_rng.integers(0, 2**31 - 1, size=n_init)
    runs = [
        _fcm_single_run(
            X,
            n_clusters=n_clusters,
            m=m,
            max_iter=max_iter,
            tol=tol,
            seed=int(seed)
        )
        for seed in seeds
    ]

    converged_runs = [run for run in runs if run["converged"]]
    eligible_runs = converged_runs if converged_runs else runs
    best = min(eligible_runs, key=lambda run: run["objective"])
    best = dict(best)
    best["converged_runs"] = len(converged_runs)
    best["total_runs"] = len(runs)
    return best


fcm_k4_result = fuzzy_c_means_multi_start(
    X_clean,
    n_clusters=4,
    m=2.0,
    n_init=20,
    max_iter=2000,
    tol=1e-6,
    random_state=42
)

if not fcm_k4_result["converged"]:
    raise RuntimeError(
        "No Fuzzy C-Means K=4 start converged. Increase max_iter or review scaling."
    )

fcm_raw_labels = fcm_k4_result["labels"]
kmeans_labels = clean_fm["final_cluster"].astype(int).to_numpy()

# Align FCM hard-label identifiers for readable reporting.
fcm_ct = pd.crosstab(
    pd.Series(kmeans_labels, name="kmeans_cluster"),
    pd.Series(fcm_raw_labels, name="fcm_cluster")
).reindex(index=range(4), columns=range(4), fill_value=0)

row_ind, col_ind = linear_sum_assignment(-fcm_ct.to_numpy())
fcm_to_kmeans = {
    int(fcm_ct.columns[col]): int(fcm_ct.index[row])
    for row, col in zip(row_ind, col_ind)
}
fcm_aligned_labels = np.array([fcm_to_kmeans[int(x)] for x in fcm_raw_labels])

fcm_membership_aligned = np.zeros_like(fcm_k4_result["membership"])
for fcm_cluster, aligned_cluster in fcm_to_kmeans.items():
    fcm_membership_aligned[:, aligned_cluster] = fcm_k4_result["membership"][:, fcm_cluster]

fcm_max_membership = fcm_membership_aligned.max(axis=1)
fcm_normalized_entropy = -np.sum(
    fcm_membership_aligned * np.log(fcm_membership_aligned + 1e-12), axis=1
) / np.log(4)

# Check whether maximum-membership hardening actually produces
# four occupied clusters.
fcm_hard_cluster_sizes = np.bincount(
    fcm_aligned_labels,
    minlength=4
)

fcm_occupied_hard_clusters = int(
    np.count_nonzero(fcm_hard_cluster_sizes)
)

fcm_hard_partition_complete = (
    fcm_occupied_hard_clusters == 4
)

# Hard-label metrics are interpreted only when all requested
# fuzzy clusters contain at least one hard-assigned household.
if fcm_hard_partition_complete:
    fcm_hard_label_silhouette = silhouette_score(
        X_clean,
        fcm_aligned_labels
    )
    fcm_ari_vs_kmeans = adjusted_rand_score(
        kmeans_labels,
        fcm_aligned_labels
    )
else:
    fcm_hard_label_silhouette = np.nan
    fcm_ari_vs_kmeans = np.nan

fcm_k4_summary = pd.DataFrame([{
    "n_households": len(clean_fm),
    "fuzzifier_m": 2.0,
    "n_init": 20,
    "converged_runs": fcm_k4_result["converged_runs"],
    "total_runs": fcm_k4_result["total_runs"],
    "selected_seed": fcm_k4_result["seed"],
    "selected_run_converged": fcm_k4_result["converged"],
    "selected_run_iterations": fcm_k4_result["n_iter"],
    "objective": fcm_k4_result["objective"],
    "occupied_hard_clusters": fcm_occupied_hard_clusters,
    "hard_partition_complete": fcm_hard_partition_complete,
    "smallest_hard_cluster": int(fcm_hard_cluster_sizes.min()),
    "largest_hard_cluster": int(fcm_hard_cluster_sizes.max()),
    "hard_label_silhouette": fcm_hard_label_silhouette,
    "adjusted_rand_index_vs_kmeans": fcm_ari_vs_kmeans,
    "mean_max_fcm_membership": float(
        fcm_max_membership.mean()
    ),
    "median_max_fcm_membership": float(
        np.median(fcm_max_membership)
    ),
    "mean_normalized_fcm_entropy": float(
        fcm_normalized_entropy.mean()
    )
}])

fcm_households = pd.DataFrame({
    "household_ID": clean_fm.index.astype(str),
    "final_kmeans_cluster": kmeans_labels,
    "aligned_fcm_cluster": fcm_aligned_labels,
    "fcm_max_membership": fcm_max_membership,
    "fcm_normalized_membership_entropy": fcm_normalized_entropy
})

for cluster_id in range(4):
    fcm_households[f"fcm_membership_cluster_{cluster_id}"] = fcm_membership_aligned[:, cluster_id]

Path("tables_outputs").mkdir(exist_ok=True)
fcm_k4_summary.to_csv("tables_outputs/supplementary_fcm_k4_summary.csv", index=False)
fcm_households.to_csv("tables_outputs/supplementary_fcm_k4_households.csv", index=False)

print("=" * 80)
print("CONVERGED MULTI-START FUZZY C-MEANS K=4 RESULT")
print("=" * 80)
display(fcm_k4_summary.round(6))
if not fcm_hard_partition_complete:
    print(
        "FCM converged numerically, but maximum-membership "
        "hardening produced fewer than four occupied clusters. "
        "Therefore, K=4 hard-label silhouette and ARI are not "
        "interpreted. The fuzzy membership results are retained "
        "as a supplementary diagnostic."
    )
print(
    "FCM memberships are interpreted within the FCM model only; the locked "
    "KMeans solution remains the primary archetype model."
)


In [ ]:
# ============================================================
# SUPPLEMENTARY K-MEDOIDS K=4 ROBUSTNESS CHECK
# Dependency-free multi-start alternate optimization on the full
# pairwise Euclidean distance matrix. This avoids the incompatible
# scikit-learn-extra binary dependency.
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import pairwise_distances, adjusted_rand_score, silhouette_score
from scipy.optimize import linear_sum_assignment


def _initialize_medoids_kpp(distance_matrix, n_clusters, rng):
    n_samples = distance_matrix.shape[0]
    medoids = [int(rng.integers(0, n_samples))]

    while len(medoids) < n_clusters:
        nearest = distance_matrix[:, medoids].min(axis=1)
        nearest[np.array(medoids, dtype=int)] = 0.0
        probabilities = nearest ** 2
        total = probabilities.sum()

        if total <= 0:
            candidates = np.setdiff1d(np.arange(n_samples), np.array(medoids))
            medoids.append(int(rng.choice(candidates)))
        else:
            probabilities /= total
            candidate = int(rng.choice(n_samples, p=probabilities))
            if candidate not in medoids:
                medoids.append(candidate)

    return np.array(medoids, dtype=int)


def _kmedoids_alternate_single(distance_matrix, n_clusters, seed, max_iter=200):
    rng = np.random.default_rng(seed)
    medoids = np.sort(
        _initialize_medoids_kpp(distance_matrix, n_clusters, rng)
    )
    converged = False
    objective_history = []

    for iteration in range(1, max_iter + 1):
        labels = np.argmin(distance_matrix[:, medoids], axis=1)
        new_medoids = np.full(n_clusters, -1, dtype=int)
        empty_clusters = []

        for cluster_id in range(n_clusters):
            members = np.flatnonzero(labels == cluster_id)
            if len(members) == 0:
                empty_clusters.append(cluster_id)
                continue

            intra = distance_matrix[np.ix_(members, members)]
            new_medoids[cluster_id] = int(
                members[np.argmin(intra.sum(axis=1))]
            )

        # Repair empty clusters with distinct points furthest from the
        # currently selected non-empty medoids.
        used = set(new_medoids[new_medoids >= 0].tolist())
        for cluster_id in empty_clusters:
            available = np.array(
                [idx for idx in range(distance_matrix.shape[0]) if idx not in used],
                dtype=int
            )
            if len(available) == 0:
                raise RuntimeError("Unable to select a distinct medoid.")

            selected = np.array(sorted(used), dtype=int)
            if len(selected):
                nearest = distance_matrix[np.ix_(available, selected)].min(axis=1)
                replacement = int(available[np.argmax(nearest)])
            else:
                replacement = int(rng.choice(available))

            new_medoids[cluster_id] = replacement
            used.add(replacement)

        if len(np.unique(new_medoids)) != n_clusters:
            raise RuntimeError("K-Medoids update produced duplicate medoids.")

        # Medoid identifiers are arbitrary; canonical ordering prevents a
        # pure label permutation from appearing as non-convergence.
        new_medoids = np.sort(new_medoids)
        new_labels = np.argmin(distance_matrix[:, new_medoids], axis=1)
        objective = float(
            distance_matrix[
                np.arange(len(new_labels)),
                new_medoids[new_labels]
            ].sum()
        )
        objective_history.append(objective)

        if np.array_equal(new_medoids, medoids):
            medoids = new_medoids
            converged = True
            break

        medoids = new_medoids

    labels = np.argmin(distance_matrix[:, medoids], axis=1)
    objective = float(
        distance_matrix[np.arange(len(labels)), medoids[labels]].sum()
    )

    return {
        "medoid_indices": medoids,
        "labels": labels,
        "objective": objective,
        "objective_history": objective_history,
        "converged": converged,
        "n_iter": iteration,
        "seed": seed
    }


def kmedoids_alternate_multi_start(
    distance_matrix,
    n_clusters,
    n_init=10,
    max_iter=200,
    random_state=42
):
    seed_rng = np.random.default_rng(random_state)
    seeds = seed_rng.integers(0, 2**31 - 1, size=n_init)
    runs = [
        _kmedoids_alternate_single(
            distance_matrix,
            n_clusters=n_clusters,
            seed=int(seed),
            max_iter=max_iter
        )
        for seed in seeds
    ]

    converged_runs = [run for run in runs if run["converged"]]
    eligible_runs = converged_runs if converged_runs else runs
    best = min(eligible_runs, key=lambda run: run["objective"])
    best = dict(best)
    best["converged_runs"] = len(converged_runs)
    best["total_runs"] = len(runs)
    return best


print("Computing the full pairwise distance matrix once for K-Medoids...")
KMEDOIDS_DISTANCE_MATRIX = pairwise_distances(X_clean, metric="euclidean")

kmedoids_k4_result = kmedoids_alternate_multi_start(
    KMEDOIDS_DISTANCE_MATRIX,
    n_clusters=4,
    n_init=10,
    max_iter=200,
    random_state=42
)

if not kmedoids_k4_result["converged"]:
    raise RuntimeError(
        "No K-Medoids K=4 start converged under alternate optimization."
    )

kmedoids_raw_labels = kmedoids_k4_result["labels"]
kmeans_labels = clean_fm["final_cluster"].astype(int).to_numpy()

km_ct = pd.crosstab(
    pd.Series(kmeans_labels, name="kmeans_cluster"),
    pd.Series(kmedoids_raw_labels, name="kmedoids_cluster")
).reindex(index=range(4), columns=range(4), fill_value=0)

row_ind, col_ind = linear_sum_assignment(-km_ct.to_numpy())
kmedoids_to_kmeans = {
    int(km_ct.columns[col]): int(km_ct.index[row])
    for row, col in zip(row_ind, col_ind)
}
kmedoids_aligned_labels = np.array([
    kmedoids_to_kmeans[int(x)] for x in kmedoids_raw_labels
])

kmedoids_k4_summary = pd.DataFrame([{
    "n_households": len(clean_fm),
    "n_init": 10,
    "converged_runs": kmedoids_k4_result["converged_runs"],
    "total_runs": kmedoids_k4_result["total_runs"],
    "selected_seed": kmedoids_k4_result["seed"],
    "selected_run_iterations": kmedoids_k4_result["n_iter"],
    "objective_sum_distance_to_medoid": kmedoids_k4_result["objective"],
    "silhouette": silhouette_score(X_clean, kmedoids_aligned_labels),
    "adjusted_rand_index_vs_kmeans": adjusted_rand_score(
        kmeans_labels, kmedoids_aligned_labels
    ),
    "smallest_cluster": int(np.bincount(kmedoids_aligned_labels, minlength=4).min()),
    "largest_cluster": int(np.bincount(kmedoids_aligned_labels, minlength=4).max())
}])

kmedoids_k4_households = pd.DataFrame({
    "household_ID": clean_fm.index.astype(str),
    "final_kmeans_cluster": kmeans_labels,
    "aligned_kmedoids_cluster": kmedoids_aligned_labels
})

Path("tables_outputs").mkdir(exist_ok=True)
kmedoids_k4_summary.to_csv("tables_outputs/supplementary_kmedoids_k4_summary.csv", index=False)
kmedoids_k4_households.to_csv("tables_outputs/supplementary_kmedoids_k4_households.csv", index=False)

print("=" * 80)
print("CONVERGED MULTI-START K-MEDOIDS K=4 RESULT")
print("=" * 80)
display(kmedoids_k4_summary.round(6))
print(
    "This is a supplementary medoid-based robustness check. The locked "
    "KMeans K=4 solution remains the primary model."
)


In [ ]:
# ============================================================
# SUPPLEMENTARY K-MEDOIDS K-SWEEP (K=2–10)
# Non-converged solutions are recorded but not scored or interpreted.
# ============================================================

if "KMEDOIDS_DISTANCE_MATRIX" not in globals():
    raise RuntimeError("Run the K-Medoids K=4 cell first.")

kmedoids_sweep_rows = []

for k in range(2, 11):
    result = kmedoids_alternate_multi_start(
        KMEDOIDS_DISTANCE_MATRIX,
        n_clusters=k,
        n_init=5,
        max_iter=200,
        random_state=42 + k
    )

    converged = bool(result["converged"])
    labels_k = result["labels"]

    silhouette_k = np.nan
    smallest_cluster = np.nan
    largest_cluster = np.nan
    ari_k4 = np.nan

    if converged:
        if len(np.unique(labels_k)) > 1:
            silhouette_k = silhouette_score(X_clean, labels_k)
        cluster_sizes = np.bincount(labels_k, minlength=k)
        smallest_cluster = int(cluster_sizes.min())
        largest_cluster = int(cluster_sizes.max())

        if k == 4:
            ari_k4 = adjusted_rand_score(
                clean_fm["final_cluster"].astype(int).to_numpy(),
                labels_k
            )

    kmedoids_sweep_rows.append({
        "k": k,
        "converged": converged,
        "converged_runs": int(result["converged_runs"]),
        "total_runs": int(result["total_runs"]),
        "selected_iterations": int(result["n_iter"]),
        "terminal_objective": float(result["objective"]),
        "silhouette": silhouette_k,
        "smallest_cluster": smallest_cluster,
        "largest_cluster": largest_cluster,
        "ari_vs_locked_kmeans_k4": ari_k4
    })

    print(
        f"K={k}: converged={converged}, "
        f"converged starts={result['converged_runs']}/{result['total_runs']}"
    )

kmedoids_k_sweep = pd.DataFrame(kmedoids_sweep_rows).set_index("k")
print("\nK-Medoids K-sweep results:")
display(kmedoids_k_sweep.round(6))

nonconverged_kmedoids_k = kmedoids_k_sweep.index[
    ~kmedoids_k_sweep["converged"]
].tolist()

if nonconverged_kmedoids_k:
    print(
        "The following K values did not converge and their silhouette, "
        "cluster-size and ARI fields are intentionally missing: "
        f"{nonconverged_kmedoids_k}"
    )
else:
    print("All selected K-Medoids K=2–10 solutions converged.")

Path("tables_outputs").mkdir(exist_ok=True)
kmedoids_k_sweep.to_csv(
    "tables_outputs/supplementary_kmedoids_k_sweep.csv"
)

converged_kmedoids = kmedoids_k_sweep[kmedoids_k_sweep["converged"]]
plt.figure(figsize=(7, 4.5))
plt.plot(
    converged_kmedoids.index,
    converged_kmedoids["silhouette"],
    marker="o"
)
plt.xlabel("Number of medoid clusters (K)")
plt.ylabel("Silhouette")
plt.title("Converged Supplementary K-Medoids Solutions")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

# The distance matrix is no longer needed after the K=4 and K-sweep checks.
del KMEDOIDS_DISTANCE_MATRIX


In [ ]:
# ============================================================
# SUPPLEMENTARY FUZZY C-MEANS K-SWEEP (K=2–10)
# Only converged solutions with K occupied hard clusters are
# scored and interpreted.
# ============================================================

if "fuzzy_c_means_multi_start" not in globals():
    raise RuntimeError(
        "Run the converged Fuzzy C-Means K=4 cell first."
    )

fcm_sweep_rows = []

for k in range(2, 11):

    result = fuzzy_c_means_multi_start(
        X_clean,
        n_clusters=k,
        m=2.0,
        n_init=10,
        max_iter=2000,
        tol=1e-6,
        random_state=42 + k
    )

    converged = bool(result["converged"])
    labels_k = np.asarray(result["labels"], dtype=int)

    # Default values for solutions that cannot be interpreted.
    silhouette_k = np.nan
    fpc = np.nan
    partition_entropy = np.nan
    smallest_hard_cluster = np.nan
    largest_hard_cluster = np.nan
    occupied_hard_clusters = np.nan
    hard_partition_complete = False

    if converged:

        cluster_sizes = np.bincount(
            labels_k,
            minlength=k
        )

        occupied_hard_clusters = int(
            np.count_nonzero(cluster_sizes)
        )

        hard_partition_complete = bool(
            occupied_hard_clusters == k
        )

        membership_k = np.asarray(
            result["membership"],
            dtype=float
        )

        # Verify that each household's memberships sum to one.
        if not np.allclose(
            membership_k.sum(axis=1),
            1.0,
            atol=1e-6
        ):
            raise RuntimeError(
                f"FCM memberships do not sum to one for K={k}."
            )

        # Fuzzy Partition Coefficient:
        # larger values indicate crisper memberships.
        fpc = float(
            np.mean(
                np.sum(
                    membership_k ** 2,
                    axis=1
                )
            )
        )

        # Partition entropy:
        # larger values indicate more diffuse memberships.
        partition_entropy = float(
            -np.mean(
                np.sum(
                    membership_k
                    * np.log(membership_k + 1e-12),
                    axis=1
                )
            )
        )

        smallest_hard_cluster = int(
            cluster_sizes.min()
        )

        largest_hard_cluster = int(
            cluster_sizes.max()
        )

        # A hard-label silhouette is meaningful only when all
        # requested fuzzy clusters have at least one assigned household.
        if hard_partition_complete:
            silhouette_k = float(
                silhouette_score(
                    X_clean,
                    labels_k
                )
            )

    fcm_sweep_rows.append({
        "k": k,
        "selected_run_converged": converged,
        "converged_runs": int(
            result["converged_runs"]
        ),
        "total_runs": int(
            result["total_runs"]
        ),
        "selected_seed": int(
            result["seed"]
        ),
        "selected_iterations": int(
            result["n_iter"]
        ),
        "terminal_objective": float(
            result["objective"]
        ),
        "occupied_hard_clusters": (
            occupied_hard_clusters
        ),
        "hard_partition_complete": (
            hard_partition_complete
        ),
        "hard_label_silhouette": silhouette_k,
        "fuzzy_partition_coefficient": fpc,
        "partition_entropy": partition_entropy,
        "smallest_hard_cluster": (
            smallest_hard_cluster
        ),
        "largest_hard_cluster": (
            largest_hard_cluster
        )
    })

    print(
        f"K={k}: "
        f"converged={converged}, "
        f"converged starts="
        f"{result['converged_runs']}/"
        f"{result['total_runs']}, "
        f"iterations={result['n_iter']}, "
        f"occupied hard clusters="
        f"{occupied_hard_clusters}"
    )


# ============================================================
# RESULTS TABLE
# ============================================================

fcm_k_sweep = (
    pd.DataFrame(fcm_sweep_rows)
    .set_index("k")
    .sort_index()
)

print("\nFuzzy C-Means K-sweep results:")
display(fcm_k_sweep.round(6))


# ============================================================
# CONVERGENCE AND HARD-PARTITION AUDIT
# ============================================================

nonconverged_fcm_k = fcm_k_sweep.index[
    ~fcm_k_sweep["selected_run_converged"]
].tolist()

incomplete_hard_partition_k = fcm_k_sweep.index[
    fcm_k_sweep["selected_run_converged"]
    & ~fcm_k_sweep["hard_partition_complete"]
].tolist()

interpretable_fcm_k = fcm_k_sweep.index[
    fcm_k_sweep["selected_run_converged"]
    & fcm_k_sweep["hard_partition_complete"]
].tolist()

if nonconverged_fcm_k:
    print(
        "\nNon-converged FCM solutions:",
        nonconverged_fcm_k
    )
    print(
        "Their silhouette, FPC, partition-entropy and "
        "cluster-size results are not interpreted."
    )
else:
    print("\nAll selected FCM solutions converged.")

if incomplete_hard_partition_k:
    print(
        "\nConverged solutions with fewer than K occupied "
        "hard clusters:",
        incomplete_hard_partition_k
    )
    print(
        "Their fuzzy statistics are retained, but their "
        "hard-label silhouettes are not interpreted."
    )
else:
    print(
        "\nAll converged FCM solutions produced complete "
        "hard partitions."
    )

print(
    "\nK values eligible for hard-partition comparison:",
    interpretable_fcm_k
)


# ============================================================
# SAVE RESULTS
# ============================================================

Path("tables_outputs").mkdir(
    parents=True,
    exist_ok=True
)

fcm_k_sweep.to_csv(
    "tables_outputs/supplementary_fcm_k_sweep.csv"
)


# ============================================================
# PLOT ONLY INTERPRETABLE HARD PARTITIONS
# ============================================================

interpretable_fcm = fcm_k_sweep[
    fcm_k_sweep["selected_run_converged"]
    & fcm_k_sweep["hard_partition_complete"]
].copy()

if not interpretable_fcm.empty:

    plt.figure(figsize=(7, 4.5))

    plt.plot(
        interpretable_fcm.index,
        interpretable_fcm[
            "hard_label_silhouette"
        ],
        marker="o",
        label="Hard-label silhouette"
    )

    plt.xlabel(
        "Number of fuzzy clusters (K)"
    )
    plt.ylabel("Silhouette")
    plt.title(
        "Interpretable Fuzzy C-Means Solutions"
    )
    plt.xticks(
        interpretable_fcm.index
    )
    plt.grid(
        True,
        linestyle="--",
        alpha=0.5
    )
    plt.tight_layout()
    plt.show()

else:
    print(
        "\nNo FCM solution both converged and produced "
        "a complete hard partition. No silhouette plot "
        "was generated."
    )

## Supplementary temporal robustness and coverage checks

The following cells document golden-window coverage, strict early-vs-late temporal split validation, and corrected monthly dominant-state retention.

In [ ]:
# ============================================================
# GOLDEN_DF VS FINAL CLUSTERED POPULATION AUDIT
# Purpose: verify that final clustered households are contained in the
# golden-window telemetry source before temporal validation.
# ============================================================

print("=" * 80)
print("GOLDEN_DF VS FINAL CLUSTERED POPULATION AUDIT")
print("=" * 80)

required_objects = ["golden_df", "clean_fm"]
missing = [obj for obj in required_objects if obj not in globals()]
if missing:
    raise RuntimeError(f"Missing required objects: {missing}. Run the main pipeline first.")

golden_hhs = set(golden_df["household_ID"].astype(str).unique())

clean_fm_lookup = clean_fm.copy()
clean_fm_lookup.index = clean_fm_lookup.index.astype(str)
final_hhs = set(clean_fm_lookup.index)

print("Golden_df date range:")
print("Min meter_dt:", golden_df["meter_dt"].min())
print("Max meter_dt:", golden_df["meter_dt"].max())

print("\nHousehold counts:")
print(f"Households in golden_df: {len(golden_hhs):,}")
print(f"Final clustered households in clean_fm: {len(final_hhs):,}")

print("\nPopulation relationship:")
print(f"Final households present in golden_df: {len(final_hhs & golden_hhs):,}")
print(f"Final households missing from golden_df: {len(final_hhs - golden_hhs):,}")
print(f"Golden_df households not in final clustering: {len(golden_hhs - final_hhs):,}")

if len(final_hhs - golden_hhs) == 0:
    print("\nPASS: All final clustered households are present in golden_df.")
else:
    print("\nWARNING: Some final clustered households are missing from golden_df.")

print("\nInterpretation:")
print("golden_df is the time-series source window.")
print("clean_fm is the final locked household feature matrix used for clustering.")
print("Final temporal validation should use golden_df restricted to clean_fm households.")


In [ ]:
# ============================================================
# GOLDEN-WINDOW TEMPORAL COVERAGE AUDIT
# Valid month: at least 20 days, each with at least 90 valid intervals.
# Uses external grouping vectors and avoids copying golden_df.
# ============================================================

print("=" * 80)
print("GOLDEN-WINDOW TEMPORAL COVERAGE AUDIT")
print("=" * 80)

if "golden_df" not in globals():
    raise RuntimeError("Run the main pipeline first.")

golden_year_month = pd.Series(
    golden_df["meter_dt"].dt.to_period("M"),
    index=golden_df.index,
    name="year_month"
)

daily_coverage = (
    golden_df["true_15min_kwh"]
    .groupby(
        [
            golden_df["household_ID"],
            golden_year_month,
            golden_df["date_only"]
        ],
        observed=True
    )
    .agg(
        valid_intervals="count",
        daily_kwh="sum"
    )
    .reset_index()
)
daily_coverage["is_valid_day"] = (
    daily_coverage["valid_intervals"] >= 90
)

monthly_coverage = (
    daily_coverage
    .groupby(
        ["household_ID", "year_month"],
        observed=True
    )
    .agg(
        observed_days=("date_only", "nunique"),
        valid_days=("is_valid_day", "sum"),
        total_valid_intervals=("valid_intervals", "sum"),
        total_kwh=("daily_kwh", "sum")
    )
    .reset_index()
)
monthly_coverage["passes_20_valid_day_threshold"] = (
    monthly_coverage["valid_days"] >= 20
)

coverage_summary_by_month = (
    monthly_coverage
    .groupby("year_month", observed=True)
    .agg(
        households_present=("household_ID", "nunique"),
        households_passing_20_valid_days=(
            "passes_20_valid_day_threshold",
            "sum"
        ),
        median_observed_days=("observed_days", "median"),
        median_valid_days=("valid_days", "median"),
        median_total_valid_intervals=(
            "total_valid_intervals",
            "median"
        )
    )
    .reset_index()
)

household_valid_months = (
    monthly_coverage.loc[
        monthly_coverage["passes_20_valid_day_threshold"]
    ]
    .groupby("household_ID", observed=True)["year_month"]
    .nunique()
    .rename("valid_month_count")
)

print(
    "\nEligible golden-window households:",
    golden_df["household_ID"].nunique()
)
print("\nMonthly coverage summary:")
display(coverage_summary_by_month)
print("\nValid month count per household summary:")
display(household_valid_months.describe())
print("\nHouseholds by number of valid months:")
display(household_valid_months.value_counts().sort_index())
print(
    "\nThis cell audits coverage only. Monthly archetype states are "
    "assigned later after reconstructing the six features from valid days "
    "and applying the locked annual scaler and centroids."
)

del golden_year_month


In [ ]:
# ============================================================
# STRICT TEMPORAL SPLIT-WINDOW VALIDATION
# Early-window vs late-window archetype reproducibility
# Addresses reviewer concern about temporal cross-validation.
# This does NOT replace the final full-window KMeans model.
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import entropy
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)
from scipy.optimize import linear_sum_assignment

try:
    from IPython.display import display
except Exception:
    display = print

print("=" * 80)
print("STRICT TEMPORAL SPLIT-WINDOW VALIDATION")
print("=" * 80)

# ------------------------------------------------------------
# 0. Required object check
# ------------------------------------------------------------

needed = ["golden_df", "clean_fm", "CORE_FEATURES", "CLIP_BOUNDS", "scaler", "optimal_kmeans"]
missing = [x for x in needed if x not in globals()]

if missing:
    raise RuntimeError(
        f"Missing required objects: {missing}. "
        "Run the master pipeline and final KMeans model first."
    )

FEATURES = list(CORE_FEATURES)

CLUSTER_NAME_MAP = dict(CLUSTER_NAME_MAP)

RANDOM_STATE = 42
N_INIT = 20
EXPECTED_K = 4

# ------------------------------------------------------------
# 0A. Temporal split windows
# ------------------------------------------------------------
# IMPORTANT:
# This does NOT change the main golden window.
# It only uses the actual timestamps already available inside golden_df.
# Main pipeline now uses the corrected exclusive endpoint (< 2024-12-01), so the full final day is available when present.

GOLDEN_AVAILABLE_MIN = pd.to_datetime(golden_df["meter_dt"].min())
GOLDEN_AVAILABLE_MAX = pd.to_datetime(golden_df["meter_dt"].max())

EARLY_START = max(pd.Timestamp("2024-05-01 00:00:00"), GOLDEN_AVAILABLE_MIN)
EARLY_END   = min(pd.Timestamp("2024-07-31 23:59:59"), GOLDEN_AVAILABLE_MAX)

LATE_START  = max(pd.Timestamp("2024-09-01 00:00:00"), GOLDEN_AVAILABLE_MIN)
LATE_END    = GOLDEN_AVAILABLE_MAX

EARLY_WINDOW_LABEL = f"{EARLY_START.date()} to {EARLY_END.date()}"
LATE_WINDOW_LABEL  = f"{LATE_START.date()} to {LATE_END.date()}"

print("Golden dataframe available range:")
print("Minimum meter_dt:", GOLDEN_AVAILABLE_MIN)
print("Maximum meter_dt:", GOLDEN_AVAILABLE_MAX)

print("\nTemporal split validation windows actually used:")
print("Early window:", EARLY_START, "to", EARLY_END)
print("Late window :", LATE_START, "to", LATE_END)

MIN_VALID_DAYS_PER_WINDOW = 30
MIN_VALID_INTERVALS_PER_DAY = 90

out_dir = Path("tables_outputs")
out_dir.mkdir(exist_ok=True)

# ------------------------------------------------------------
# 1. Helper functions
# ------------------------------------------------------------

def align_labels(reference_labels, candidate_labels):
    """
    Align candidate cluster IDs to reference cluster IDs using Hungarian matching.
    ARI/NMI do not need alignment, but agreement and crosstabs do.
    """
    ref = np.asarray(reference_labels)
    cand = np.asarray(candidate_labels)

    ref_unique = np.sort(np.unique(ref))
    cand_unique = np.sort(np.unique(cand))

    ct = pd.crosstab(
        pd.Series(ref, name="reference"),
        pd.Series(cand, name="candidate")
    ).reindex(index=ref_unique, columns=cand_unique, fill_value=0)

    row_ind, col_ind = linear_sum_assignment(-ct.to_numpy())

    mapping = {
        ct.columns[j]: ct.index[i]
        for i, j in zip(row_ind, col_ind)
    }

    aligned = np.array([mapping[x] for x in cand])

    return aligned, mapping


def build_window_feature_matrix(
    source_df,
    start_date,
    end_date,
    window_name,
    min_valid_days=30,
    min_valid_intervals_per_day=90
):
    """
    Build household-level feature matrix for a specific time window.
    Matches the locked annual six-feature logic as closely as possible.
    Restricts to final locked clean_fm households and uses only days meeting
    the minimum valid-interval threshold for feature construction.
    """

    clean_fm_lookup = clean_fm.copy()
    clean_fm_lookup.index = clean_fm_lookup.index.astype(str)
    final_hhs = set(clean_fm_lookup.index)

    required_cols = ["household_ID", "meter_dt", "true_15min_kwh"]

    for col in required_cols:
        if col not in source_df.columns:
            raise ValueError(f"Required column missing from golden_df: {col}")

    optional_cols = []
    for col in ["is_non_working_day", "ceb_block"]:
        if col in source_df.columns:
            optional_cols.append(col)

    slim_cols = required_cols + optional_cols

    temp = source_df.loc[
        (source_df["meter_dt"] >= pd.to_datetime(start_date)) &
        (source_df["meter_dt"] <= pd.to_datetime(end_date)),
        slim_cols
    ].copy()

    temp["household_ID"] = temp["household_ID"].astype(str)
    temp = temp[temp["household_ID"].isin(final_hhs)].copy()

    if temp.empty:
        raise RuntimeError(f"{window_name}: no rows after date and final-household filtering.")

    temp["true_15min_kwh"] = pd.to_numeric(
        temp["true_15min_kwh"],
        errors="coerce"
    ).astype("float32")

    temp["date_only"] = temp["meter_dt"].dt.normalize()

    # Valid-day filter, matching the annual 90-valid-interval logic.
    day_counts = (
        temp.groupby(["household_ID", "date_only"], observed=True)["true_15min_kwh"]
        .count()
        .reset_index(name="valid_intervals")
    )

    valid_days = day_counts[
        day_counts["valid_intervals"] >= min_valid_intervals_per_day
    ]

    valid_hh_day_counts = valid_days.groupby("household_ID", observed=True).size()
    valid_hhs = valid_hh_day_counts[
        valid_hh_day_counts >= min_valid_days
    ].index

    # Apply the valid-day rule to the rows used for feature construction,
    # not only to household eligibility.
    valid_day_keys = (
        valid_days.loc[
            valid_days["household_ID"].isin(valid_hhs),
            ["household_ID", "date_only"]
        ]
        .drop_duplicates()
    )

    temp = temp.merge(
        valid_day_keys,
        on=["household_ID", "date_only"],
        how="inner",
        validate="many_to_one"
    )

    if temp.empty:
        raise RuntimeError(
            f"{window_name}: no households passed the {min_valid_days}-valid-day filter."
        )

    retained_day_counts = (
        temp.groupby(["household_ID", "date_only"], observed=True)["true_15min_kwh"]
        .count()
    )
    assert retained_day_counts.min() >= min_valid_intervals_per_day, (
        f"{window_name}: incomplete days entered temporal feature construction."
    )

    # Non-working day logic.
    if "is_non_working_day" not in temp.columns:
        if "sl_holidays_2024" in globals():
            holidays = pd.to_datetime(list(sl_holidays_2024)).normalize()
            temp["is_non_working_day"] = (
                (temp["meter_dt"].dt.dayofweek >= 5) |
                (temp["date_only"].isin(holidays))
            )
        else:
            print(f"WARNING: {window_name}: sl_holidays_2024 not found. Using weekend-only.")
            temp["is_non_working_day"] = temp["meter_dt"].dt.dayofweek >= 5

    temp["is_non_working_day"] = temp["is_non_working_day"].astype(bool)

    # CEB block logic.
    if "ceb_block" not in temp.columns:
        minutes_since_midnight = (
            temp["meter_dt"].dt.hour * 60 + temp["meter_dt"].dt.minute
        )

        temp["ceb_block"] = np.select(
            [
                (minutes_since_midnight >= 5 * 60 + 30) &
                (minutes_since_midnight < 18 * 60 + 30),

                (minutes_since_midnight >= 18 * 60 + 30) &
                (minutes_since_midnight < 22 * 60 + 30)
            ],
            ["Day", "Peak"],
            default="Off-Peak"
        )

    temp["interval_id"] = (
        temp["meter_dt"].dt.hour * 4 +
        temp["meter_dt"].dt.minute // 15
    ).astype("int8")

    rows = []

    for hh_id, grp in temp.groupby("household_ID", observed=True, sort=False):
        total_kwh = grp["true_15min_kwh"].sum()

        if total_kwh < 0.1:
            continue

        daily_block_usage = (
            grp.groupby(["date_only", "ceb_block"], observed=True)["true_15min_kwh"]
            .sum()
            .unstack(fill_value=0)
        )

        for block in ["Day", "Peak", "Off-Peak"]:
            if block not in daily_block_usage.columns:
                daily_block_usage[block] = 0.0

        avg_daily_block = daily_block_usage[["Day", "Peak", "Off-Peak"]].mean()
        total_avg_daily = avg_daily_block.sum()

        if total_avg_daily < 0.01:
            continue

        profile = (
            grp.groupby("interval_id")["true_15min_kwh"]
            .mean()
            .reindex(range(96))
        )

        if profile.isna().any():
            continue

        raw_mean = grp["true_15min_kwh"].mean()
        raw_max = max(grp["true_15min_kwh"].max(), 0.01)

        daily_totals = (
            grp.groupby(["date_only", "is_non_working_day"])["true_15min_kwh"]
            .sum()
            .reset_index()
        )

        day_type_avg = daily_totals.groupby("is_non_working_day")["true_15min_kwh"].mean()

        non_working_avg = day_type_avg.get(True, np.nan)
        working_avg = day_type_avg.get(False, np.nan)

        if pd.isna(non_working_avg) or pd.isna(working_avg) or working_avg <= 0:
            continue

        rows.append({
            "household_ID": hh_id,
            "evening_peak_ratio": avg_daily_block["Peak"] / max(total_avg_daily, 0.01),
            "night_baseload_ratio": avg_daily_block["Off-Peak"] / max(total_avg_daily, 0.01),
            "load_uniformity": entropy(profile.values + 1e-9),
            "load_factor": raw_mean / raw_max,
            "leisure_usage_ratio": np.log1p(non_working_avg / max(working_avg, 0.01)),
            "log_daily_kwh": np.log1p(total_avg_daily)
        })

    fm = pd.DataFrame(rows)

    if fm.empty:
        raise RuntimeError(f"{window_name}: feature matrix is empty after feature extraction.")

    fm = fm.set_index("household_ID")
    fm.index = fm.index.astype(str)

    fm = fm.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURES)

    # Clip to the explicitly locked annual 1st–99th percentile bounds.
    for col in FEATURES:
        lower, upper = CLIP_BOUNDS[col]
        fm[col] = fm[col].clip(lower=lower, upper=upper)

    print(
        f"{window_name}: {len(fm):,} households retained "
        f"from {pd.to_datetime(start_date).date()} to {pd.to_datetime(end_date).date()}"
    )

    return fm[FEATURES].copy()


# ------------------------------------------------------------
# 2. Build early and late feature matrices
# ------------------------------------------------------------

early_fm = build_window_feature_matrix(
    golden_df,
    EARLY_START,
    EARLY_END,
    "Early window",
    min_valid_days=MIN_VALID_DAYS_PER_WINDOW,
    min_valid_intervals_per_day=MIN_VALID_INTERVALS_PER_DAY
)

late_fm = build_window_feature_matrix(
    golden_df,
    LATE_START,
    LATE_END,
    "Late window",
    min_valid_days=MIN_VALID_DAYS_PER_WINDOW,
    min_valid_intervals_per_day=MIN_VALID_INTERVALS_PER_DAY
)

common_hhs = early_fm.index.intersection(late_fm.index)

print(f"\nCommon households in both early and late windows: {len(common_hhs):,}")

if len(common_hhs) < 100:
    raise RuntimeError(
        "Too few common households for reliable temporal split validation. "
        "Consider reducing MIN_VALID_DAYS_PER_WINDOW."
    )

early_common = early_fm.loc[common_hhs].copy()
late_common = late_fm.loc[common_hhs].copy()

# ------------------------------------------------------------
# 3. Locked-model temporal assignment comparison
# Uses final full-window scaler and KMeans centroids.
# ------------------------------------------------------------

locked_dtype = optimal_kmeans.cluster_centers_.dtype
X_early_locked = scaler.transform(early_common[FEATURES]).astype(locked_dtype, copy=False)
X_late_locked = scaler.transform(late_common[FEATURES]).astype(locked_dtype, copy=False)

early_locked_labels = optimal_kmeans.predict(X_early_locked)
late_locked_labels = optimal_kmeans.predict(X_late_locked)

locked_same_cluster = early_locked_labels == late_locked_labels

locked_temporal_summary = pd.DataFrame([{
    "validation_type": "locked_model_prediction",
    "early_window": EARLY_WINDOW_LABEL,
    "late_window": LATE_WINDOW_LABEL,
    "common_households": len(common_hhs),
    "same_cluster_share": float(np.mean(locked_same_cluster)),
    "changed_cluster_share": float(1 - np.mean(locked_same_cluster)),
    "ari_early_vs_late": float(adjusted_rand_score(early_locked_labels, late_locked_labels)),
    "nmi_early_vs_late": float(normalized_mutual_info_score(early_locked_labels, late_locked_labels))
}])

locked_alignment_table = pd.crosstab(
    pd.Series(early_locked_labels, name="Early locked assignment"),
    pd.Series(late_locked_labels, name="Late locked assignment")
)

locked_alignment_percent = locked_alignment_table.div(
    locked_alignment_table.sum(axis=1),
    axis=0
) * 100

# ------------------------------------------------------------
# 4. Independent re-clustering comparison
# Fits separate KMeans K=4 models to early and late windows.
# ------------------------------------------------------------

split_scaler = StandardScaler()
combined_split = pd.concat([early_common[FEATURES], late_common[FEATURES]], axis=0)
split_scaler.fit(combined_split)

X_early_split = split_scaler.transform(early_common[FEATURES])
X_late_split = split_scaler.transform(late_common[FEATURES])

early_kmeans = KMeans(
    n_clusters=EXPECTED_K,
    random_state=RANDOM_STATE,
    n_init=N_INIT
)

late_kmeans = KMeans(
    n_clusters=EXPECTED_K,
    random_state=RANDOM_STATE,
    n_init=N_INIT
)

early_independent_labels = early_kmeans.fit_predict(X_early_split)
late_independent_raw_labels = late_kmeans.fit_predict(X_late_split)

late_independent_aligned_labels, late_to_early_mapping = align_labels(
    early_independent_labels,
    late_independent_raw_labels
)

independent_ari = adjusted_rand_score(
    early_independent_labels,
    late_independent_raw_labels
)

independent_nmi = normalized_mutual_info_score(
    early_independent_labels,
    late_independent_raw_labels
)

independent_agreement = np.mean(
    early_independent_labels == late_independent_aligned_labels
)

independent_summary = pd.DataFrame([{
    "validation_type": "independent_reclustering",
    "early_window": EARLY_WINDOW_LABEL,
    "late_window": LATE_WINDOW_LABEL,
    "common_households": len(common_hhs),
    "early_silhouette": float(silhouette_score(X_early_split, early_independent_labels)),
    "late_silhouette": float(silhouette_score(X_late_split, late_independent_raw_labels)),
    "early_davies_bouldin": float(davies_bouldin_score(X_early_split, early_independent_labels)),
    "late_davies_bouldin": float(davies_bouldin_score(X_late_split, late_independent_raw_labels)),
    "early_calinski_harabasz": float(calinski_harabasz_score(X_early_split, early_independent_labels)),
    "late_calinski_harabasz": float(calinski_harabasz_score(X_late_split, late_independent_raw_labels)),
    "ari_early_vs_late": float(independent_ari),
    "nmi_early_vs_late": float(independent_nmi),
    "aligned_agreement_early_vs_late": float(independent_agreement)
}])

independent_alignment_table = pd.crosstab(
    pd.Series(early_independent_labels, name="Early independent KMeans"),
    pd.Series(late_independent_aligned_labels, name="Late independent KMeans aligned")
)

independent_alignment_percent = independent_alignment_table.div(
    independent_alignment_table.sum(axis=1),
    axis=0
) * 100

# ------------------------------------------------------------
# 5. Compare early/late locked predictions to final annual clusters
# ------------------------------------------------------------

clean_fm_lookup = clean_fm.copy()
clean_fm_lookup.index = clean_fm_lookup.index.astype(str)

annual_labels_common = (
    clean_fm_lookup.loc[common_hhs, "final_cluster"]
    .astype(int)
    .to_numpy()
)

early_vs_annual = pd.DataFrame([{
    "comparison": "early_locked_vs_final_annual",
    "early_window": EARLY_WINDOW_LABEL,
    "late_window": LATE_WINDOW_LABEL,
    "common_households": len(common_hhs),
    "agreement": float(np.mean(early_locked_labels == annual_labels_common)),
    "ari": float(adjusted_rand_score(annual_labels_common, early_locked_labels)),
    "nmi": float(normalized_mutual_info_score(annual_labels_common, early_locked_labels))
}])

late_vs_annual = pd.DataFrame([{
    "comparison": "late_locked_vs_final_annual",
    "early_window": EARLY_WINDOW_LABEL,
    "late_window": LATE_WINDOW_LABEL,
    "common_households": len(common_hhs),
    "agreement": float(np.mean(late_locked_labels == annual_labels_common)),
    "ari": float(adjusted_rand_score(annual_labels_common, late_locked_labels)),
    "nmi": float(normalized_mutual_info_score(annual_labels_common, late_locked_labels))
}])

annual_comparison_summary = pd.concat(
    [early_vs_annual, late_vs_annual],
    ignore_index=True
)

# ------------------------------------------------------------
# 6. Save outputs
# ------------------------------------------------------------

locked_temporal_summary.to_csv(
    out_dir / "supplementary_temporal_split_locked_model_summary.csv",
    index=False
)

locked_alignment_table.to_csv(
    out_dir / "supplementary_temporal_split_locked_model_alignment_table.csv"
)

locked_alignment_percent.to_csv(
    out_dir / "supplementary_temporal_split_locked_model_alignment_percent.csv"
)

independent_summary.to_csv(
    out_dir / "supplementary_temporal_split_independent_reclustering_summary.csv",
    index=False
)

independent_alignment_table.to_csv(
    out_dir / "supplementary_temporal_split_independent_alignment_table.csv"
)

independent_alignment_percent.to_csv(
    out_dir / "supplementary_temporal_split_independent_alignment_percent.csv"
)

annual_comparison_summary.to_csv(
    out_dir / "supplementary_temporal_split_vs_annual_summary.csv",
    index=False
)

temporal_household_output = pd.DataFrame({
    "household_ID": common_hhs.astype(str),
    "final_annual_cluster": annual_labels_common,
    "early_locked_cluster": early_locked_labels,
    "late_locked_cluster": late_locked_labels,
    "locked_same_cluster_early_late": locked_same_cluster,
    "early_independent_cluster": early_independent_labels,
    "late_independent_cluster_aligned": late_independent_aligned_labels
})

temporal_household_output["final_annual_cluster_name"] = (
    temporal_household_output["final_annual_cluster"].map(CLUSTER_NAME_MAP)
)

temporal_household_output.to_csv(
    out_dir / "supplementary_temporal_split_household_assignments.csv",
    index=False
)

# ------------------------------------------------------------
# 7. Display outputs
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TEMPORAL SPLIT-WINDOW VALIDATION COMPLETED")
print("=" * 80)

print("\nA. Locked final-model early vs late assignment summary:")
display(locked_temporal_summary)

print("\nLocked final-model early vs late alignment table:")
display(locked_alignment_table)

print("\nLocked final-model early vs late alignment table (% within early assignment):")
display(locked_alignment_percent.round(2))

print("\nB. Independent early-window vs late-window re-clustering summary:")
display(independent_summary)

print("\nIndependent early vs late alignment table:")
display(independent_alignment_table)

print("\nIndependent early vs late alignment table (% within early model):")
display(independent_alignment_percent.round(2))

print("\nC. Early/late locked assignments compared with final annual cluster:")
display(annual_comparison_summary)

print("\nSaved temporal split-window validation outputs to:")
print(out_dir.resolve())


In [ ]:
# ============================================================
# MEMORY-SAFE CORRECTED MONTHLY TEMPORAL STABILITY CHECK
# Locked-population monthly feature matrix + dominant-state retention.
# This is supplementary temporal consistency, not strict temporal CV.
# The consecutive-transition result is produced in the later authoritative
# corrected monthly transition cell to avoid duplicate persistence numbers.
# ============================================================

import gc
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import entropy

try:
    from IPython.display import display
except Exception:
    display = print

print("=" * 80)
print("MEMORY-SAFE CORRECTED MONTHLY TEMPORAL STABILITY CHECK")
print("=" * 80)

# ------------------------------------------------------------
# 0. Required object check
# ------------------------------------------------------------

needed = ["golden_df", "clean_fm", "CORE_FEATURES", "CLIP_BOUNDS", "scaler", "optimal_kmeans"]
missing = [x for x in needed if x not in globals()]

if missing:
    raise RuntimeError(
        f"Missing required objects: {missing}. "
        "Run the final clustering pipeline first."
    )

FEATURES = list(CORE_FEATURES)

CLUSTER_NAME_MAP = dict(CLUSTER_NAME_MAP)

out_dir = Path("tables_outputs")
out_dir.mkdir(exist_ok=True)

# ------------------------------------------------------------
# 1. Prepare final locked household IDs without copying golden_df
# ------------------------------------------------------------

clean_fm_lookup = clean_fm.copy()
clean_fm_lookup.index = clean_fm_lookup.index.astype(str)

final_hhs_str = set(clean_fm_lookup.index)

print(f"Final locked clustered households: {len(final_hhs_str):,}")

required_cols = ["household_ID", "meter_dt", "true_15min_kwh"]

for col in required_cols:
    if col not in golden_df.columns:
        raise ValueError(f"Required column missing from golden_df: {col}")

# Optional only if already exists.
base_cols = required_cols.copy()
if "is_non_working_day" in golden_df.columns:
    base_cols.append("is_non_working_day")

# ------------------------------------------------------------
# 2. Build monthly feature rows month-by-month to avoid RAM crash
# ------------------------------------------------------------

print("\n--- PHASE 1: BUILDING MONTHLY FEATURE MATRIX MONTH-BY-MONTH ---")

golden_min = pd.to_datetime(golden_df["meter_dt"].min())
golden_max = pd.to_datetime(golden_df["meter_dt"].max())

months = pd.period_range(
    golden_min.to_period("M"),
    golden_max.to_period("M"),
    freq="M"
)

print(f"Golden available range: {golden_min} to {golden_max}")
print(f"Months processed: {[str(m) for m in months]}")

monthly_rows = []

for ym in months:
    month_start = ym.to_timestamp()
    month_end_exclusive = (ym + 1).to_timestamp()

    print(f"\nProcessing month: {ym}")

    month_mask = (
        (golden_df["meter_dt"] >= month_start) &
        (golden_df["meter_dt"] < month_end_exclusive)
    )

    month_df = golden_df.loc[month_mask, base_cols].copy()

    if month_df.empty:
        print(f"  No rows for {ym}. Skipped.")
        continue

    month_df["household_ID"] = month_df["household_ID"].astype(str)
    month_df = month_df[month_df["household_ID"].isin(final_hhs_str)].copy()

    if month_df.empty:
        print(f"  No final locked households for {ym}. Skipped.")
        del month_df
        gc.collect()
        continue

    month_df["true_15min_kwh"] = pd.to_numeric(
        month_df["true_15min_kwh"],
        errors="coerce"
    ).astype("float32")

    month_df = month_df.dropna(subset=["true_15min_kwh"])

    month_df["date_only"] = month_df["meter_dt"].dt.normalize()

    # Strict day-level completeness: first retain days with at least 90
    # valid intervals, then require at least 20 such days in the month.
    monthly_day_counts = (
        month_df.groupby(["household_ID", "date_only"], observed=True)["true_15min_kwh"]
        .count()
        .reset_index(name="valid_intervals")
    )

    valid_month_days = monthly_day_counts[
        monthly_day_counts["valid_intervals"] >= 90
    ].copy()

    eligible_month_hhs = (
        valid_month_days.groupby("household_ID", observed=True).size()
    )
    eligible_month_hhs = eligible_month_hhs[
        eligible_month_hhs >= 20
    ].index

    valid_month_day_keys = (
        valid_month_days.loc[
            valid_month_days["household_ID"].isin(eligible_month_hhs),
            ["household_ID", "date_only"]
        ]
        .drop_duplicates()
    )

    month_df = month_df.merge(
        valid_month_day_keys,
        on=["household_ID", "date_only"],
        how="inner",
        validate="many_to_one"
    )

    if month_df.empty:
        print(f"  No household-months passed the strict valid-day rule for {ym}.")
        del month_df
        gc.collect()
        continue

    assert (
        month_df.groupby(["household_ID", "date_only"], observed=True)["true_15min_kwh"]
        .count()
        .min() >= 90
    )

    if "is_non_working_day" in month_df.columns:
        day_type_col = "is_non_working_day"
        month_df[day_type_col] = month_df[day_type_col].astype(bool)
    else:
        if "sl_holidays_2024" in globals():
            holidays = pd.to_datetime(list(sl_holidays_2024)).normalize()
            month_df["is_non_working_day"] = (
                (month_df["meter_dt"].dt.dayofweek >= 5) |
                (month_df["date_only"].isin(holidays))
            )
        else:
            month_df["is_non_working_day"] = month_df["meter_dt"].dt.dayofweek >= 5

        day_type_col = "is_non_working_day"

    month_df["minute_of_day"] = (
        month_df["meter_dt"].dt.hour * 60 +
        month_df["meter_dt"].dt.minute
    ).astype("int16")

    month_df["interval_id"] = (
        month_df["meter_dt"].dt.hour * 4 +
        month_df["meter_dt"].dt.minute // 15
    ).astype("int8")

    kept_hh_months = 0

    for hh_id, grp in month_df.groupby("household_ID", observed=True, sort=False):

        # The dataframe already contains only valid days; retain household-
        # months with at least 20 such days.
        if grp["date_only"].nunique() < 20:
            continue

        total_kwh = grp["true_15min_kwh"].sum()

        if total_kwh < 0.1:
            continue

        unique_days = grp["date_only"].nunique()

        if unique_days <= 0:
            continue

        peak_kwh = grp[
            (grp["minute_of_day"] >= 18 * 60 + 30) &
            (grp["minute_of_day"] < 22 * 60 + 30)
        ]["true_15min_kwh"].sum()

        offpeak_kwh = grp[
            (grp["minute_of_day"] >= 22 * 60 + 30) |
            (grp["minute_of_day"] < 5 * 60 + 30)
        ]["true_15min_kwh"].sum()

        profile = (
            grp.groupby("interval_id")["true_15min_kwh"]
            .mean()
            .reindex(range(96))
        )

        if profile.isna().any():
            continue

        raw_mean = grp["true_15min_kwh"].mean()
        raw_max = max(grp["true_15min_kwh"].max(), 0.01)

        daily_totals = (
            grp.groupby(["date_only", day_type_col])["true_15min_kwh"]
            .sum()
            .reset_index()
        )

        day_type_avg = daily_totals.groupby(day_type_col)["true_15min_kwh"].mean()

        non_working_avg = day_type_avg.get(True, np.nan)
        working_avg = day_type_avg.get(False, np.nan)

        if pd.isna(non_working_avg) or pd.isna(working_avg) or working_avg <= 0:
            continue

        monthly_rows.append({
            "household_ID": hh_id,
            "year_month": ym,
            "evening_peak_ratio": peak_kwh / max(total_kwh, 0.01),
            "night_baseload_ratio": offpeak_kwh / max(total_kwh, 0.01),
            "load_uniformity": entropy(profile.values + 1e-9),
            "load_factor": raw_mean / raw_max,
            "leisure_usage_ratio": np.log1p(non_working_avg / max(working_avg, 0.01)),
            "log_daily_kwh": np.log1p(total_kwh / unique_days)
        })

        kept_hh_months += 1

    print(f"  Kept household-months: {kept_hh_months:,}")

    del month_df
    gc.collect()

# ------------------------------------------------------------
# 3. Create monthly feature matrix
# ------------------------------------------------------------

monthly_retention_fm = pd.DataFrame(monthly_rows)

if monthly_retention_fm.empty:
    raise RuntimeError("Corrected monthly feature matrix is empty. Check filtering thresholds.")

monthly_retention_fm = monthly_retention_fm.set_index(["household_ID", "year_month"])
monthly_retention_fm = monthly_retention_fm.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURES)

for col in FEATURES:
    lower, upper = CLIP_BOUNDS[col]
    monthly_retention_fm[col] = monthly_retention_fm[col].clip(lower=lower, upper=upper)

print("\nBuilt corrected monthly matrix:")
print(f"  Household-months: {len(monthly_retention_fm):,}")
print(f"  Households represented: {monthly_retention_fm.reset_index()['household_ID'].nunique():,}")

# ------------------------------------------------------------
# 4. Predict monthly clusters using locked final scaler and KMeans
# ------------------------------------------------------------

print("\n--- PHASE 2: LOCKED-MODEL MONTHLY CLUSTER ASSIGNMENT ---")

X_monthly_retention = scaler.transform(
    monthly_retention_fm[FEATURES]
).astype(optimal_kmeans.cluster_centers_.dtype, copy=False)
monthly_retention_fm["monthly_cluster"] = optimal_kmeans.predict(X_monthly_retention)
monthly_retention_fm["monthly_cluster_name"] = monthly_retention_fm["monthly_cluster"].map(CLUSTER_NAME_MAP)

retention_state_df = monthly_retention_fm.reset_index()

month_counts = retention_state_df.groupby("household_ID", observed=True)["year_month"].nunique()
valid_hhs = month_counts[month_counts >= 3].index

retention_state_df = retention_state_df[
    retention_state_df["household_ID"].isin(valid_hhs)
].copy()

print(f"Households evaluated with ≥3 valid monthly states: {len(valid_hhs):,}")

# ------------------------------------------------------------
# 5. Dominant-state retention
# ------------------------------------------------------------

dominant_clusters = (
    retention_state_df.groupby("household_ID", observed=True)["monthly_cluster"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
    .rename("dominant_cluster")
)

retention_state_df = retention_state_df.merge(
    dominant_clusters,
    on="household_ID",
    how="left"
)

retention_state_df["dominant_cluster_name"] = retention_state_df["dominant_cluster"].map(CLUSTER_NAME_MAP)

retention_state_df["is_stable"] = (
    retention_state_df["monthly_cluster"] == retention_state_df["dominant_cluster"]
)

household_retention = (
    retention_state_df.groupby("household_ID", observed=True)["is_stable"]
    .mean()
    .rename("dominant_state_retention_rate")
)

retention_summary = pd.DataFrame([{
    "monthly_feature_rows": len(monthly_retention_fm),
    "monthly_households": monthly_retention_fm.reset_index()["household_ID"].nunique(),
    "households_evaluated_ge_3_months": len(valid_hhs),
    "mean_dominant_state_retention_rate": float(household_retention.mean()),
    "median_dominant_state_retention_rate": float(household_retention.median()),
    "highly_stable_share_ge_0_80": float((household_retention >= 0.80).mean()),
    "moderately_stable_share_0_60_to_0_79": float(
        ((household_retention >= 0.60) & (household_retention < 0.80)).mean()
    ),
    "unstable_share_lt_0_60": float((household_retention < 0.60).mean())
}])

print("\n--- PHASE 3: DOMINANT-STATE TEMPORAL STABILITY ---")
display(retention_summary)

highly_stable = int((household_retention >= 0.80).sum())
mod_stable = int(((household_retention >= 0.60) & (household_retention < 0.80)).sum())
unstable = int((household_retention < 0.60).sum())

print(f"\nAverage dominant-state retention rate: {household_retention.mean() * 100:.1f}%")
print(f"Highly stable     (≥80%):   {highly_stable:,} HHs ({highly_stable / len(household_retention) * 100:.1f}%)")
print(f"Moderately stable (60–79%): {mod_stable:,} HHs ({mod_stable / len(household_retention) * 100:.1f}%)")
print(f"Unstable          (<60%):   {unstable:,} HHs ({unstable / len(household_retention) * 100:.1f}%)")

# ------------------------------------------------------------
# 6. Save outputs
# ------------------------------------------------------------

monthly_retention_fm.reset_index().to_csv(
    out_dir / "corrected_monthly_retention_feature_matrix_with_locked_clusters.csv",
    index=False
)

retention_state_df.to_csv(
    out_dir / "corrected_monthly_dominant_state_retention_households.csv",
    index=False
)

household_retention.reset_index().to_csv(
    out_dir / "corrected_monthly_household_retention_rates.csv",
    index=False
)

retention_summary.to_csv(
    out_dir / "corrected_monthly_dominant_state_retention_summary.csv",
    index=False
)

print("\nSaved corrected monthly dominant-state retention outputs to:")
print(out_dir.resolve())

print("\nMemory-safe corrected monthly temporal stability check completed.")


In [ ]:
# ============================================================
# TIMESTAMP-BOUNDARY DESCRIPTIVE DIAGNOSTIC
# This cell does not claim to identify interval-start versus interval-end
# semantics from aggregate consumption means alone.
# ============================================================

boundary_window = feature_df[
    feature_df["meter_dt"].dt.strftime("%H:%M").isin(
        ["18:00", "18:15", "18:30", "18:45", "22:15", "22:30", "22:45"]
    )
].copy()

boundary_means = (
    boundary_window
    .assign(clock_time=boundary_window["meter_dt"].dt.strftime("%H:%M"))
    .groupby("clock_time")["true_15min_kwh"]
    .agg(["count", "mean", "median"])
)

print("Descriptive consumption around tariff boundaries:")
display(boundary_means)

print(
    "\nInterpretation: aggregate means around 18:30 or 22:30 cannot by "
    "themselves establish whether meter timestamps denote interval starts or "
    "interval ends. The main 18:30–22:30 peak definition is retained as the "
    "documented modelling convention, and the next cell tests ±15-minute "
    "boundary sensitivity."
)


In [ ]:
# ============================================================
# ±15-MINUTE TARIFF-BOUNDARY SENSITIVITY
# Recomputes the two tariff-share features under peak starts of
# 18:15, 18:30 and 18:45, then refits the full preprocessing/KMeans chain.
# Uses external grouping vectors to avoid copying the full feature_df.
# ============================================================

from pathlib import Path
from sklearn.metrics import adjusted_rand_score, silhouette_score

baseline_boundary_labels = clean_fm["final_cluster"].astype(int)
boundary_sensitivity_rows = []
boundary_assignments = {}

minute_of_day = (
    feature_df["meter_dt"].dt.hour * 60
    + feature_df["meter_dt"].dt.minute
).astype("uint16")

for peak_start in [18 * 60 + 15, 18 * 60 + 30, 18 * 60 + 45]:
    peak_end = 22 * 60 + 30

    alt_block_series = pd.Series(
        pd.Categorical(
            np.select(
                [
                    (minute_of_day >= 5 * 60 + 30)
                    & (minute_of_day < peak_start),
                    (minute_of_day >= peak_start)
                    & (minute_of_day < peak_end)
                ],
                ["Day", "Peak"],
                default="Off-Peak"
            ),
            categories=["Day", "Peak", "Off-Peak"],
            ordered=False
        ),
        index=feature_df.index,
        name="alt_block"
    )

    daily_alt = (
        feature_df["true_15min_kwh"]
        .groupby(
            [
                feature_df["household_ID"],
                feature_df["date_only"],
                alt_block_series
            ],
            observed=True
        )
        .sum()
        .unstack(fill_value=0)
        .reindex(columns=["Day", "Peak", "Off-Peak"], fill_value=0)
    )
    avg_alt = daily_alt.groupby("household_ID", observed=True).mean()
    total_alt = avg_alt.sum(axis=1)

    alt_features = feature_matrix[CORE_FEATURES].copy()
    alt_features["evening_peak_ratio"] = (
        avg_alt["Peak"] / total_alt.clip(lower=0.01)
    )
    alt_features["night_baseload_ratio"] = (
        avg_alt["Off-Peak"] / total_alt.clip(lower=0.01)
    )

    alt_features = alt_features[
        alt_features["log_daily_kwh"] > 0.05
    ]
    alt_features = (
        alt_features
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=CORE_FEATURES)
    )

    clipped = alt_features[CORE_FEATURES].copy()
    for col in CORE_FEATURES:
        lower = clipped[col].quantile(0.01)
        upper = clipped[col].quantile(0.99)
        clipped[col] = clipped[col].clip(lower=lower, upper=upper)

    alt_scaler = StandardScaler()
    X_alt = alt_scaler.fit_transform(clipped)
    alt_model = KMeans(
        n_clusters=4,
        random_state=42,
        n_init=20
    )
    labels_alt = pd.Series(
        alt_model.fit_predict(X_alt),
        index=clipped.index,
        name=f"peak_start_{peak_start}"
    )

    common = baseline_boundary_labels.index.intersection(labels_alt.index)
    if len(common) == 0:
        raise RuntimeError(
            f"No common households for peak-start sensitivity {peak_start}."
        )

    ari = adjusted_rand_score(
        baseline_boundary_labels.loc[common],
        labels_alt.loc[common]
    )

    boundary_assignments[peak_start] = labels_alt
    boundary_sensitivity_rows.append({
        "peak_start": (
            f"{peak_start // 60:02d}:{peak_start % 60:02d}"
        ),
        "population": len(labels_alt),
        "common_households": len(common),
        "ari_vs_locked_18_30_solution": ari,
        "silhouette": silhouette_score(
            X_alt,
            labels_alt.to_numpy()
        )
    })

    del (
        alt_block_series,
        daily_alt,
        avg_alt,
        total_alt,
        alt_features,
        clipped,
        X_alt
    )

boundary_sensitivity_summary = pd.DataFrame(
    boundary_sensitivity_rows
)
print("Tariff-boundary sensitivity summary:")
display(boundary_sensitivity_summary.round(6))

Path("tables_outputs").mkdir(exist_ok=True)
boundary_sensitivity_summary.to_csv(
    "tables_outputs/tariff_boundary_plus_minus_15min_sensitivity.csv",
    index=False
)

print(
    "The 18:30 boundary remains the primary documented convention. "
    "The alternative starts quantify sensitivity; they do not infer "
    "timestamp semantics from aggregate load shapes."
)


In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_clean)

plt.figure(figsize=(10, 7))
for c in CLUSTER_ORDER:
    mask = clean_fm['final_cluster'] == c
    plt.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=colors[c], label=c_names[c], alpha=0.5, s=20)
plt.title("t-SNE Projection of K=4 Clusters", fontsize=14, fontweight='bold')
plt.xlabel("t-SNE Component 1")
plt.ylabel("t-SNE Component 2")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Check who is between 0.5 and 1.0 kWh — are they noise or real?
borderline = solar_export_delta[
    (solar_export_delta > 0.5) & (solar_export_delta <= 1.0)]
print(f"Households with delta 0.5–1.0 kWh: {len(borderline)}")
print(borderline.describe())
print(borderline.values)

In [ ]:
print("=== VALID FILTERED 15-MIN kWh DISTRIBUTION ===")
print(clean_df['true_15min_kwh'].describe(
    percentiles=[.01,.05,.25,.5,.75,.95,.99,.999]))

# Check what percentage falls in different bands
for threshold in [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]:
    pct = (clean_df['true_15min_kwh'] > threshold).mean() * 100
    print(f"  > {threshold} kWh: {pct:.4f}%")

In [ ]:
print("=== RAW kWh DIFF BEFORE PHYSICS CAP ===")
print(clean_df['raw_kwh_diff'].describe(
    percentiles=[.25, .5, .75, .95, .99, .999]
))

valid_interval = (
    (clean_df['time_delta_mins'] >= 14.5) &
    (clean_df['time_delta_mins'] <= 15.5)
)
wrong_interval = clean_df['time_delta_mins'].notna() & (~valid_interval)

print(f"\nTotal raw diff readings:         "
      f"{clean_df['raw_kwh_diff'].notna().sum():,}")
print(f"Readings above 6.0 kWh (capped): "
      f"{(clean_df['raw_kwh_diff'] > 6.0).sum():,}")
print(f"Readings below 0 (negative):     "
      f"{(clean_df['raw_kwh_diff'] < 0).sum():,}")
print(f"Readings wrong time interval:    "
      f"{wrong_interval.sum():,}")

print(f"\n% excluded by physics cap only:  "
      f"{(clean_df['raw_kwh_diff'] > 6.0).mean()*100:.4f}%")
print(f"% excluded by negative values:   "
      f"{(clean_df['raw_kwh_diff'] < 0).mean()*100:.4f}%")

print(f"\nMax raw diff value:              "
      f"{clean_df['raw_kwh_diff'].max():.4f} kWh")
print(f"Values between 6.0 and 10.5 kWh:"
      f"{((clean_df['raw_kwh_diff'] > 6.0) & (clean_df['raw_kwh_diff'] <= 10.5)).sum():,}")
print(f"Values above 10.5 kWh:           "
      f"{(clean_df['raw_kwh_diff'] > 10.5).sum():,}")

In [ ]:
print("=== AUDITING 6.0–10.5 kWh READINGS BEFORE PHYSICS CAP ===")

borderline_mask = (
    clean_df["raw_kwh_diff"].gt(6.0)
    & clean_df["raw_kwh_diff"].le(10.5)
)
borderline_cap = clean_df.loc[
    borderline_mask,
    [
        "household_ID",
        "meter_dt",
        "time_delta_mins",
        "raw_kwh_diff"
    ]
].copy()

print(
    f"\nTotal readings in 6.0–10.5 kWh range: "
    f"{len(borderline_cap):,}"
)

unique_hh = borderline_cap["household_ID"].nunique()
print(f"Unique households producing them:       {unique_hh}")
if unique_hh:
    print(
        "Average readings per household:         "
        f"{len(borderline_cap) / unique_hh:.1f}"
    )

print("\nTime distribution of 6.0–10.5 kWh readings:")
print(
    borderline_cap["meter_dt"]
    .dt.hour
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nTime delta distribution for these readings:")
print(borderline_cap["time_delta_mins"].describe())

valid_interval_mask = borderline_cap[
    "time_delta_mins"
].between(14.5, 15.5, inclusive="both")
print(
    "Readings with valid time interval (14.5–15.5 min): "
    f"{valid_interval_mask.sum():,}"
)
print(
    "Readings with invalid time interval:               "
    f"{(~valid_interval_mask).sum():,}"
)

# clean_df was sorted by household and timestamp in the master pipeline.
# Retrieve neighbours only for the flagged positions instead of copying the
# full telemetry dataframe and adding two full-length shift columns.
raw_values = clean_df["raw_kwh_diff"].to_numpy(copy=False)
if isinstance(clean_df["household_ID"].dtype, pd.CategoricalDtype):
    household_codes = (
        clean_df["household_ID"]
        .cat.codes
        .to_numpy(copy=False)
    )
else:
    household_codes = pd.factorize(
        clean_df["household_ID"],
        sort=False
    )[0]

positions = np.flatnonzero(
    borderline_mask.to_numpy(dtype=bool)
)
previous_values = np.full(len(positions), np.nan, dtype=float)
next_values = np.full(len(positions), np.nan, dtype=float)

has_previous = (
    (positions > 0)
    & (
        household_codes[positions - 1]
        == household_codes[positions]
    )
)
has_next = (
    (positions + 1 < len(clean_df))
    & (
        household_codes[
            np.minimum(positions + 1, len(clean_df) - 1)
        ]
        == household_codes[positions]
    )
)

previous_values[has_previous] = raw_values[
    positions[has_previous] - 1
]
next_values[has_next] = raw_values[
    positions[has_next] + 1
]

borderline_context = borderline_cap.copy()
borderline_context["prev_raw_kwh_diff"] = previous_values
borderline_context["next_raw_kwh_diff"] = next_values

print("\nNeighbouring-readings check:")
print(
    "Mean reading BEFORE flagged interval: "
    f"{borderline_context['prev_raw_kwh_diff'].mean():.4f} kWh"
)
print(
    "Mean reading AFTER flagged interval:  "
    f"{borderline_context['next_raw_kwh_diff'].mean():.4f} kWh"
)
print(
    "Median reading BEFORE flagged interval: "
    f"{borderline_context['prev_raw_kwh_diff'].median():.4f} kWh"
)
print(
    "Median reading AFTER flagged interval:  "
    f"{borderline_context['next_raw_kwh_diff'].median():.4f} kWh"
)

print("\nHouseholds producing 6.0–10.5 kWh readings, sample of 10:")
borderline_household_counts = (
    borderline_cap["household_ID"]
    .value_counts()
)
borderline_household_counts = borderline_household_counts[
    borderline_household_counts > 0
]
print(borderline_household_counts.head(10).to_string())

del (
    raw_values,
    household_codes,
    positions,
    previous_values,
    next_values,
    has_previous,
    has_next
)


### Private diagnostic omitted — Two-household membership investigation

This source cell concerned identified households and was intentionally withheld from the public review draft. It remains unchanged in the privately retained notebook. The original case-specific analysis cannot be reproduced from this public copy alone.

### Private diagnostic omitted — Single-household interval-level diagnostic

This source cell concerned identified households and was intentionally withheld from the public review draft. It remains unchanged in the privately retained notebook. The original case-specific analysis cannot be reproduced from this public copy alone.

### Private diagnostic omitted — Case-conditioned anomaly distribution

This source cell concerned identified households and was intentionally withheld from the public review draft. It remains unchanged in the privately retained notebook. The original case-specific analysis cannot be reproduced from this public copy alone.

### Private diagnostic omitted — Case-conditioned golden-window anomaly follow-up

This source cell concerned identified households and was intentionally withheld from the public review draft. It remains unchanged in the privately retained notebook. The original case-specific analysis cannot be reproduced from this public copy alone.

In [ ]:
# ============================================================
# PHYSICS-CAP SENSITIVITY: 6.0 vs 8.0 vs 10.5 kWh/15-min
# Rebuilds affected-household features under alternative caps, then
# recalculates ghost removal, clipping, scaling and KMeans.
# ============================================================

from pathlib import Path
from scipy.stats import entropy
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from scipy.optimize import linear_sum_assignment


def _build_valid_day_features_for_households(raw_subset, cap_value):
    df = raw_subset.copy()
    valid_time = df["time_delta_mins"].between(14.5, 15.5, inclusive="both")
    valid_energy = df["raw_kwh_diff"].ge(0) & df["raw_kwh_diff"].le(cap_value)
    df["true_alt_kwh"] = np.where(valid_time & valid_energy, df["raw_kwh_diff"], np.nan)
    df["date_only"] = df["meter_dt"].dt.normalize()

    day_counts_alt = (
        df.groupby(["household_ID", "date_only"], observed=True)["true_alt_kwh"]
        .count()
        .rename("valid_intervals")
        .reset_index()
    )
    valid_days_alt = day_counts_alt[day_counts_alt["valid_intervals"] >= 90]
    eligible_alt_ids = valid_days_alt.groupby("household_ID", observed=True).size()
    eligible_alt_ids = eligible_alt_ids[eligible_alt_ids >= 30].index

    keys = valid_days_alt[
        valid_days_alt["household_ID"].isin(eligible_alt_ids)
    ][["household_ID", "date_only"]].drop_duplicates()

    df = df.merge(keys, on=["household_ID", "date_only"], how="inner")
    if df.empty:
        return pd.DataFrame(columns=CORE_FEATURES)

    df["is_non_working_day"] = (
        (df["meter_dt"].dt.dayofweek >= 5)
        | df["date_only"].isin(sl_holidays_2024)
    )
    minute = df["meter_dt"].dt.hour * 60 + df["meter_dt"].dt.minute
    df["ceb_block_alt"] = np.select(
        [
            (minute >= 5 * 60 + 30) & (minute < 18 * 60 + 30),
            (minute >= 18 * 60 + 30) & (minute < 22 * 60 + 30)
        ],
        ["Day", "Peak"],
        default="Off-Peak"
    )
    df["interval_id"] = df["meter_dt"].dt.hour * 4 + df["meter_dt"].dt.minute // 15

    daily_block = (
        df.groupby(["household_ID", "date_only", "ceb_block_alt"], observed=True)["true_alt_kwh"]
        .sum()
        .unstack(fill_value=0)
        .reindex(columns=["Day", "Peak", "Off-Peak"], fill_value=0)
    )
    avg_block = daily_block.groupby("household_ID", observed=True).mean()
    total_avg = avg_block.sum(axis=1)

    profiles = (
        df.groupby(["household_ID", "interval_id"], observed=True)["true_alt_kwh"]
        .mean()
        .unstack()
        .reindex(columns=range(96))
    )
    complete_profiles = profiles.dropna(axis=0, how="any")

    stats = df.groupby("household_ID", observed=True)["true_alt_kwh"].agg(["mean", "max"])

    daily_totals = (
        df.groupby(["household_ID", "date_only", "is_non_working_day"], observed=True)["true_alt_kwh"]
        .sum()
        .reset_index()
    )
    day_type_avg = (
        daily_totals
        .groupby(["household_ID", "is_non_working_day"], observed=True)["true_alt_kwh"]
        .mean()
        .unstack()
    )

    result = pd.DataFrame(index=avg_block.index)
    result["evening_peak_ratio"] = avg_block["Peak"] / total_avg.clip(lower=0.01)
    result["night_baseload_ratio"] = avg_block["Off-Peak"] / total_avg.clip(lower=0.01)
    result["load_uniformity"] = complete_profiles.apply(
        lambda row: entropy(row.to_numpy(dtype=float) + 1e-9), axis=1
    )
    result["load_factor"] = stats["mean"] / stats["max"].clip(lower=0.01)

    if True in day_type_avg.columns and False in day_type_avg.columns:
        result["leisure_usage_ratio"] = np.log1p(
            day_type_avg[True] / day_type_avg[False].clip(lower=0.01)
        )
    else:
        result["leisure_usage_ratio"] = np.nan

    result["log_daily_kwh"] = np.log1p(total_avg)
    return result[CORE_FEATURES].replace([np.inf, -np.inf], np.nan).dropna()


def _align_and_count_changes(reference, candidate):
    common = reference.index.intersection(candidate.index)
    ct = pd.crosstab(reference.loc[common], candidate.loc[common])
    row_ind, col_ind = linear_sum_assignment(-ct.to_numpy())
    mapping = {
        ct.columns[col]: ct.index[row]
        for row, col in zip(row_ind, col_ind)
    }
    aligned = candidate.loc[common].map(mapping)
    changed = reference.loc[common] != aligned
    return common, aligned, changed


baseline_labels_cap = clean_fm["final_cluster"].astype(int)
cap_sensitivity_rows = []
cap_sensitivity_assignments = {}

raw_window_mask = (
    (clean_df["meter_dt"] >= START_DATE)
    & (clean_df["meter_dt"] < END_DATE_EXCLUSIVE)
)

for cap_value in [8.0, 10.5]:
    affected_ids = clean_df.loc[
        raw_window_mask
        & clean_df["raw_kwh_diff"].gt(PHYSICAL_LIMIT)
        & clean_df["raw_kwh_diff"].le(cap_value)
        & clean_df["time_delta_mins"].between(14.5, 15.5, inclusive="both"),
        "household_ID"
    ].dropna().unique()

    affected_subset = clean_df.loc[
        raw_window_mask & clean_df["household_ID"].isin(affected_ids),
        ["household_ID", "meter_dt", "time_delta_mins", "raw_kwh_diff"]
    ].copy()

    alternative_affected_features = _build_valid_day_features_for_households(
        affected_subset, cap_value
    )

    alternative_feature_matrix = feature_matrix.drop(
        index=affected_ids, errors="ignore"
    ).copy()
    alternative_feature_matrix = pd.concat(
        [alternative_feature_matrix, alternative_affected_features], axis=0
    )
    alternative_feature_matrix = alternative_feature_matrix[~alternative_feature_matrix.index.duplicated(keep="last")]

    active_alt = alternative_feature_matrix[
        alternative_feature_matrix["log_daily_kwh"] > 0.05
    ].copy()
    active_alt = active_alt.replace([np.inf, -np.inf], np.nan).dropna(subset=CORE_FEATURES)

    clipped_alt = active_alt[CORE_FEATURES].copy()
    for col in CORE_FEATURES:
        lower = clipped_alt[col].quantile(0.01)
        upper = clipped_alt[col].quantile(0.99)
        clipped_alt[col] = clipped_alt[col].clip(lower=lower, upper=upper)

    alt_scaler = StandardScaler()
    X_alt_cap = alt_scaler.fit_transform(clipped_alt)
    alt_model = KMeans(n_clusters=4, random_state=42, n_init=20)
    alt_labels = pd.Series(
        alt_model.fit_predict(X_alt_cap),
        index=clipped_alt.index,
        name=f"cap_{cap_value:g}"
    )

    common, aligned, changed = _align_and_count_changes(
        baseline_labels_cap, alt_labels
    )

    cap_sensitivity_assignments[cap_value] = alt_labels
    cap_sensitivity_rows.append({
        "cap_kwh_per_15min": cap_value,
        "affected_households_rebuilt": len(affected_ids),
        "alternative_population": len(alt_labels),
        "population_change_vs_6kwh": len(alt_labels) - len(baseline_labels_cap),
        "common_households": len(common),
        "ari_vs_6kwh_solution": adjusted_rand_score(
            baseline_labels_cap.loc[common], alt_labels.loc[common]
        ),
        "changed_common_households_after_alignment": int(changed.sum()),
        "changed_common_share": float(changed.mean()),
        "alternative_silhouette": silhouette_score(X_alt_cap, alt_labels.to_numpy())
    })

cap_sensitivity_summary = pd.DataFrame(cap_sensitivity_rows)
print("Physics-cap sensitivity summary:")
display(cap_sensitivity_summary.round(6))

Path("tables_outputs").mkdir(exist_ok=True)
cap_sensitivity_summary.to_csv(
    "tables_outputs/physics_cap_6_vs_8_vs_10_5_sensitivity.csv",
    index=False
)

print(
    "The 6.0 kWh cap remains the conservative primary rule. Alternative caps "
    "are reported only as population and partition sensitivity checks."
)

In [ ]:
# ============================================================
# EXTENDED FEATURE VALIDATION — VALID-DAY-ONLY SUPPLEMENTARY CHECK
# This does not replace the locked six-feature primary model.
# ============================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

required = ['feature_df', 'clean_fm', 'CORE_FEATURES']
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Missing required objects: {missing}")

print('=== STEP 0: BUILDING EXTENDED FEATURES FROM VALID DAYS ONLY ===')
active_ids = pd.Index(clean_fm.index)
active_row_mask = feature_df['household_ID'].isin(active_ids)

ext_fm = pd.DataFrame(index=active_ids)

# Daily totals are compact and support variability, holiday, monsoon, and scale features.
daily_ext = (
    feature_df.loc[active_row_mask]
    .groupby(['household_ID', 'date_only'], observed=True)
    .agg(
        daily_kwh=('true_15min_kwh', 'sum'),
        dayofweek=('meter_dt', lambda x: int(x.iloc[0].dayofweek)),
        month=('meter_dt', lambda x: int(x.iloc[0].month))
    )
    .reset_index()
)

# Block 1: tariff ratios using exactly the retained valid days.
daily_block_ext = (
    feature_df.loc[active_row_mask]
    .groupby(['household_ID', 'date_only', 'ceb_block'], observed=True)['true_15min_kwh']
    .sum()
    .unstack(fill_value=0)
)
for block in ['Day', 'Peak', 'Off-Peak']:
    if block not in daily_block_ext.columns:
        daily_block_ext[block] = 0.0
avg_block_ext = daily_block_ext[['Day', 'Peak', 'Off-Peak']].groupby('household_ID', observed=True).mean()
total_avg_ext = avg_block_ext.sum(axis=1)
ext_fm['tariff_peak_ratio'] = avg_block_ext['Peak'] / total_avg_ext.clip(lower=0.01)
ext_fm['tariff_offpeak_ratio'] = avg_block_ext['Off-Peak'] / total_avg_ext.clip(lower=0.01)

# Block 2: daily variability.
daily_group = daily_ext.groupby('household_ID', observed=True)['daily_kwh']
ext_fm['cv_daily'] = daily_group.std() / daily_group.mean().clip(lower=0.01)

# Block 3: interval-profile features. Missing interval positions remain NaN.
ext_interval_profiles = (
    feature_df.loc[active_row_mask]
    .groupby(['household_ID', 'interval_id'], observed=True)['true_15min_kwh']
    .mean()
    .unstack()
    .reindex(index=active_ids, columns=range(96))
)
complete_ext_profiles = ext_interval_profiles.notna().all(axis=1)
if not complete_ext_profiles.all():
    print('Incomplete extended interval profiles:', int((~complete_ext_profiles).sum()))

active_values = feature_df.loc[active_row_mask, 'true_15min_kwh'].dropna()
ACTIVE_THRESHOLD = float(active_values.median())
P95 = float(active_values.quantile(0.95))
ext_fm['mean_active_hours'] = (ext_interval_profiles > ACTIVE_THRESHOLD).sum(axis=1) * 0.25
ext_fm['peak_to_valley_ratio'] = (
    ext_interval_profiles.max(axis=1)
    / ext_interval_profiles.min(axis=1).clip(lower=0.001)
)
ext_fm['high_intensity_ratio'] = (ext_interval_profiles > P95).mean(axis=1)
peak_interval = ext_interval_profiles.idxmax(axis=1)
peak_hour_decimal = peak_interval / 4.0
ext_fm['peak_hour_sin'] = np.sin(2 * np.pi * peak_hour_decimal / 24)
ext_fm['peak_hour_cos'] = np.cos(2 * np.pi * peak_hour_decimal / 24)

# Block 4: robust load factor from retained valid-day rows.
active_interval_group = feature_df.loc[active_row_mask].groupby('household_ID', observed=True)['true_15min_kwh']
p99_per_hh = active_interval_group.quantile(0.99)
mean_per_hh = active_interval_group.mean()
ext_fm['load_factor_robust'] = mean_per_hh / p99_per_hh.clip(lower=0.001)

# Block 5: holiday/weekend ratios from daily totals. Missing categories remain NaN
# and are transparently median-imputed only for this supplementary experiment.
poya_dates = set(pd.to_datetime([
    '2024-05-23', '2024-06-21', '2024-07-20',
    '2024-08-19', '2024-09-17', '2024-10-17', '2024-11-15'
]).date)
general_holiday_dates = set(pd.to_datetime([
    '2024-05-01', '2024-05-24', '2024-09-16', '2024-10-31'
]).date)
all_holidays = poya_dates | general_holiday_dates

daily_ext['is_poya'] = daily_ext['date_only'].isin(poya_dates)
daily_ext['is_general_holiday'] = daily_ext['date_only'].isin(general_holiday_dates)
daily_ext['is_weekend'] = (daily_ext['dayofweek'] >= 5) & (~daily_ext['date_only'].isin(all_holidays))
daily_ext['is_workday'] = (daily_ext['dayofweek'] < 5) & (~daily_ext['date_only'].isin(all_holidays))

def daily_ratio(flag_col):
    flag_avg = daily_ext.loc[daily_ext[flag_col]].groupby('household_ID', observed=True)['daily_kwh'].mean()
    work_avg = daily_ext.loc[daily_ext['is_workday']].groupby('household_ID', observed=True)['daily_kwh'].mean()
    ratio_frame = pd.DataFrame({'flag': flag_avg, 'work': work_avg}).reindex(active_ids)
    return np.log1p(ratio_frame['flag'] / ratio_frame['work'].clip(lower=0.01))

ext_fm['poya_ratio'] = daily_ratio('is_poya')
ext_fm['general_holiday_ratio'] = daily_ratio('is_general_holiday')
ext_fm['weekend_ratio'] = daily_ratio('is_weekend')

# Block 6: monsoon ratio using mean valid-day consumption.
daily_ext['monsoon'] = np.where(daily_ext['month'].isin([5, 6, 7, 8, 9]), 'SW', 'NE')
monsoon_avg = daily_ext.groupby(['household_ID', 'monsoon'], observed=True)['daily_kwh'].mean().unstack()
monsoon_avg = monsoon_avg.reindex(active_ids).reindex(columns=['SW', 'NE'])
ext_fm['sw_ne_monsoon_ratio'] = np.log1p(
    monsoon_avg['SW'] / monsoon_avg['NE'].clip(lower=0.001)
)

# Block 7: scale features.
ext_fm['log_daily_kwh'] = np.log1p(total_avg_ext.reindex(active_ids))
ext_fm['log_median_daily_kwh'] = np.log1p(daily_group.median().reindex(active_ids))

ext_fm.replace([np.inf, -np.inf], np.nan, inplace=True)
ext_missing_before = ext_fm.isna().sum().sort_values(ascending=False)
print('\nMissing values before supplementary median imputation:')
display(ext_missing_before[ext_missing_before > 0].to_frame('missing_households'))

all_missing_cols = ext_missing_before[ext_missing_before == len(ext_fm)].index.tolist()
if all_missing_cols:
    print(f"Dropping supplementary features with no observable values: {all_missing_cols}")
    ext_fm = ext_fm.drop(columns=all_missing_cols)

ext_fm = ext_fm.fillna(ext_fm.median(numeric_only=True))
for col in ext_fm.columns:
    lower = ext_fm[col].quantile(0.01)
    upper = ext_fm[col].quantile(0.99)
    ext_fm[col] = ext_fm[col].clip(lower=lower, upper=upper)

assert ext_fm.index.equals(active_ids)
assert ext_fm.notna().all().all()
print(f"\nExtended matrix: {len(ext_fm)} households, {len(ext_fm.columns)} features")

print('\n=== CORRELATION WITH ORIGINAL SIX FEATURES ===')
new_features = [c for c in ext_fm.columns if c not in CORE_FEATURES]
print(f"\n{'New Feature':<28} {'Best corr with original':<30} {'r':>8}")
print('-' * 70)
for new_feature in new_features:
    correlations = {
        original: abs(ext_fm[new_feature].corr(clean_fm[original]))
        for original in CORE_FEATURES
    }
    best_original = max(correlations, key=correlations.get)
    print(f"  {new_feature:<26} {best_original:<30} {correlations[best_original]:>8.3f}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEPS 1, 2, 3 — VALIDATION TESTS
# ═══════════════════════════════════════════════════════════════

# Step 1: Verify base labels
print("=== STEP 1: VERIFY BASE_LABELS ===")
orig_pipeline_labels = clean_fm['final_cluster'].values
base_labels_fresh = KMeans(
    n_clusters=4, random_state=42, n_init=20).fit_predict(X_clean)
ari_check = adjusted_rand_score(orig_pipeline_labels, base_labels_fresh)
print(f"ARI between X_clean KMeans and pipeline labels: {ari_check:.4f}")
if ari_check > 0.99:
    print("PASS — base reference valid")
else:
    print("WARNING — mismatch, results below may not be reliable")
base_labels = orig_pipeline_labels
base_sil    = silhouette_score(X_clean, base_labels)
print(f"Baseline silhouette: {base_sil:.4f}")

# Step 2: Overall comparison
print("\n=== STEP 2: ALTERNATIVE FEATURE SET vs LOCKED SIX-FEATURE SET ===")
X_ext  = StandardScaler().fit_transform(ext_fm)
l_ext  = KMeans(n_clusters=4, random_state=42, n_init=20).fit_predict(X_ext)
s_ext  = silhouette_score(X_ext, l_ext)
ari_overall = adjusted_rand_score(base_labels, l_ext)
print(f"Original 6-feature silhouette:   {base_sil:.3f}")
print(f"Alternative feature-set silhouette: {s_ext:.3f}")
print(f"ARI agreement:                   {ari_overall:.3f}")
if s_ext > base_sil + 0.02:
    print("Alternative feature set produced meaningfully higher separation.")
elif abs(s_ext - base_sil) <= 0.02:
    print("No meaningful separation gain from the alternative feature set.")
else:
    print("The alternative feature set produced weaker separation and was not retained for the locked clustering solution.")

# Step 3: Mean active hours cross-tabulation
print("\n=== STEP 3: MEAN_ACTIVE_HOURS CROSS-TABULATION ===")
mah_df = clean_fm[CORE_FEATURES].copy()
mah_df['mean_active_hours'] = ext_fm['mean_active_hours']
X_mah  = StandardScaler().fit_transform(
    mah_df[CORE_FEATURES + ['mean_active_hours']])
l_mah  = KMeans(n_clusters=4, random_state=42, n_init=20).fit_predict(X_mah)
s_mah  = silhouette_score(X_mah, l_mah)
a_mah  = adjusted_rand_score(base_labels, l_mah)
print(f"Silhouette: {s_mah:.4f}  delta: {s_mah-base_sil:+.4f}")
print(f"ARI vs original: {a_mah:.4f}")
print("\nCross-tabulation (rows=original, cols=with MAH):")
print(pd.crosstab(
    pd.Series(base_labels, name='original'),
    pd.Series(l_mah, name='with_MAH')))
print("\nOne dominant per row = rotation | Split = genuinely different")

# Step 4: Individual contribution test
print("\n=== STEP 4: INDIVIDUAL FEATURE CONTRIBUTION ===")
print(f"Baseline: {base_sil:.4f}\n")
print(f"  {'Feature':<26} {'Sil':>8} {'Delta':>8} {'ARI':>10}  Note")
print("  " + "-" * 70)

test_list = [
    'mean_active_hours','cv_daily','sw_ne_monsoon_ratio',
    'poya_ratio','general_holiday_ratio','peak_hour_sin',
    'peak_hour_cos','peak_to_valley_ratio','load_factor_robust',
    'high_intensity_ratio','tariff_peak_ratio','tariff_offpeak_ratio',
    'log_median_daily_kwh','weekend_ratio',
]

for nf in test_list:
    if nf not in ext_fm.columns:
        print(f"  {nf:<26} MISSING")
        continue
    td = clean_fm[CORE_FEATURES].copy()
    td[nf] = ext_fm[nf]
    Xt = StandardScaler().fit_transform(td[CORE_FEATURES + [nf]])
    lt = KMeans(n_clusters=4, random_state=42, n_init=20).fit_predict(Xt)
    st = silhouette_score(Xt, lt)
    at = adjusted_rand_score(base_labels, lt)
    delta = st - base_sil
    if   delta > 0.01 and at > 0.85: note = "✅ IMPROVES — stable"
    elif delta > 0.01 and at <= 0.85: note = "⚠️  IMPROVES but changes clusters"
    elif abs(delta) <= 0.01:          note = "⚪ NEUTRAL"
    else:                             note = "❌ HARMS"
    print(f"  {nf:<26} {st:>8.4f} {delta:>+8.4f} {at:>10.4f}  {note}")

print(f"\n  Baseline (original 6): {base_sil:.4f}")
print("\n✅ = delta>0.01 AND ARI>0.85 | ⚠️ = delta>0.01 AND ARI<0.85")
print("⚪ = |delta|<=0.01           | ❌ = delta<-0.01")

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import kruskal, spearmanr

print("=" * 70)
print("  COVERAGE HETEROGENEITY AUDIT")
print("  Tests whether valid-day count biases cluster assignment")
print("=" * 70)

# Recompute valid days per household from golden_df
# golden_df is still in memory from the main pipeline
day_counts = (
    golden_df.groupby(['household_ID', 'date_only'], observed=True)['true_15min_kwh']
    .count()
    .reset_index(name='readings')
)
valid_days_per_hh = (
    day_counts[day_counts['readings'] >= 90]
    .groupby('household_ID', observed=True)
    .size()
    .reset_index(name='valid_days')
)

# Merge with cluster labels
coverage = pd.merge(
    clean_fm[['final_cluster']].reset_index(),
    valid_days_per_hh,
    on='household_ID', how='left'
)

c_names = dict(CLUSTER_NAME_MAP)

print(f"\n=== VALID DAYS DISTRIBUTION BY CLUSTER ===")
print(f"  {'Cluster':<24} {'N':>5} {'Min':>5} {'P25':>5} "
      f"{'Median':>7} {'P75':>5} {'Max':>5} {'Mean':>7}")
print("  " + "-" * 70)
for c in range(4):
    sub = coverage[coverage['final_cluster'] == c]['valid_days'].dropna()
    print(f"  {c_names[c]:<24} {len(sub):>5} "
          f"{sub.min():>5.0f} {sub.quantile(0.25):>5.0f} "
          f"{sub.median():>7.0f} {sub.quantile(0.75):>5.0f} "
          f"{sub.max():>5.0f} {sub.mean():>7.1f}")

# Kruskal-Wallis: is valid_days distribution different across clusters?
groups = [coverage[coverage['final_cluster']==c]['valid_days'].dropna().values
          for c in range(4)]
H, p = kruskal(*groups)
print(f"\nKruskal-Wallis on valid_days across clusters:")
print(f"  H={H:.3f}  p={p:.4f}  "
      f"{'*** SIGNIFICANT — coverage differs by cluster' if p < 0.05 else '✅ ns — coverage uniform across clusters'}")

# If significant, check Cramér's V (effect size)
from scipy.stats import chi2_contingency
coverage['coverage_quartile'] = pd.qcut(
    coverage['valid_days'], q=4,
    labels=['Q1: lowest coverage', 'Q2', 'Q3', 'Q4: highest coverage']
)
ct = pd.crosstab(coverage['final_cluster'], coverage['coverage_quartile'])
chi2, p_chi, dof, _ = chi2_contingency(ct)
n = ct.sum().sum()
v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
print(f"\nCramér's V (coverage quartile vs cluster): V={v:.3f}")
print(f"  < 0.10 = negligible association")
print(f"  0.10–0.30 = small association")
print(f"  > 0.30 = meaningful association")

# Coverage quartile distribution per cluster
print(f"\n=== COVERAGE QUARTILE DISTRIBUTION BY CLUSTER (%) ===")
ct_pct = pd.crosstab(
    coverage['final_cluster'],
    coverage['coverage_quartile'],
    normalize='index') * 100
ct_pct.index = [c_names[i] for i in ct_pct.index]
print(ct_pct.round(1).to_string())

# Specific check: load_factor vs valid_days correlation
print(f"\n=== LOAD_FACTOR vs VALID_DAYS CORRELATION ===")
lf_cov = pd.merge(
    clean_fm[['load_factor', 'final_cluster']].reset_index(),
    valid_days_per_hh, on='household_ID', how='left'
)
r, p_r = spearmanr(lf_cov['load_factor'], lf_cov['valid_days'])
print(f"  Spearman r = {r:.4f}  p = {p_r:.4f}")
print(f"  {'⚠️  load_factor is correlated with coverage' if abs(r) > 0.1 and p_r < 0.05 else '✅ load_factor is NOT meaningfully correlated with coverage'}")

# Per-cluster load_factor vs valid_days
print(f"\nPer-cluster Spearman r (load_factor vs valid_days):")
for c in range(4):
    sub = lf_cov[lf_cov['final_cluster'] == c].dropna()
    r_c, p_c = spearmanr(sub['load_factor'], sub['valid_days'])
    print(f"  {c_names[c]:<24}: r={r_c:+.3f}  p={p_c:.4f}  "
          f"{'⚠️' if abs(r_c) > 0.1 and p_c < 0.05 else '✅'}")

print(f"\n=== CONCLUSION ===")
print("If Kruskal-Wallis is non-significant, Cramér's V is negligible, and load-factor correlation is small:")
print("  → No material coverage imbalance was detected across clusters.")
print("If any test is significant with meaningful effect size:")
print("  → Report the specific coverage imbalance as a limitation.")

In [ ]:
import pandas as pd
import numpy as np

print('=' * 70)
print('  STANDARDIZED LOAD PROFILES (SLPs)')
print('  Household-balanced valid-day interval profiles')
print('=' * 70)

required = ['interval_profiles', 'clean_fm', 'CLUSTER_NAME_MAP', 'CLUSTER_ORDER']
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Missing required objects: {missing}")

household_profiles = interval_profiles.reindex(clean_fm.index).copy()
if household_profiles.isna().any().any():
    raise RuntimeError('Final clustered population contains incomplete 96-slot profiles.')

household_profiles['final_cluster'] = clean_fm['final_cluster'].astype(int)
slp_raw = household_profiles.groupby('final_cluster').median().T
slp_raw = slp_raw.reindex(index=range(96), columns=CLUSTER_ORDER)
slp_raw.columns = [CLUSTER_NAME_MAP[c] for c in slp_raw.columns]
slp_norm = slp_raw.divide(slp_raw.max(axis=0).replace(0, np.nan), axis=1)

time_labels = [f'{hour:02d}:{minute:02d}' for hour in range(24) for minute in (0, 15, 30, 45)]
slp_raw.index = time_labels
slp_norm.index = time_labels

print('\n=== RAW MEDIAN HOUSEHOLD SLP (kWh per 15-min interval) ===')
print(slp_raw.round(5).to_string())
print('\n=== NORMALIZED SLP (peak = 1.000 per archetype) ===')
print(slp_norm.round(4).to_string())

print('\n=== SLP SUMMARY STATISTICS ===')
print(f"  {'Archetype':<24} {'Peak kWh':>9} {'Peak Time':>10} {'Trough kWh':>11} {'Sum of interval medians':>24}")
print('  ' + '-' * 82)
for col in slp_raw.columns:
    print(f"  {col:<24} {slp_raw[col].max():>9.4f} {slp_raw[col].idxmax():>10} "
          f"{slp_raw[col].min():>11.4f} {slp_raw[col].sum():>24.2f}")

slp_raw.to_csv('CEB_SLP_raw.csv')
slp_norm.to_csv('CEB_SLP_normalized.csv')
print('\nExported: CEB_SLP_raw.csv')
print('Exported: CEB_SLP_normalized.csv')


In [ ]:
needed = [
    'golden_df', 'feature_df', 'interval_profiles', 'clean_fm',
    'CLIP_BOUNDS', 'scaler', 'optimal_kmeans', 'CORE_FEATURES'
]
for name in needed:
    print(name, name in globals())


In [ ]:
# ============================================================
# AUTHORITATIVE MONTH-TO-MONTH TRANSITION ANALYSIS
# Reuses the exact monthly feature matrix built once in Cell 41.
# No second monthly feature-engineering pipeline is permitted.
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

if "monthly_retention_fm" not in globals():
    raise RuntimeError(
        "monthly_retention_fm is missing. Run the corrected monthly temporal "
        "stability cell before this transition analysis."
    )

required_monthly_columns = set(CORE_FEATURES) | {"monthly_cluster"}
missing_monthly_columns = required_monthly_columns.difference(monthly_retention_fm.columns)
if missing_monthly_columns:
    raise RuntimeError(
        f"Authoritative monthly matrix is missing: {sorted(missing_monthly_columns)}"
    )

# Preserve a named authoritative alias for downstream audits and exports.
monthly_fm_corrected = monthly_retention_fm.copy()
monthly_export_corrected = monthly_fm_corrected.reset_index().copy()

print("=" * 80)
print("AUTHORITATIVE MONTHLY MATRIX REUSED FOR TRANSITIONS")
print("=" * 80)
print(f"Household-month rows: {len(monthly_export_corrected):,}")
print(
    "Households represented:",
    monthly_export_corrected["household_ID"].nunique()
)

# Ensure monthly identifiers are periods for exact consecutive-month logic.
monthly_export_corrected["year_month"] = monthly_export_corrected["year_month"].astype("period[M]")

month_counts = monthly_export_corrected.groupby("household_ID", observed=True)["year_month"].nunique()
valid_hhs = month_counts[month_counts >= 3].index

transition_source = monthly_export_corrected[
    monthly_export_corrected["household_ID"].isin(valid_hhs)
].copy()
transition_source = transition_source.sort_values(["household_ID", "year_month"])

transition_source["prev_month"] = transition_source.groupby("household_ID", observed=True)["year_month"].shift(1)
transition_source["prev_cluster"] = transition_source.groupby("household_ID", observed=True)["monthly_cluster"].shift(1)
transition_source["is_consecutive"] = (
    transition_source["year_month"] == transition_source["prev_month"] + 1
)

transitions = transition_source[
    transition_source["prev_cluster"].notna()
    & transition_source["is_consecutive"]
].copy()
transitions["prev_cluster"] = transitions["prev_cluster"].astype(int)
transitions["monthly_cluster"] = transitions["monthly_cluster"].astype(int)

cluster_names = dict(CLUSTER_NAME_MAP)

transition_counts = pd.crosstab(
    transitions["prev_cluster"],
    transitions["monthly_cluster"]
).reindex(index=range(4), columns=range(4), fill_value=0)

transition_counts.index = [cluster_names[i] for i in transition_counts.index]
transition_counts.columns = [cluster_names[i] for i in transition_counts.columns]
transition_probs = transition_counts.div(
    transition_counts.sum(axis=1).replace(0, np.nan), axis=0
)

same_state = transitions["prev_cluster"] == transitions["monthly_cluster"]
overall_persistence = float(same_state.mean() * 100)

transition_summary = pd.DataFrame([{
    "authoritative_monthly_feature_rows": len(monthly_export_corrected),
    "authoritative_monthly_households": monthly_export_corrected["household_ID"].nunique(),
    "households_with_at_least_3_months": len(valid_hhs),
    "valid_consecutive_transitions": len(transitions),
    "same_archetype_persistence_percent": overall_persistence
}])

print("\nTransition counts:")
display(transition_counts)
print("\nTransition probabilities (%):")
display((transition_probs * 100).round(2))
print("\nTransition summary:")
display(transition_summary.round(4))

out_dir = Path("tables_outputs")
out_dir.mkdir(exist_ok=True)
monthly_export_corrected.to_pickle(out_dir / "authoritative_monthly_fm_with_clusters.pkl")
monthly_export_corrected.to_csv(
    out_dir / "authoritative_monthly_fm_with_clusters.csv",
    index=False
)
transition_counts.to_csv(out_dir / "authoritative_transition_counts.csv")
(transition_probs * 100).round(2).to_csv(
    out_dir / "authoritative_transition_probabilities_percent.csv"
)
transition_summary.to_csv(
    out_dir / "authoritative_transition_summary.csv",
    index=False
)

print(
    "\nMonthly dominant-state retention and consecutive transitions now use "
    "one identical authoritative monthly feature matrix."
)


In [ ]:
# Check whether monthly-state households are all in final clustered population

final_hhs = set(clean_fm.index.astype(str))

monthly_hhs = set(monthly_export_corrected["household_ID"].astype(str))

not_final = monthly_hhs - final_hhs

print("Final clustered households:", len(final_hhs))
print("Monthly-state households:", len(monthly_hhs))
print("Monthly households not in final clustered population:", len(not_final))

if len(not_final) > 0:
    print("Example non-final households:", list(not_final)[:20])
else:
    print("✅ All monthly transition households are from the final clustered population.")

In [ ]:
final_hhs = set(clean_fm.index.astype(str))
monthly_hhs = set(monthly_export_corrected["household_ID"].astype(str))

missing_monthly_state = final_hhs - monthly_hhs

print("Final clustered households:", len(final_hhs))
print("Households with at least one valid monthly state:", len(monthly_hhs))
print("Final clustered households with no valid monthly state:", len(missing_monthly_state))
print(sorted(list(missing_monthly_state)))

In [ ]:
# Diagnose why final clustered households may lack a valid monthly state.
# Uses the same strict definition as the monthly feature matrix:
# at least 20 days in a month, each with at least 90 valid intervals.

missing_ids = sorted(list(missing_monthly_state))

tmp = golden_df.loc[
    golden_df['household_ID'].astype(str).isin(missing_ids),
    ['household_ID', 'meter_dt', 'true_15min_kwh']
].copy()
tmp['household_ID'] = tmp['household_ID'].astype(str)
tmp['year_month'] = tmp['meter_dt'].dt.to_period('M')
tmp['date_only'] = tmp['meter_dt'].dt.normalize()

daily_missing_audit = (
    tmp.groupby(['household_ID', 'year_month', 'date_only'], observed=True)['true_15min_kwh']
    .count()
    .reset_index(name='valid_intervals')
)
daily_missing_audit['is_valid_day'] = daily_missing_audit['valid_intervals'] >= 90

monthly_missing_audit = (
    daily_missing_audit.groupby(['household_ID', 'year_month'], observed=True)
    .agg(
        observed_days=('date_only', 'nunique'),
        valid_days=('is_valid_day', 'sum'),
        minimum_daily_valid_intervals=('valid_intervals', 'min'),
        median_daily_valid_intervals=('valid_intervals', 'median')
    )
    .reset_index()
)
monthly_missing_audit['passes_20_valid_day_rule'] = (
    monthly_missing_audit['valid_days'] >= 20
)

display(monthly_missing_audit)


### Private diagnostic omitted — Three-household strict-month debug audit

This source cell concerned identified households and was intentionally withheld from the public review draft. It remains unchanged in the privately retained notebook. The original case-specific analysis cannot be reproduced from this public copy alone.

In [ ]:
# Check whether LUR is complete in the final clustering matrix
print("Missing LUR in final clean_fm:", clean_fm["leisure_usage_ratio"].isna().sum())
print("Inf LUR in final clean_fm:", np.isinf(clean_fm["leisure_usage_ratio"]).sum())

# Check LUR distribution by final cluster
cluster_names = dict(CLUSTER_NAME_MAP)

clean_fm_temp = clean_fm.copy()
clean_fm_temp["cluster_name"] = clean_fm_temp["final_cluster"].map(cluster_names)

display(
    clean_fm_temp.groupby("cluster_name")["leisure_usage_ratio"]
    .describe()
)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print("--- GENERATING FINAL NSRSIT TWO-COLUMN PCA FIGURE ---")

# =========================
# Load pickle file
# =========================
pkl_path = "clean_fm_clusters.pkl"   # change if needed
plot_clean_fm = pd.read_pickle(pkl_path)

# =========================
# Final six features
# =========================
features = [
    "evening_peak_ratio",
    "night_baseload_ratio",
    "load_uniformity",
    "load_factor",
    "leisure_usage_ratio",
    "log_daily_kwh"
]

cluster_col = "final_cluster"

# Keep only needed columns and clean
plot_df = plot_clean_fm[features + [cluster_col]].replace([np.inf, -np.inf], np.nan).dropna().copy()
plot_df[cluster_col] = plot_df[cluster_col].astype(int)

# Reconstruct dynamic cluster labels if this figure cell is run standalone after loading the pickle.
if 'CLUSTER_NAME_MAP' not in globals():
    if 'cluster_name' in plot_clean_fm.columns:
        CLUSTER_NAME_MAP = (
            plot_clean_fm[[cluster_col, 'cluster_name']]
            .dropna().drop_duplicates()
            .set_index(cluster_col)['cluster_name']
            .astype(str).to_dict()
        )
        ARCHETYPE_ORDER = ['Peak-Exposed', 'Typical Household', 'Heavy Continuous', 'Daytime-Active']
        CLUSTER_ORDER = [cid for name in ARCHETYPE_ORDER for cid, nm in CLUSTER_NAME_MAP.items() if nm == name]
        ARCHETYPE_COLOR_MAP = {
            'Peak-Exposed': '#e74c3c',
            'Typical Household': '#3498db',
            'Heavy Continuous': '#2ecc71',
            'Daytime-Active': '#95a5a6'
        }
        CLUSTER_COLOR_MAP = {cid: ARCHETYPE_COLOR_MAP.get(name, '#7f8c8d') for cid, name in CLUSTER_NAME_MAP.items()}
        _counts = plot_clean_fm[cluster_col].value_counts().to_dict()
        CLUSTER_LABEL_MAP = {cid: f"{CLUSTER_NAME_MAP[cid]} ({_counts.get(cid,0)/len(plot_clean_fm)*100:.1f}%)" for cid in CLUSTER_NAME_MAP}
    else:
        raise RuntimeError('CLUSTER_NAME_MAP is unavailable. Run the main clustering cell first or use a pickle with cluster_name.')


X = plot_df[features]
y = plot_df[cluster_col]

# =========================
# Standardize and PCA
# =========================
plot_scaler = StandardScaler()
X_scaled = plot_scaler.fit_transform(X)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plot_df["PC1"] = X_pca[:, 0]
plot_df["PC2"] = X_pca[:, 1]

# =========================
# Labels and colors
# =========================
cluster_labels = dict(CLUSTER_LABEL_MAP)

colors = dict(CLUSTER_COLOR_MAP)

# =========================
# NSRSIT / IEEE two-column size
# =========================
fig, ax = plt.subplots(figsize=(6.6, 4.2))

for c_id in CLUSTER_ORDER:
    subset = plot_df[plot_df[cluster_col] == c_id]
    ax.scatter(
        subset["PC1"],
        subset["PC2"],
        label=cluster_labels[c_id],
        color=colors[c_id],
        s=22,
        alpha=0.60
    )

# Axes formatting
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}% variance)", fontsize=10)
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}% variance)", fontsize=10)
ax.tick_params(axis="both", labelsize=9)

ax.grid(True, linestyle="--", linewidth=0.35, alpha=0.6)

for spine in ax.spines.values():
    spine.set_linewidth(0.8)

ax.legend(
    loc="upper left",
    fontsize=8.5,
    frameon=True,
    framealpha=1.0,
    edgecolor="0.65",
    handlelength=1.6,
    borderpad=0.5,
    labelspacing=0.4
)

# No title inside figure for conference-paper style
plt.tight_layout()

plt.savefig(
    "Fig_PCA_Projection_K4_TwoColumn.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "Fig_PCA_Projection_K4_TwoColumn.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("--- GENERATING ORIGINAL FEATURE-SPACE CLUSTER DOT PLOT ---")

# =========================
# Load pickle file
# =========================
pkl_path = "clean_fm_clusters.pkl"   # change if needed
plot_clean_fm = pd.read_pickle(pkl_path)

# =========================
# Feature pair for dot plot
# =========================
x_feature = "log_daily_kwh"
y_feature = "evening_peak_ratio"
cluster_col = "final_cluster"

plot_df = (
    plot_clean_fm[[x_feature, y_feature, cluster_col]]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .copy()
)

plot_df[cluster_col] = plot_df[cluster_col].astype(int)

# Reconstruct dynamic cluster labels if this figure cell is run standalone after loading the pickle.
if 'CLUSTER_NAME_MAP' not in globals():
    if 'cluster_name' in plot_clean_fm.columns:
        CLUSTER_NAME_MAP = (
            plot_clean_fm[[cluster_col, 'cluster_name']]
            .dropna().drop_duplicates()
            .set_index(cluster_col)['cluster_name']
            .astype(str).to_dict()
        )
        ARCHETYPE_ORDER = ['Peak-Exposed', 'Typical Household', 'Heavy Continuous', 'Daytime-Active']
        CLUSTER_ORDER = [cid for name in ARCHETYPE_ORDER for cid, nm in CLUSTER_NAME_MAP.items() if nm == name]
        ARCHETYPE_COLOR_MAP = {
            'Peak-Exposed': '#e74c3c',
            'Typical Household': '#3498db',
            'Heavy Continuous': '#2ecc71',
            'Daytime-Active': '#95a5a6'
        }
        CLUSTER_COLOR_MAP = {cid: ARCHETYPE_COLOR_MAP.get(name, '#7f8c8d') for cid, name in CLUSTER_NAME_MAP.items()}
        _counts = plot_clean_fm[cluster_col].value_counts().to_dict()
        CLUSTER_LABEL_MAP = {cid: f"{CLUSTER_NAME_MAP[cid]} ({_counts.get(cid,0)/len(plot_clean_fm)*100:.1f}%)" for cid in CLUSTER_NAME_MAP}
    else:
        raise RuntimeError('CLUSTER_NAME_MAP is unavailable. Run the main clustering cell first or use a pickle with cluster_name.')


cluster_labels = dict(CLUSTER_LABEL_MAP)

colors = dict(CLUSTER_COLOR_MAP)

# =========================
# NSRSIT / IEEE two-column size
# =========================
fig, ax = plt.subplots(figsize=(6.6, 4.2))

for c_id in CLUSTER_ORDER:
    subset = plot_df[plot_df[cluster_col] == c_id]
    ax.scatter(
        subset[x_feature],
        subset[y_feature],
        label=cluster_labels[c_id],
        color=colors[c_id],
        s=24,
        alpha=0.60,
        edgecolors="none"
    )

# Optional: show cluster median centres
centroids = plot_df.groupby(cluster_col)[[x_feature, y_feature]].median()

for c_id in CLUSTER_ORDER:
    if c_id in centroids.index:
        ax.scatter(
            centroids.loc[c_id, x_feature],
            centroids.loc[c_id, y_feature],
            marker="X",
            s=95,
            color="black",
            edgecolors="white",
            linewidths=0.8,
            zorder=5
        )

ax.set_xlabel("Log daily kWh", fontsize=10)
ax.set_ylabel("Evening Peak Ratio", fontsize=10)
ax.tick_params(axis="both", labelsize=9)

ax.grid(True, linestyle="--", linewidth=0.35, alpha=0.6)

for spine in ax.spines.values():
    spine.set_linewidth(0.8)

ax.legend(
    loc="upper right",
    fontsize=8.5,
    frameon=True,
    framealpha=1.0,
    edgecolor="0.65",
    handlelength=1.6,
    borderpad=0.5,
    labelspacing=0.4
)

# No title inside figure
plt.tight_layout()

plt.savefig(
    "Fig_Feature_Scatter_LDK_EPR_TwoColumn.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "Fig_Feature_Scatter_LDK_EPR_TwoColumn.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# STAGE 3 — EXPLORATORY MONTHLY TREND ANALYSIS
# Uses locked annual archetypes and does not refit clustering
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

CORE_FEATURES = [
    "evening_peak_ratio",
    "night_baseload_ratio",
    "load_uniformity",
    "load_factor",
    "leisure_usage_ratio",
    "log_daily_kwh"
]

CLUSTER_NAMES = dict(CLUSTER_NAME_MAP)

if "monthly_export_corrected" in globals():
    monthly_trend_df = monthly_export_corrected.copy()
else:
    monthly_trend_df = pd.read_pickle(
        "monthly_fm_with_clusters_CORRECTED.pkl"
    )

if "household_ID" not in monthly_trend_df.columns:
    monthly_trend_df = monthly_trend_df.reset_index()

locked = (
    pd.read_pickle("clean_fm_clusters.pkl")
    [["final_cluster"]]
    .reset_index()
)

if "index" in locked.columns:
    locked = locked.rename(
        columns={"index": "household_ID"}
    )

monthly_trend_df["household_ID"] = (
    monthly_trend_df["household_ID"].astype(str)
)

locked["household_ID"] = (
    locked["household_ID"].astype(str)
)

monthly_trend_df = monthly_trend_df.merge(
    locked.rename(
        columns={"final_cluster": "baseline_cluster"}
    ),
    on="household_ID",
    how="inner"
)

monthly_trend_df["month"] = pd.PeriodIndex(
    monthly_trend_df["year_month"].astype(str),
    freq="M"
).to_timestamp()

missing_features = [
    c for c in CORE_FEATURES
    if c not in monthly_trend_df.columns
]

if missing_features:
    raise KeyError(
        f"Missing monthly features: {missing_features}"
    )

monthly_medians = (
    monthly_trend_df
    .groupby(
        ["month", "baseline_cluster"]
    )[CORE_FEATURES]
    .median()
    .reset_index()
)

monthly_medians["cluster_name"] = (
    monthly_medians["baseline_cluster"]
    .map(CLUSTER_NAMES)
)

monthly_medians.to_csv(
    "monthly_feature_medians_by_locked_cluster.csv",
    index=False
)

trend_rows = []

for cluster_id, cluster_name in CLUSTER_NAMES.items():
    cluster_data = (
        monthly_medians[
            monthly_medians["baseline_cluster"].eq(cluster_id)
        ]
        .sort_values("month")
        .copy()
    )

    cluster_data["month_order"] = np.arange(
        len(cluster_data)
    )

    for feature in CORE_FEATURES:
        d = cluster_data[
            ["month_order", feature]
        ].dropna()

        if len(d) >= 4 and d[feature].nunique() > 1:
            rho, p = spearmanr(
                d["month_order"],
                d[feature]
            )
        else:
            rho, p = np.nan, np.nan

        trend_rows.append({
            "cluster": cluster_name,
            "feature": feature,
            "months_available": len(d),
            "spearman_rho": rho,
            "p_raw": p
        })

trend_results = pd.DataFrame(trend_rows)

valid = trend_results["p_raw"].notna()

trend_results["p_fdr_bh"] = np.nan
trend_results["significant_after_fdr"] = False

if valid.any():
    adjusted = multipletests(
        trend_results.loc[valid, "p_raw"],
        alpha=0.05,
        method="fdr_bh"
    )

    trend_results.loc[
        valid,
        "p_fdr_bh"
    ] = adjusted[1]

    trend_results.loc[
        valid,
        "significant_after_fdr"
    ] = adjusted[0]

trend_results = trend_results.sort_values(
    ["p_fdr_bh", "cluster", "feature"],
    na_position="last"
)

print("Monthly medians by locked baseline archetype:")
print(monthly_medians.round(4).to_string(index=False))

print("\nExploratory Spearman trend results:")
print(trend_results.round(6).to_string(index=False))

trend_results.to_csv(
    "monthly_feature_trend_tests_FDR_corrected.csv",
    index=False
)

print("\nSaved:")
print("monthly_feature_medians_by_locked_cluster.csv")
print("monthly_feature_trend_tests_FDR_corrected.csv")

print("\nInterpretation rule:")
print("- Report these as exploratory seven-month trends.")
print(
    "- Do not call them recurring seasonal effects because "
    "only May–November is observed."
)


### Private diagnostic omitted — Single-household export-register comparison

This source cell concerned identified households and was intentionally withheld from the public review draft. It remains unchanged in the privately retained notebook. The original case-specific analysis cannot be reproduced from this public copy alone.

In [ ]:
# ============================================================
# SUPPLEMENTARY IMPORT–EXPORT RELATIONSHIP AUDIT
# For export-flagged households only
# Corrected version: removes extreme export-register jumps
# Descriptive grid-exchange analysis; does not alter final clustering.
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

print("=" * 80)
print("SUPPLEMENTARY IMPORT–EXPORT RELATIONSHIP AUDIT")
print("=" * 80)

required_objects = [
    "merged_df",
    "solar_export_delta",
    "START_DATE",
    "END_DATE",
    "PHYSICAL_LIMIT"
]

missing = [obj for obj in required_objects if obj not in globals()]
if missing:
    raise RuntimeError(
        f"Missing required objects: {missing}. Run the master pipeline first."
    )

# ------------------------------------------------------------
# 1. Identify export-flagged households using final pipeline rule
# ------------------------------------------------------------
export_flagged_ids = set(
    solar_export_delta.loc[solar_export_delta > 1.0]
    .index
    .astype(str)
)

print("Export-flagged households:", len(export_flagged_ids))

needed_cols = [
    "household_ID",
    "meter_dt",
    "importkwh(kwh)",
    "exportkwh(kwh)"
]

missing_cols = [c for c in needed_cols if c not in merged_df.columns]
if missing_cols:
    raise RuntimeError(f"Missing columns in merged_df: {missing_cols}")

# Pull only needed columns for export-flagged households
ex_df = merged_df.loc[
    merged_df["household_ID"].astype(str).isin(export_flagged_ids),
    needed_cols
].copy()

ex_df["household_ID"] = ex_df["household_ID"].astype(str)
ex_df["meter_dt"] = pd.to_datetime(ex_df["meter_dt"], errors="coerce")

ex_df["import_register"] = pd.to_numeric(
    ex_df["importkwh(kwh)"],
    errors="coerce"
)

ex_df["export_register"] = pd.to_numeric(
    ex_df["exportkwh(kwh)"],
    errors="coerce"
)

# Keep the same corrected exclusive date expression as the locked pipeline
ex_df = ex_df.loc[
    (ex_df["meter_dt"] >= pd.Timestamp(START_DATE)) &
    (ex_df["meter_dt"] < pd.Timestamp(END_DATE))
].copy()

ex_df = ex_df.sort_values(["household_ID", "meter_dt"])

# ------------------------------------------------------------
# 2. Compute register deltas
# ------------------------------------------------------------
ex_df["time_delta_mins"] = (
    ex_df.groupby("household_ID", observed=True)["meter_dt"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

ex_df["import_delta_kwh"] = (
    ex_df.groupby("household_ID", observed=True)["import_register"].diff()
)

ex_df["export_delta_kwh"] = (
    ex_df.groupby("household_ID", observed=True)["export_register"].diff()
)

valid_interval = ex_df["time_delta_mins"].between(14.5, 15.5)

# ------------------------------------------------------------
# 3. Clean import and export deltas
# ------------------------------------------------------------
# Import uses the same physical cap as the main pipeline.
ex_df["valid_import_kwh"] = np.where(
    valid_interval &
    ex_df["import_delta_kwh"].between(0, PHYSICAL_LIMIT),
    ex_df["import_delta_kwh"],
    np.nan
)

# Export-register jumps also need anomaly filtering.
# Values above PHYSICAL_LIMIT per 15 minutes are treated as extreme
# export-register jumps for this supplementary grid-exchange audit.
# This is not a claim about true solar generation capacity.
EXPORT_DELTA_LIMIT = PHYSICAL_LIMIT

raw_positive_export_mask = (
    valid_interval &
    ex_df["export_delta_kwh"].notna() &
    (ex_df["export_delta_kwh"] >= 0)
)

extreme_export_jump_mask = (
    raw_positive_export_mask &
    (ex_df["export_delta_kwh"] > EXPORT_DELTA_LIMIT)
)

print("\nRaw export-register delta diagnostics:")
print("Valid-interval non-negative export deltas:", int(raw_positive_export_mask.sum()))
print("Extreme export-register jumps excluded:", int(extreme_export_jump_mask.sum()))
print(
    "Households with extreme export-register jumps:",
    ex_df.loc[extreme_export_jump_mask, "household_ID"].nunique()
)

if raw_positive_export_mask.sum() > 0:
    print(
        "Share of valid export deltas excluded as extreme:",
        f"{extreme_export_jump_mask.sum() / raw_positive_export_mask.sum() * 100:.4f}%"
    )

    print("\nRaw positive export-delta distribution before filtering:")
    print(
        ex_df.loc[raw_positive_export_mask, "export_delta_kwh"]
        .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
        .round(3)
        .to_string()
    )

ex_df["valid_export_kwh"] = np.where(
    valid_interval &
    ex_df["export_delta_kwh"].between(0, EXPORT_DELTA_LIMIT),
    ex_df["export_delta_kwh"],
    np.nan
)

# ------------------------------------------------------------
# 4. Daily grid-exchange summary
# ------------------------------------------------------------
ex_df["date_only"] = ex_df["meter_dt"].dt.date

daily_exchange = (
    ex_df
    .groupby(["household_ID", "date_only"], observed=True)
    .agg(
        valid_import_intervals=("valid_import_kwh", "count"),
        valid_export_intervals=("valid_export_kwh", "count"),
        daily_grid_import_kwh=("valid_import_kwh", "sum"),
        daily_grid_export_kwh=("valid_export_kwh", "sum")
    )
    .reset_index()
)

# For relationship analysis, require both import and export registers
# to have reasonably complete same-day valid interval coverage.
daily_exchange = daily_exchange.loc[
    (daily_exchange["valid_import_intervals"] >= 90) &
    (daily_exchange["valid_export_intervals"] >= 90)
].copy()

daily_exchange["export_active_day"] = (
    daily_exchange["daily_grid_export_kwh"] > 0
)

# Avoid unstable ratios when grid import is extremely close to zero.
daily_exchange["export_to_import_ratio"] = np.where(
    daily_exchange["daily_grid_import_kwh"] >= 0.1,
    daily_exchange["daily_grid_export_kwh"] /
    daily_exchange["daily_grid_import_kwh"],
    np.nan
)

print("\nDaily grid-exchange summary after export-delta cleaning:")
if daily_exchange.empty:
    print("No daily records remained after completeness filtering.")
else:
    print(
        daily_exchange[
            [
                "daily_grid_import_kwh",
                "daily_grid_export_kwh",
                "export_to_import_ratio"
            ]
        ]
        .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95])
        .round(3)
        .to_string()
    )

# ------------------------------------------------------------
# 5. Household-level summary
# ------------------------------------------------------------
if daily_exchange.empty:
    household_exchange = pd.DataFrame()
else:
    household_exchange = (
        daily_exchange
        .groupby("household_ID", observed=True)
        .agg(
            observed_days=("date_only", "nunique"),
            export_active_days=("export_active_day", "sum"),
            mean_daily_grid_import_kwh=("daily_grid_import_kwh", "mean"),
            median_daily_grid_import_kwh=("daily_grid_import_kwh", "median"),
            mean_daily_grid_export_kwh=("daily_grid_export_kwh", "mean"),
            median_daily_grid_export_kwh=("daily_grid_export_kwh", "median"),
            median_export_to_import_ratio=("export_to_import_ratio", "median")
        )
        .reset_index()
    )

    household_exchange["export_active_day_share"] = (
        household_exchange["export_active_days"] /
        household_exchange["observed_days"].clip(lower=1)
    )

    print("\nHousehold-level import–export relationship summary:")
    print(
        household_exchange[
            [
                "observed_days",
                "export_active_days",
                "export_active_day_share",
                "mean_daily_grid_import_kwh",
                "mean_daily_grid_export_kwh",
                "median_export_to_import_ratio"
            ]
        ]
        .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95])
        .round(3)
        .to_string()
    )

# ------------------------------------------------------------
# 6. Daily import-export correlation
# ------------------------------------------------------------
if not daily_exchange.empty:
    corr_data = daily_exchange[
        ["daily_grid_import_kwh", "daily_grid_export_kwh"]
    ].dropna()

    if len(corr_data) >= 10 and corr_data["daily_grid_export_kwh"].nunique() > 1:
        rho, p = spearmanr(
            corr_data["daily_grid_import_kwh"],
            corr_data["daily_grid_export_kwh"]
        )

        print("\nDaily import–export Spearman correlation:")
        print(f"rho = {rho:.4f}, p = {p:.4g}")

        if abs(rho) < 0.1:
            print(
                "Interpretation: No meaningful monotonic relationship between "
                "observed daily grid import and cleaned daily export-register "
                "movement was detected. Any statistical significance should be "
                "interpreted cautiously because the effect size is very small."
            )
        elif rho < -0.1:
            print(
                "Interpretation: Higher export days tend to have lower observed "
                "grid import, but this is grid exchange only, not total demand."
            )
        else:
            print(
                "Interpretation: Higher export days also tend to have higher "
                "observed grid import, suggesting larger or more active households; "
                "total demand is still unobserved."
            )
    else:
        print("\nNot enough usable daily import/export variation for correlation testing.")
else:
    print("\nCorrelation skipped because daily_exchange is empty.")

# ------------------------------------------------------------
# 7. Save supplementary outputs
# ------------------------------------------------------------
daily_exchange.to_csv(
    "supplementary_daily_import_export_exchange_export_flagged_CLEANED.csv",
    index=False
)

household_exchange.to_csv(
    "supplementary_household_import_export_exchange_export_flagged_CLEANED.csv",
    index=False
)

print("\nSaved:")
print("- supplementary_daily_import_export_exchange_export_flagged_CLEANED.csv")
print("- supplementary_household_import_export_exchange_export_flagged_CLEANED.csv")

print("\nSafe interpretation:")
print(
    "This audit describes observed grid exchange among export-flagged households "
    "after removing extreme export-register jumps. Grid import is electricity "
    "drawn from the grid, while cleaned export-register movement is observed "
    "electricity sent back to the grid. The relationship is not interpreted as "
    "total household demand or total solar generation because behind-the-meter "
    "self-consumption is not observed. Extreme export-register jumps are treated "
    "as register anomalies for this behavioural relationship audit and are not "
    "reported as real daily export behaviour."
)


In [ ]:
# ============================================================
# FINAL CONSERVATIVE CAPSTONE SUMMARY
# Run only after every primary and supplementary validation cell.
# ============================================================

print("=" * 76)
print("FINAL THESIS RESULTS: CEB RESIDENTIAL K=4 ARCHETYPES")
print("=" * 76)

cluster_names = dict(CLUSTER_NAME_MAP)

print(f"\nTotal active households analysed: {len(clean_fm):,}")
print(
    "Data window: 2024-05-01 <= meter_dt < 2024-12-01 "
    "(full May–November golden window)"
)
print(
    f"Primary model: KMeans | K={OPTIMAL_K} | StandardScaler | "
    f"random_state={RANDOM_STATE} | n_init={OPTIMAL_N_INIT}"
)

print("\n--- FINAL CLUSTER POPULATIONS ---")
for cluster_id in CLUSTER_ORDER:
    n = int((clean_fm["final_cluster"] == cluster_id).sum())
    pct = n / len(clean_fm) * 100
    print(
        f"{cluster_id}: {cluster_names[cluster_id]:<18} | "
        f"{n:>4} households | {pct:>5.1f}%"
    )

print("\n--- PRIMARY INTERNAL VALIDATION ---")
if "sil_score" in globals():
    print(f"Overall KMeans K=4 silhouette: {sil_score:.3f}")
if "ari_seeds" in globals():
    print(f"Seed-stability minimum ARI: {np.min(ari_seeds):.3f}")
if "ari_boot" in globals():
    print(f"Bootstrap mean ARI: {np.mean(ari_boot):.3f}")

print("\n--- COMPLETED SUPPLEMENTARY ALGORITHM CHECKS ---")
if "gmm_k4_summary" in globals():
    row = gmm_k4_summary.iloc[0]
    print(
        f"GMM K=4: converged={bool(row['gmm_converged'])}, "
        f"ARI vs KMeans={row['adjusted_rand_index_vs_kmeans']:.3f}."
    )
    print(
        "GMM posterior probabilities are confidence measures within the GMM "
        "partition, not KMeans membership probabilities."
    )
if "fcm_k4_summary" in globals():
    row = fcm_k4_summary.iloc[0]

    if bool(row["hard_partition_complete"]):
        print(
            f"Fuzzy C-Means K=4: converged starts="
            f"{int(row['converged_runs'])}/"
            f"{int(row['total_runs'])}, "
            f"ARI vs KMeans="
            f"{row['adjusted_rand_index_vs_kmeans']:.3f}."
        )
    else:
        print(
            f"Fuzzy C-Means K=4: converged starts="
            f"{int(row['converged_runs'])}/"
            f"{int(row['total_runs'])}, but maximum-membership "
            f"hardening produced only "
            f"{int(row['occupied_hard_clusters'])} occupied clusters. "
            "Hard-label silhouette and ARI were therefore not interpreted."
        )
if "kmedoids_k4_summary" in globals():
    row = kmedoids_k4_summary.iloc[0]
    print(
        f"K-Medoids K=4: converged starts="
        f"{int(row['converged_runs'])}/{int(row['total_runs'])}, "
        f"ARI vs KMeans={row['adjusted_rand_index_vs_kmeans']:.3f}."
    )
print(
    "All alternative algorithms are supplementary robustness checks; none "
    "replaces the interpretation-led locked KMeans solution."
)

print("\n--- TEMPORAL CONSISTENCY ---")
if "locked_temporal_summary" in globals():
    row = locked_temporal_summary.iloc[0]
    print(
        f"Strict early-vs-late same-cluster share: "
        f"{row['same_cluster_share'] * 100:.1f}% across "
        f"{int(row['common_households']):,} common households."
    )
    if "ari_early_vs_late" in row.index:
     print(
        f"Strict early-vs-late ARI: "
        f"{row['ari_early_vs_late']:.3f}."
    )
    print(
        "This indicates moderate/partial temporal consistency rather than "
        "perfect longitudinal stability."
    )

if "retention_summary" in globals():
    row = retention_summary.iloc[0]
    print(
        f"Dominant-state monthly retention: "
        f"{row['mean_dominant_state_retention_rate'] * 100:.1f}% across "
        f"{int(row['households_evaluated_ge_3_months']):,} households."
    )

if "transition_summary" in globals():
    row = transition_summary.iloc[0]
    print(
        f"Consecutive month-to-month persistence: "
        f"{row['same_archetype_persistence_percent']:.2f}% across "
        f"{int(row['valid_consecutive_transitions']):,} transitions."
    )

print("\n--- SENSITIVITY AND LIMITATIONS ---")
print(
    "The targeted one-household sensitivity investigation is documented "
    "in the private research record and is omitted from this public notebook."
)

print(
    "The 1st–99th percentile clipping step is retained because it improves "
    "seed robustness while preserving the four substantive archetypes."
)
print(
    "The 6.0 kWh/15-minute cap is conservative. Higher-cap alternatives are "
    "reported as sensitivity analyses rather than silently replacing the rule."
)
print(
    "Monthly analyses are conditional on household-months passing the strict "
    "valid-day threshold, and coverage is uneven across months."
)
print(
    "The tariff boundary is treated as a documented modelling convention; "
    "±15-minute sensitivity is reported because aggregate telemetry cannot "
    "identify interval-start versus interval-end semantics by itself."
)

print("\n--- FINAL INTERPRETATION ---")
print(
    "The locked KMeans K=4 result is a reproducible and interpretable "
    "behavioural archetype segmentation, not proof of perfectly separated "
    "natural classes. Boundary movement and temporal state changes are "
    "expected and are reported transparently."
)

print("\n" + "=" * 76)
print("FINAL PIPELINE COMPLETED WITH CONSERVATIVE INTERPRETATION.")
print("=" * 76)


In [ ]:
# ============================================================
# FINAL AUTOMATED QUALITY-ASSURANCE GATE
# Raises an error if the primary pipeline, model inputs, supplementary
# algorithms or downstream sensitivity analyses are incomplete or inconsistent.
# ============================================================

EXPECTED_FINAL_POPULATION = 2227
EXPECTED_RAW_ELIGIBLE_POPULATION = 2316
EXPECTED_GHOSTS_REMOVED = 89
EXPECTED_K_VALUES = set(range(2, 11))

qa_checks = {}

# Primary dataset and dataframe contract
qa_checks["raw_feature_population_2316"] = (
    len(feature_matrix) == EXPECTED_RAW_ELIGIBLE_POPULATION
)
qa_checks["ghosts_removed_89"] = (
    int((feature_matrix["log_daily_kwh"] <= 0.05).sum())
    == EXPECTED_GHOSTS_REMOVED
)
qa_checks["final_population_2227"] = (
    len(clean_fm) == EXPECTED_FINAL_POPULATION
)
qa_checks["unique_household_index"] = bool(clean_fm.index.is_unique)
qa_checks["no_missing_locked_features"] = bool(
    clean_fm[CORE_FEATURES]
    .replace([np.inf, -np.inf], np.nan)
    .notna()
    .all()
    .all()
)
qa_checks["population_matches_X_clean"] = (
    X_clean.shape == (len(clean_fm), len(CORE_FEATURES))
)
qa_checks["X_clean_finite"] = bool(np.isfinite(X_clean).all())
qa_checks["four_primary_clusters"] = (
    clean_fm["final_cluster"].nunique() == 4
)
qa_checks["cluster_population_sums"] = (
    int(clean_fm["final_cluster"].value_counts().sum())
    == len(clean_fm)
)

# Exact locked-input alignment
qa_checks["model_input_audit_exists"] = bool(
    "model_input_audit" in globals()
    and not model_input_audit.empty
)
qa_checks["household_order_locked"] = bool(
    np.array_equal(
        MODEL_HOUSEHOLD_ORDER,
        clean_fm.index.astype(str).to_numpy()
    )
)
qa_checks["feature_order_locked"] = (
    tuple(CORE_FEATURES) == MODEL_FEATURE_ORDER
)
qa_checks["kmeans_labels_aligned"] = bool(
    np.array_equal(
        optimal_kmeans.labels_.astype(int),
        clean_fm["final_cluster"].astype(int).to_numpy()
    )
)

# GMM K=4 and K sweep
qa_checks["gmm_k4_converged"] = bool(
    gmm_k4_summary.iloc[0]["gmm_converged"]
)
qa_checks["gmm_k4_population_aligned"] = (
    int(gmm_k4_summary.iloc[0]["n_households"]) == len(clean_fm)
    and len(gmm_posterior_households) == len(clean_fm)
)
qa_checks["gmm_k_sweep_complete"] = (
    set(gmm_k_sweep.index.astype(int)) == EXPECTED_K_VALUES
)
gmm_nonconverged = ~gmm_k_sweep["converged"]
qa_checks["gmm_nonconverged_metrics_are_missing"] = bool(
    gmm_k_sweep.loc[
        gmm_nonconverged,
        [
            "bic",
            "aic",
            "hard_label_silhouette",
            "smallest_component",
            "largest_component"
        ]
    ].isna().all().all()
)
qa_checks["gmm_k4_sweep_converged"] = bool(
    gmm_k_sweep.loc[4, "converged"]
)
qa_checks["gmm_dna_convergence_recorded"] = (
    set(gmm_dna_convergence.index.astype(int)) == {3, 4, 5, 6}
)

# Fuzzy C-Means K=4 and K sweep
qa_checks["fcm_k4_converged"] = bool(
    fcm_k4_summary.iloc[0]["selected_run_converged"]
)
qa_checks["fcm_k4_population_aligned"] = (
    int(fcm_k4_summary.iloc[0]["n_households"]) == len(clean_fm)
    and len(fcm_households) == len(clean_fm)
)
qa_checks["fcm_memberships_sum_to_one"] = bool(
    np.allclose(
        fcm_k4_result["membership"].sum(axis=1),
        1.0,
        atol=1e-6
    )
)
fcm_k4_row = fcm_k4_summary.iloc[0]

qa_checks["fcm_k4_hard_partition_handled"] = bool(
    (
        bool(fcm_k4_row["hard_partition_complete"])
        and int(fcm_k4_row["occupied_hard_clusters"]) == 4
        and pd.notna(
            fcm_k4_row["hard_label_silhouette"]
        )
        and pd.notna(
            fcm_k4_row[
                "adjusted_rand_index_vs_kmeans"
            ]
        )
    )
    or
    (
        not bool(
            fcm_k4_row["hard_partition_complete"]
        )
        and int(
            fcm_k4_row["occupied_hard_clusters"]
        ) < 4
        and pd.isna(
            fcm_k4_row["hard_label_silhouette"]
        )
        and pd.isna(
            fcm_k4_row[
                "adjusted_rand_index_vs_kmeans"
            ]
        )
    )
) 

qa_checks["fcm_k_sweep_complete"] = (
    set(fcm_k_sweep.index.astype(int)) == EXPECTED_K_VALUES
)
fcm_nonconverged = ~fcm_k_sweep["selected_run_converged"]

qa_checks["fcm_nonconverged_metrics_are_missing"] = bool(
    fcm_k_sweep.loc[
        fcm_nonconverged,
        [
            "hard_label_silhouette",
            "fuzzy_partition_coefficient",
            "partition_entropy",
            "smallest_hard_cluster",
            "largest_hard_cluster"
        ]
    ].isna().all().all()
)
fcm_incomplete_hard = (
    fcm_k_sweep["selected_run_converged"]
    & ~fcm_k_sweep["hard_partition_complete"]
)

qa_checks[
    "fcm_incomplete_hard_silhouettes_are_missing"
] = bool(
    fcm_k_sweep.loc[
        fcm_incomplete_hard,
        ["hard_label_silhouette"]
    ]
    .isna()
    .all()
    .all()
)

fcm_complete_hard = (
    fcm_k_sweep["selected_run_converged"]
    & fcm_k_sweep["hard_partition_complete"]
)

qa_checks[
    "fcm_complete_hard_silhouettes_exist"
] = bool(
    fcm_k_sweep.loc[
        fcm_complete_hard,
        ["hard_label_silhouette"]
    ]
    .notna()
    .all()
    .all()
) 
qa_checks["fcm_k4_sweep_converged"] = bool(
    fcm_k_sweep.loc[4, "selected_run_converged"]
)

# K-Medoids K=4 and K sweep
qa_checks["kmedoids_k4_converged"] = bool(
    kmedoids_k4_result["converged"]
    and kmedoids_k4_result["converged_runs"] >= 1
)
qa_checks["kmedoids_k4_population_aligned"] = (
    int(kmedoids_k4_summary.iloc[0]["n_households"]) == len(clean_fm)
    and len(kmedoids_k4_households) == len(clean_fm)
)
qa_checks["kmedoids_k4_unique_medoids"] = (
    len(np.unique(kmedoids_k4_result["medoid_indices"])) == 4
)
qa_checks["kmedoids_k_sweep_complete"] = (
    set(kmedoids_k_sweep.index.astype(int)) == EXPECTED_K_VALUES
)
kmedoids_nonconverged = ~kmedoids_k_sweep["converged"]
qa_checks["kmedoids_nonconverged_metrics_are_missing"] = bool(
    kmedoids_k_sweep.loc[
        kmedoids_nonconverged,
        [
            "silhouette",
            "smallest_cluster",
            "largest_cluster",
            "ari_vs_locked_kmeans_k4"
        ]
    ].isna().all().all()
)
qa_checks["kmedoids_k4_sweep_converged"] = bool(
    kmedoids_k_sweep.loc[4, "converged"]
)

# Downstream temporal and sensitivity analyses
qa_checks["single_authoritative_monthly_matrix"] = bool(
    len(monthly_fm_corrected) == len(monthly_retention_fm)
    and monthly_fm_corrected.index.equals(monthly_retention_fm.index)
)
qa_checks["transition_summary_exists"] = bool(
    "transition_summary" in globals()
    and not transition_summary.empty
)
qa_checks["block_coverage_audited"] = bool(
    "block_coverage_summary" in globals()
    and not block_coverage_summary.empty
)
qa_checks["boundary_sensitivity_completed"] = bool(
    "boundary_sensitivity_summary" in globals()
    and set(boundary_sensitivity_summary["peak_start"])
    == {"18:15", "18:30", "18:45"}
)
qa_checks["cap_sensitivity_completed"] = bool(
    "cap_sensitivity_summary" in globals()
    and set(cap_sensitivity_summary["cap_kwh_per_15min"])
    == {8.0, 10.5}
)

qa_checks["majority_bootstrap_100_runs"] = bool(
    "ari_boot_majority" in globals()
    and len(ari_boot_majority) == 100
    and np.isfinite(
        np.asarray(
            ari_boot_majority,
            dtype=float
        )
    ).all()
)

qa_table = pd.Series(qa_checks, name="passed").to_frame()
print("Final automated QA results:")
display(qa_table)

failed_checks = qa_table.index[~qa_table["passed"]].tolist()
if failed_checks:
    raise RuntimeError(f"FINAL QA FAILED: {failed_checks}")

print("ALL REQUIRED FINAL QA CHECKS PASSED.")


In [ ]:
import hashlib
from pathlib import Path

pickle_path = Path("clean_fm_clusters.pkl")

if not pickle_path.exists():
    raise FileNotFoundError(
        "clean_fm_clusters.pkl was not found."
    )

sha256 = hashlib.sha256(
    pickle_path.read_bytes()
).hexdigest()

print("File:", pickle_path.name)
print("SHA-256:", sha256)